# MOPAC cuBLAS/cuSOLVER + MOZYME GPU Benchmark

This notebook builds the current production GPU branch from a freshly uploaded source zip. Dense linear algebra is routed through NVIDIA libraries, and the default proof path runs the strict resident full-SCF GPU readiness gate before any long publication benchmark is started. The optional resident SCF smoke is a non-proof diagnostic; only the strict readiness artifact proves full-SCF GPU execution.

Use a newly generated `mopac_colab_gpu_bench.zip`, upload its `.zip.expected.json`, and paste the printed `source_zip_sha256` into `trusted_expected_source_zip_sha256` before proof mode. The setup cell deletes old `/content/mopac_src` and `/content/mopac_colab_build` directories so Colab cannot silently reuse stale files.


## 1. Confirm GPU

In [ ]:
from pathlib import Path, PurePosixPath, PureWindowsPath
import hashlib
import os
import re
import shutil
import subprocess

CONTENT_DIR = Path('/content')
UPLOAD_DIR = CONTENT_DIR / 'mopac_upload'
SOURCE_ROOT = CONTENT_DIR / 'mopac_src'
BUILD_DIR = CONTENT_DIR / 'mopac_colab_build'
EXPECTED_MOPAC_GPU_FEATURE_SET = 'mozyme-full-scf-gpu-makvec-relocal-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v38-20260702'
EXPECTED_MOPAC_GPU_CONTRACT_VERSION = 'resident-scf-strict-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-resident-cg-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v58'
EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_VERSION = 'mopac-colab-source-markers-explicit-proof-v83'
EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256 = '17aa11cd4550db1b4d2bbf2f4b15bfc13b9c6e06faad28f6609ee42c7ab69240'
# Dirty source zips are proof-ineligible by default. Enable the non-proof/dev path explicitly for dirty diagnostics.
allow_development_dirty_zip = False
allow_expected_dirty_source_zip = False
# Exact source freshness is required by default. The uploaded sidecar validates manifest/provenance,
# but proof mode also requires an out-of-band SHA pasted below from the local packager output.
require_pinned_source_zip = True
trusted_expected_source_zip_sha256 = ''
trusted_expected_source_git_commit = ''
expected_source_zip_sha256 = ''
expected_source_manifest_sha256 = ''
expected_source_git_commit = ''
expected_source_git_dirty = None
expected_source_dirty_status_sha256 = ''
expected_source_proof_eligible = None
expected_source_proof_ineligible_reason = ''
GPU_ERROR_MARKERS = (
    '[GPU ERROR]',
    'ACCURACY_FAIL',
    'BENCH_FAIL',
    'cuBLAS status',
    'cuSOLVER status',
    'CUBLAS_STATUS_',
    'CUSOLVER_STATUS_',
    'CUDA error',
    'cudaError',
    'illegal memory access',
    'device-side assert',
    'device assert',
    'misaligned address',
    'unspecified launch failure',
    'out of memory',
    'Segmentation fault',
    'SIGSEGV',
    'core dumped',
)
DEFERRED_LOW_LEVEL_BENCHMARKS = {}
def low_level_benchmarks_ready():
    return bool(globals().get('allow_non_proof_run', False) or (globals().get('full_scf_readiness_passed', False) and globals().get('direct_cosmo_readiness_passed', False)))
def run_or_defer_low_level(name, callback):
    if low_level_benchmarks_ready():
        callback()
    else:
        DEFERRED_LOW_LEVEL_BENCHMARKS[name] = callback
        print(f'Deferred {name} until strict readiness proof passes; set allow_non_proof_run=True for diagnostics.')
def run_deferred_low_level_benchmarks():
    if not DEFERRED_LOW_LEVEL_BENCHMARKS:
        return
    if not low_level_benchmarks_ready():
        print('Deferred low-level benchmarks remain blocked until strict readiness proof passes.')
        return
    pending = list(DEFERRED_LOW_LEVEL_BENCHMARKS.items())
    DEFERRED_LOW_LEVEL_BENCHMARKS.clear()
    for name, callback in pending:
        print('Running deferred low-level benchmark:', name)
        callback()


def run(cmd, *, cwd=None, env=None, log_path=None, check=True):
    cmd = [str(item) for item in cmd]
    print('\n+', ' '.join(cmd), flush=True)
    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    if log_path is not None:
        log_path = Path(log_path)
        log_path.write_text(result.stdout, encoding='utf-8')
        print(f'[log] {log_path}', flush=True)
    print(result.stdout, end='')
    if check and result.returncode != 0:
        if log_path is not None:
            print(f'Command failed. Full log: {log_path}', flush=True)
        raise subprocess.CalledProcessError(result.returncode, result.args, output=result.stdout)
    if check or result.returncode == 0:
        lower_stdout = result.stdout.lower()
        for marker in GPU_ERROR_MARKERS:
            if marker.lower() in lower_stdout:
                raise RuntimeError(f"Command output contained GPU error marker {marker!r}: {' '.join(cmd)}")
    return result


def clean_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
    return path

gpu_check = run(['nvidia-smi', '-L'], check=False)
if gpu_check.returncode != 0 or 'GPU' not in gpu_check.stdout:
    raise RuntimeError('This notebook requires a Colab runtime with an NVIDIA GPU. Select Runtime > Change runtime type > GPU.')
cuda_check = run(['nvcc', '--version'], check=False)
if cuda_check.returncode != 0:
    raise RuntimeError('CUDA compiler nvcc is not available in this runtime; cannot build the GPU implementation.')


A100 and H100 are both valid targets. The benchmark uses vendor FP64 library paths, so no architecture-specific CUDA math source is required.

## 2. Upload And Extract Source Zip

In [ ]:
from google.colab import files
from pathlib import Path
import json
import os
import shutil
import zipfile

os.chdir(CONTENT_DIR)
clean_dir(UPLOAD_DIR)
os.chdir(UPLOAD_DIR)
print('Upload the NEW mopac_colab_gpu_bench.zip and its .expected.json sidecar.')
uploaded = files.upload()
if len(uploaded) not in (1, 2):
    raise RuntimeError('Upload the source zip and, by default, its .expected.json sidecar.')
zip_uploads = [name for name in uploaded if Path(name).suffix.lower() == '.zip']
expected_uploads = [name for name in uploaded if Path(name).name.endswith('.zip.expected.json')]
if len(zip_uploads) != 1:
    raise RuntimeError('Upload exactly one .zip source archive.')
if len(expected_uploads) > 1:
    raise RuntimeError('Upload at most one .zip.expected.json sidecar.')
if expected_uploads:
    expected_payload = json.loads(uploaded[expected_uploads[0]].decode('utf-8'))
    if expected_payload.get('schema') != 'mopac-colab-expected-source-v1':
        raise RuntimeError('The uploaded expected-source sidecar has an unknown schema.')
    if expected_payload.get('feature_set') != EXPECTED_MOPAC_GPU_FEATURE_SET:
        raise RuntimeError('The uploaded expected-source sidecar feature_set does not match this notebook contract.')
    if expected_payload.get('full_scf_contract_version') != EXPECTED_MOPAC_GPU_CONTRACT_VERSION:
        raise RuntimeError('The uploaded expected-source sidecar full_scf_contract_version does not match this notebook contract.')
    if expected_payload.get('source_marker_contract_version') != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_VERSION:
        raise RuntimeError('The uploaded expected-source sidecar source_marker_contract_version does not match this notebook contract.')
    if expected_payload.get('source_marker_contract_sha256') != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256:
        raise RuntimeError('The uploaded expected-source sidecar source_marker_contract_sha256 does not match this notebook contract.')
    if not isinstance(expected_payload.get('proof_eligible'), bool):
        raise RuntimeError('The uploaded expected-source sidecar is missing boolean proof_eligible.')
    if not isinstance(expected_payload.get('proof_ineligible_reason', ''), str):
        raise RuntimeError('The uploaded expected-source sidecar proof_ineligible_reason is not a string.')
    expected_source_zip_sha256 = expected_payload.get('source_zip_sha256', '')
    expected_source_manifest_sha256 = expected_payload.get('source_manifest_sha256', '')
    expected_source_git_commit = expected_payload.get('source_git_commit', '')
    expected_source_git_dirty = expected_payload.get('source_git_dirty')
    expected_source_dirty_status_sha256 = expected_payload.get('source_dirty_status_sha256', '')
    expected_source_proof_eligible = expected_payload.get('proof_eligible')
    expected_source_proof_ineligible_reason = expected_payload.get('proof_ineligible_reason', '')
if require_pinned_source_zip and not expected_source_zip_sha256:
    raise RuntimeError('Source zip freshness is unpinned. Upload the .zip.expected.json sidecar or set expected_source_zip_sha256.')
upload_key = zip_uploads[0]
zip_name = Path(upload_key).name
if Path(zip_name).suffix.lower() != '.zip':
    raise RuntimeError(f'Uploaded file must be a .zip archive, got {zip_name!r}')
zip_path = UPLOAD_DIR / 'mopac_colab_gpu_bench.zip'
zip_path.write_bytes(uploaded[upload_key])
source_zip_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
trusted_expected_source_zip_sha256 = str(trusted_expected_source_zip_sha256 or '').strip().lower()
if require_pinned_source_zip:
    if not re.fullmatch(r'[0-9a-f]{64}', trusted_expected_source_zip_sha256):
        raise RuntimeError('require_pinned_source_zip=True requires trusted_expected_source_zip_sha256 as a 64-character lowercase SHA-256 hex string.')
    if source_zip_sha256 != trusted_expected_source_zip_sha256:
        raise RuntimeError('The uploaded source zip SHA-256 does not match trusted_expected_source_zip_sha256.')
if expected_source_zip_sha256 and source_zip_sha256 != expected_source_zip_sha256:
    raise RuntimeError('The uploaded source zip SHA-256 does not match expected_source_zip_sha256.')
print(f'Using zip: {zip_path} ({zip_path.stat().st_size / 1_000_000:.1f} MB)')
print('Uploaded zip sha256:', source_zip_sha256)

if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)
SOURCE_ROOT.mkdir(parents=True)

with zipfile.ZipFile(zip_path) as zf:
    unsafe_members = []
    duplicate_members = []
    seen_members = set()
    for member in zf.infolist():
        name = member.filename
        normalized = name.replace('\\', '/')
        posix_path = PurePosixPath(normalized)
        windows_path = PureWindowsPath(name)
        if normalized in seen_members:
            duplicate_members.append(name)
        seen_members.add(normalized)
        if posix_path.is_absolute() or windows_path.is_absolute() or '..' in posix_path.parts:
            unsafe_members.append(name)
    if unsafe_members:
        raise RuntimeError('The uploaded zip contains unsafe member paths: ' + ', '.join(unsafe_members[:20]))
    if duplicate_members:
        raise RuntimeError('The uploaded zip contains duplicate member paths: ' + ', '.join(duplicate_members[:20]))
    zf.extractall(SOURCE_ROOT)

candidates = sorted(
    [item.parent for item in SOURCE_ROOT.rglob('CMakeLists.txt')],
    key=lambda path: (len(path.parts), str(path)),
)
if not candidates:
    raise RuntimeError('Could not find CMakeLists.txt after extraction')

source_dir = candidates[0]
os.chdir(source_dir)
print('Source directory:', source_dir)

required_paths = [
    'CMakeLists.txt',
    'src/CMakeLists.txt',
    'src/run_mopac.F90',
    'src/MOZYME/CMakeLists.txt',
    'src/MOZYME/mozyme_resident_fock.F90',
    'src/MOZYME/mozyme_section_timers.F90',
    'src/MOZYME/mozyme_gpu_plan.F90',
    'src/MOZYME/mozyme_gpu_makvec.F90',
    'src/MOZYME/mozyme_gpu_relocalize.F90',
    'src/MOZYME/mozyme_gpu_reorth.F90',
    'src/MOZYME/mozyme_gpu_tidy.F90',
    'src/MOZYME/mozyme_gpu_scf_driver.F90',
    'src/MOZYME/mozyme_diagg1_state.F90',
    'src/MOZYME/mozyme_diagg2_state.F90',
    'src/MOZYME/mozyme_fock1_batch.F90',
    'src/MOZYME/mozyme_fock2_4x1_batch.F90',
    'src/MOZYME/mozyme_gpu_int_utils.F90',
    'src/MOZYME/mozyme_isitsc_state.F90',
    'src/MOZYME/iter_for_MOZYME.F90',
    'src/MOZYME/eimp.F90',
    'src/MOZYME/addhb.F90',
    'src/MOZYME/add_more_interactions.F90',
    'src/MOZYME/check.F90',
    'src/MOZYME/cnvgz.F90',
    'src/MOZYME/helecz.F90',
    'src/MOZYME/density_for_MOZYME.F90',
    'src/MOZYME/diagg.F90',
    'src/MOZYME/diagg1.F90',
    'src/MOZYME/diagg2.F90',
    'src/MOZYME/fillij.F90',
    'src/MOZYME/fock1_for_MOZYME.F90',
    'src/MOZYME/fock2z.F90',
    'src/MOZYME/isitsc.F90',
    'src/MOZYME/reorth.F90',
    'src/MOZYME/set_up_MOZYME_arrays.F90',
    'src/MOZYME/setupk.F90',
    'src/MOZYME/tidy.F90',
    'src/MOZYME/pinout.F90',
    'src/output/writmo.F90',
    'src/solvation/linear_cosmo.F90',
    'src/gpu/cuda_wrappers.cu',
    'src/gpu/cublas_interfaces.F90',
    'src/gpu/fock_kernels.cu',
    'src/gpu/grad_launch.h',
    'src/gpu/gpu_bmat_interfaces.F90',
    'src/gpu/gpu_density_interfaces.F90',
    'src/gpu/gpu_diis_interfaces.F90',
    'src/gpu/gpu_eig_mg_interfaces.F90',
    'src/gpu/gpu_fock_interfaces.F90',
    'src/gpu/gpu_grad_interfaces.F90',
    'src/gpu/gpu_hmtr_interfaces.F90',
    'src/gpu/gpu_mozyme_scf_interfaces.F90',
    'src/gpu/gpu_ortho_interfaces.F90',
    'src/gpu/gpu_runtime_interfaces.F90',
    'src/gpu/gpu_scf_interfaces.F90',
    'src/gpu/gpu_scf_stream_driver.F90',
    'src/gpu/gpu_scf_stream_interfaces.F90',
    'src/gpu/gpu_scf_stream_trace.F90',
    'src/gpu/gpu_scf_types.F90',
    'src/gpu/gpu_small_solve_interfaces.F90',
    'src/gpu/gpu_transform_interfaces.F90',
    'src/gpu/grad_kernels.cu',
    'src/gpu/hmtr_optimizer.cu',
    'src/gpu/packed_utils.h',
    'src/gpu/scf_driver.cu',
    'src/gpu/mozyme_scf_context.cu',
    'tests/CMakeLists.txt',
    'tests/gpu_bench.F90',
    'tests/gpu_resident_fock_pair_compare.F90',
    'tests/check_gpu_env_flag_parsing.py',
    'tests/check_gpu_source_build_contract.py',
    'tests/check_mozyme_strict_resident_scf.py',
    'docs/GPU_GUIDE.md',
    'scripts/gpu_benchmark_report.py',
    'scripts/molecule_benchmark_report.py',
    'scripts/create_colab_gpu_zip.py',
    'scripts/verify_colab_gpu_proof_zip.py',
    'scripts/prepare_publication_benchmark_inputs.py',
    'scripts/hydrogenate_publication_benchmark_inputs.py',
    'scripts/collect_existing_mopac_references.py',
    'colab/mopac_cublas_gpu_bench_colab.ipynb',
]
missing = []
for rel in required_paths:
    ok = (source_dir / rel).exists()
    print(f'{rel}:', 'OK' if ok else 'MISSING')
    if not ok:
        missing.append(rel)
if missing:
    raise RuntimeError('The uploaded zip is stale or incomplete. Missing: ' + ', '.join(missing))

SOURCE_ALLOWLIST_POLICY_SCHEMA = 'mopac-colab-source-allowlist-v1'
ALLOWED_SOURCE_ROOT_FILES = {
    '.gitignore',
    'AUTHORS.rst',
    'CITATION.cff',
    'CMakeLists.txt',
    'CODE_OF_CONDUCT.md',
    'CONTRIBUTING.rst',
    'Dockerfile',
    'LICENSE',
    'NOTICE',
    'README.md',
}
ALLOWED_SOURCE_ROOT_DIRS = {
    '.github',
    'benchmarks',
    'cmake',
    'colab',
    'data',
    'docs',
    'examples',
    'include',
    'logo',
    'scripts',
    'src',
    'tests',
}

def colab_source_path_allowed(rel):
    parts = PurePosixPath(rel).parts
    if len(parts) == 1:
        return rel in ALLOWED_SOURCE_ROOT_FILES
    return bool(parts) and parts[0] in ALLOWED_SOURCE_ROOT_DIRS

manifest_path = source_dir / 'MOPAC_COLAB_SOURCE_MANIFEST.sha256'
if not manifest_path.exists():
    raise RuntimeError('The uploaded zip has no SHA-256 source manifest. Recreate it with scripts/create_colab_gpu_zip.py.')
manifest_failures = []
manifest_entries = 0
manifest_paths = set()
for raw_line in manifest_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if not line:
        continue
    manifest_entries += 1
    try:
        expected_hash, rel = line.split(None, 1)
    except ValueError:
        manifest_failures.append(f'manifest line {manifest_entries}: malformed')
        continue
    rel_path = PurePosixPath(rel)
    if rel_path.is_absolute() or '..' in rel_path.parts:
        manifest_failures.append(f'{rel}: unsafe manifest path')
        continue
    if not colab_source_path_allowed(rel):
        manifest_failures.append(f'{rel}: outside source allowlist policy')
        continue
    manifest_paths.add(rel)
    path = source_dir / rel
    if not path.exists():
        manifest_failures.append(f'{rel}: missing')
        continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != expected_hash:
        manifest_failures.append(f'{rel}: sha256 mismatch')
if manifest_failures:
    raise RuntimeError('The uploaded zip failed source manifest validation: ' + '; '.join(manifest_failures[:20]))
manifest_sha256 = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
if expected_source_manifest_sha256 and manifest_sha256 != expected_source_manifest_sha256:
    raise RuntimeError('The uploaded zip manifest SHA-256 does not match expected_source_manifest_sha256.')
print('SHA-256 source manifest validated.')

provenance_path = source_dir / 'MOPAC_COLAB_SOURCE_PROVENANCE.json'
if not provenance_path.exists():
    raise RuntimeError('The uploaded zip has no source provenance. Recreate it with the hardened scripts/create_colab_gpu_zip.py.')
try:
    source_provenance = json.loads(provenance_path.read_text(encoding='utf-8'))
except json.JSONDecodeError as exc:
    raise RuntimeError(f'The uploaded zip has invalid source provenance JSON: {exc}') from exc
provenance_failures = []
if source_provenance.get('schema') != 'mopac-colab-source-provenance-v1':
    provenance_failures.append('schema')
if source_provenance.get('zip_source_tree_marker') != 'mopac-colab-gpu-proof-source-v1':
    provenance_failures.append('zip_source_tree_marker')
if source_provenance.get('feature_set') != 'mozyme-full-scf-gpu-makvec-relocal-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v38-20260702':
    provenance_failures.append('feature_set')
if source_provenance.get('full_scf_contract_version') != 'resident-scf-strict-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-resident-cg-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v58':
    provenance_failures.append('full_scf_contract_version')
if source_provenance.get('source_marker_contract_version') != 'mopac-colab-source-markers-explicit-proof-v83':
    provenance_failures.append('source_marker_contract_version')
if source_provenance.get('source_marker_contract_sha256') != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256:
    provenance_failures.append('source_marker_contract_sha256')
source_proof_eligible = source_provenance.get('proof_eligible')
source_proof_ineligible_reason = source_provenance.get('proof_ineligible_reason', '')
if not isinstance(source_proof_eligible, bool):
    provenance_failures.append('source.proof_eligible')
if not isinstance(source_proof_ineligible_reason, str):
    provenance_failures.append('source.proof_ineligible_reason')
manifest_info = source_provenance.get('manifest')
if not isinstance(manifest_info, dict):
    provenance_failures.append('manifest')
else:
    if manifest_info.get('path') != 'MOPAC_COLAB_SOURCE_MANIFEST.sha256':
        provenance_failures.append('manifest.path')
    if manifest_info.get('sha256') != manifest_sha256:
        provenance_failures.append('manifest.sha256')
    if manifest_info.get('file_count') != manifest_entries:
        provenance_failures.append('manifest.file_count')
if not source_provenance.get('generated_at_utc'):
    provenance_failures.append('generated_at_utc')
git_info = source_provenance.get('git')
if not isinstance(git_info, dict) or 'commit' not in git_info or 'dirty' not in git_info:
    provenance_failures.append('git')
else:
    git_commit = git_info.get('commit')
    if not isinstance(git_commit, str) or len(git_commit) != 40 or any(ch not in '0123456789abcdefABCDEF' for ch in git_commit):
        provenance_failures.append('git.commit')
    if expected_source_git_commit and git_commit.lower() != expected_source_git_commit.lower():
        provenance_failures.append('git.commit_expected')
    if not isinstance(git_info.get('dirty'), bool):
        provenance_failures.append('git.dirty')
    if isinstance(expected_source_git_dirty, bool) and git_info.get('dirty') is not expected_source_git_dirty:
        provenance_failures.append('git.dirty_expected')
    if expected_source_dirty_status_sha256 and git_info.get('dirty_status_sha256') != expected_source_dirty_status_sha256:
        provenance_failures.append('git.dirty_status_sha256_expected')
    if git_info.get('dirty') and source_proof_eligible is True:
        provenance_failures.append('source.proof_eligible')
    if not git_info.get('dirty') and source_proof_eligible is False:
        provenance_failures.append('source.proof_ineligible_reason')
    if git_info.get('dirty'):
        dirty_matches_expected_sidecar = bool(allow_expected_dirty_source_zip and expected_source_git_dirty is True and expected_source_dirty_status_sha256 and git_info.get('dirty_status_sha256') == expected_source_dirty_status_sha256)
        if not (allow_development_dirty_zip or dirty_matches_expected_sidecar):
            provenance_failures.append('git.dirty_proof_blocked')
        if git_info.get('dirty_allowed') is not True:
            provenance_failures.append('git.dirty_allowed')
        if not isinstance(git_info.get('dirty_entries'), list) or not git_info.get('dirty_entries'):
            provenance_failures.append('git.dirty_entries')
        if not isinstance(git_info.get('dirty_status_sha256'), str) or len(git_info.get('dirty_status_sha256')) != 64:
            provenance_failures.append('git.dirty_status_sha256')
if provenance_failures:
    raise RuntimeError('The uploaded zip failed source provenance validation: ' + ', '.join(provenance_failures))

features_path = source_dir / 'MOPAC_COLAB_FEATURES.json'
if not features_path.exists():
    raise RuntimeError('The uploaded zip has no feature manifest. Recreate it with scripts/create_colab_gpu_zip.py.')
try:
    source_features = json.loads(features_path.read_text(encoding='utf-8'))
except json.JSONDecodeError as exc:
    raise RuntimeError(f'The uploaded zip has invalid feature manifest JSON: {exc}') from exc
feature_failures = []
if source_features.get('schema') != 'mopac-colab-feature-manifest-v1':
    feature_failures.append('schema')
if source_features.get('feature_set') != 'mozyme-full-scf-gpu-makvec-relocal-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v38-20260702':
    feature_failures.append('feature_set')
if source_features.get('full_scf_contract_version') != 'resident-scf-strict-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-resident-cg-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v58':
    feature_failures.append('full_scf_contract_version')
if source_features.get('source_marker_contract_version') != 'mopac-colab-source-markers-explicit-proof-v83':
    feature_failures.append('source_marker_contract_version')
if source_features.get('source_marker_contract_sha256') != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256:
    feature_failures.append('source_marker_contract_sha256')
if source_features.get('zip_source_tree_marker') != 'mopac-colab-gpu-proof-source-v1':
    feature_failures.append('zip_source_tree_marker')
expected_source_policy = {
    'schema': SOURCE_ALLOWLIST_POLICY_SCHEMA,
    'root_files': sorted(ALLOWED_SOURCE_ROOT_FILES),
    'root_directories': sorted(ALLOWED_SOURCE_ROOT_DIRS),
}
if source_features.get('source_allowlist_policy') != expected_source_policy:
    feature_failures.append('source_allowlist_policy')
feature_manifest_info = source_features.get('manifest')
if not isinstance(feature_manifest_info, dict):
    feature_failures.append('manifest')
else:
    if feature_manifest_info.get('path') != 'MOPAC_COLAB_SOURCE_MANIFEST.sha256':
        feature_failures.append('manifest.path')
    if feature_manifest_info.get('sha256') != manifest_sha256:
        feature_failures.append('manifest.sha256')
    if feature_manifest_info.get('file_count') != manifest_entries:
        feature_failures.append('manifest.file_count')
critical_files = source_features.get('critical_files')
required_critical_files = (
    'CMakeLists.txt',
    'src/CMakeLists.txt',
    'src/run_mopac.F90',
    'src/MOZYME/CMakeLists.txt',
    'src/MOZYME/addhb.F90',
    'src/MOZYME/check.F90',
    'src/MOZYME/cnvgz.F90',
    'src/MOZYME/density_for_MOZYME.F90',
    'src/MOZYME/diagg.F90',
    'src/MOZYME/diagg1.F90',
    'src/MOZYME/diagg2.F90',
    'src/MOZYME/eimp.F90',
    'src/MOZYME/fillij.F90',
    'src/MOZYME/fock1_for_MOZYME.F90',
    'src/MOZYME/fock2z.F90',
    'src/MOZYME/helecz.F90',
    'src/MOZYME/isitsc.F90',
    'src/MOZYME/iter_for_MOZYME.F90',
    'src/MOZYME/mozyme_diagg1_state.F90',
    'src/MOZYME/mozyme_diagg2_state.F90',
    'src/MOZYME/mozyme_fock1_batch.F90',
    'src/MOZYME/mozyme_fock2_4x1_batch.F90',
    'src/MOZYME/mozyme_gpu_int_utils.F90',
    'src/MOZYME/mozyme_gpu_makvec.F90',
    'src/MOZYME/mozyme_gpu_plan.F90',
    'src/MOZYME/mozyme_gpu_relocalize.F90',
    'src/MOZYME/mozyme_gpu_reorth.F90',
    'src/MOZYME/mozyme_gpu_tidy.F90',
    'src/MOZYME/mozyme_gpu_scf_driver.F90',
    'src/MOZYME/mozyme_isitsc_state.F90',
    'src/MOZYME/mozyme_resident_fock.F90',
    'src/MOZYME/mozyme_section_timers.F90',
    'src/MOZYME/add_more_interactions.F90',
    'src/MOZYME/pinout.F90',
    'src/MOZYME/reorth.F90',
    'src/MOZYME/set_up_MOZYME_arrays.F90',
    'src/MOZYME/setupk.F90',
    'src/MOZYME/tidy.F90',
    'src/output/writmo.F90',
    'src/solvation/linear_cosmo.F90',
    'src/gpu/cuda_wrappers.cu',
    'src/gpu/cublas_interfaces.F90',
    'src/gpu/fock_kernels.cu',
    'src/gpu/grad_launch.h',
    'src/gpu/gpu_bmat_interfaces.F90',
    'src/gpu/gpu_density_interfaces.F90',
    'src/gpu/gpu_diis_interfaces.F90',
    'src/gpu/gpu_eig_mg_interfaces.F90',
    'src/gpu/gpu_fock_interfaces.F90',
    'src/gpu/gpu_grad_interfaces.F90',
    'src/gpu/gpu_hmtr_interfaces.F90',
    'src/gpu/gpu_mozyme_scf_interfaces.F90',
    'src/gpu/gpu_ortho_interfaces.F90',
    'src/gpu/gpu_runtime_interfaces.F90',
    'src/gpu/gpu_scf_interfaces.F90',
    'src/gpu/gpu_scf_stream_driver.F90',
    'src/gpu/gpu_scf_stream_interfaces.F90',
    'src/gpu/gpu_scf_stream_trace.F90',
    'src/gpu/gpu_scf_types.F90',
    'src/gpu/gpu_small_solve_interfaces.F90',
    'src/gpu/gpu_transform_interfaces.F90',
    'src/gpu/grad_kernels.cu',
    'src/gpu/hmtr_optimizer.cu',
    'src/gpu/packed_utils.h',
    'src/gpu/scf_driver.cu',
    'src/gpu/mozyme_scf_context.cu',
    'tests/CMakeLists.txt',
    'tests/gpu_bench.F90',
    'tests/gpu_resident_fock_pair_compare.F90',
    'tests/check_gpu_env_flag_parsing.py',
    'tests/check_gpu_source_build_contract.py',
    'tests/check_mozyme_strict_resident_scf.py',
    'docs/GPU_GUIDE.md',
    'scripts/collect_existing_mopac_references.py',
    'scripts/molecule_benchmark_report.py',
    'scripts/create_colab_gpu_zip.py',
    'scripts/verify_colab_gpu_proof_zip.py',
    'scripts/gpu_benchmark_report.py',
    'scripts/hydrogenate_publication_benchmark_inputs.py',
    'scripts/prepare_publication_benchmark_inputs.py',
    'colab/mopac_cublas_gpu_bench_colab.ipynb',
)
manifest_hashes = {}
for raw_line in manifest_path.read_text(encoding='utf-8').splitlines():
    line = raw_line.strip()
    if not line:
        continue
    expected_hash, rel = line.split(None, 1)
    manifest_hashes[rel] = expected_hash
if not isinstance(critical_files, dict):
    feature_failures.append('critical_files')
else:
    for rel in required_critical_files:
        info = critical_files.get(rel)
        path = source_dir / rel
        if not isinstance(info, dict):
            feature_failures.append(f'critical_files.{rel}')
            continue
        declared_hash = info.get('sha256')
        if declared_hash != manifest_hashes.get(rel):
            feature_failures.append(f'critical_files.{rel}.manifest_sha256')
        if not path.exists():
            feature_failures.append(f'critical_files.{rel}.missing')
            continue
        actual_hash = hashlib.sha256(path.read_bytes()).hexdigest()
        if declared_hash != actual_hash:
            feature_failures.append(f'critical_files.{rel}.sha256')
        if info.get('size') != path.stat().st_size:
            feature_failures.append(f'critical_files.{rel}.size')
source_marker_contract = source_features.get('required_source_markers')
if isinstance(source_marker_contract, list):
    source_marker_contract_canonical = json.dumps({
        'version': EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_VERSION,
        'required_source_markers': source_marker_contract,
    }, sort_keys=True, separators=(',', ':'))
    if hashlib.sha256(source_marker_contract_canonical.encode('utf-8')).hexdigest() != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256:
        feature_failures.append('required_source_markers.sha256')
required_source_marker_names = (
    'standalone_gpu_benchmark_target_before_tests',
    'standalone_resident_fock_pair_target_before_tests',
    'ctest_gpu_tests_are_gpu_gated',
    'gpu_env_flag_parsing_static_ctest',
    'gpu_env_flag_parsing_static_guard',
    'gpu_source_build_contract_static_ctest',
    'gpu_source_build_contract_static_guard',
    'gpu_core_source_inclusion_contract',
    'mozyme_core_source_inclusion_contract',
    'strict_resident_scf_ctest_fail_closed',
    'api_initialize_mopac_nogpu_false_values',
    'colab_zip_clean_snapshot_proof_generator',
    'strict_resident_stream_finalizer_fail_closed',
    'colab_proof_zip_verifier',
    'gpu_guide_resident_scf_abi_v33',
    'colab_source_allowlist_policy',
    'colab_strict_proof_defaults',
    'resident_sparse_fock_setup_host_bounds',
    'resident_fock_4x1_pair_compare',
    'resident_fock_diagonal_pair_validation',
    'resident_fock_integral_signature',
    'resident_scf_partial_fock_bounded_copy_back',
    'strict_full_scf_report_contract_final_resident_reorth_tidy_selmos_pls_cosmo_direct_resident_cg_point_kind_cnvgz_active_or_noop_cpu_compare_explicit_proof_v58',
    'resident_fock_point_dipole_pair_compare',
    'resident_scf_cuda_bounds_and_fail_closed',
    'resident_scf_planner_and_makvec_controls',
    'resident_scf_makvec_periodic_ione',
    'strict_resident_implies_request',
    'strict_lewis_uses_gpu_makvec',
    'strict_direct_cpu_makvec_guard',
    'strict_density_rebuild_fail_closed',
    'strict_old_scf_rejected_in_proof',
    'strict_relocal_uses_gpu',
    'mozyme_gpu_relocal_fortran_wrapper',
    'mozyme_gpu_relocal_cuda_helper',
    'strict_final_reorth_uses_resident_gpu',
    'strict_final_reorth_success_marker_requires_scf_success',
    'mozyme_gpu_reorth_fortran_wrapper',
    'mozyme_gpu_reorth_cuda_helper',
    'mozyme_gpu_tidy_wrapper_and_cuda_helper',
    'mozyme_gpu_tidy_cuda_helper',
    'mozyme_gpu_tidy_selmos_strict_path',
    'resident_final_reorth_validates_recomputed_gpu_stages',
    'strict_or_full_request_is_no_cpu_fallback',
    'strict_eimp_cpu_fallback_guard',
    'strict_cnvgz_cpu_fallback_guard',
    'strict_helecz_cpu_fallback_guard',
    'strict_isitsc_cpu_fallback_guard',
    'strict_addhb_cpu_fallback_guard',
    'strict_check_cpu_fallback_guard',
    'strict_diagg_cpu_fallback_guard',
    'strict_reorth_cpu_fallback_guard',
    'strict_tidy_cpu_fallback_guard',
    'strict_diagg1_cpu_fallback_guard',
    'strict_diagg2_cpu_fallback_guard',
    'full_resident_request_blocks_fortran_cpu_fallbacks',
    'full_resident_request_blocks_density_and_index_cpu_fallbacks',
    'full_resident_request_requires_nijbo_and_plan_work',
    'full_resident_request_blocks_plan_and_fock_fallbacks',
    'full_resident_request_blocks_run_mopac_cpu_disables',
    'full_resident_request_blocks_fock2z_cpu_fallback',
    'full_resident_request_blocks_fock1_cpu_fallback',
    'strict_resident_blocks_cpu_pinout',
    'strict_resident_blocks_cpu_interaction_promotion',
    'strict_fillij_uses_gpu_builder',
    'strict_fillij_cuda_builder',
    'strict_resident_fock_cuda_count_builder',
    'strict_resident_fock_gpu_pack_plan',
    'resident_fock_direct_d_shell_gpu_pack',
    'strict_resident_fock_cuda_pack_builder',
    'strict_resident_fock_cuda_pack_point_weight_builder',
    'strict_resident_blocks_cpu_writmo_outputs',
    'resident_fock_no_fallback_request_envs',
    'resident_scf_abi_v33_final_publication_reorth_pls_cosmo_direct',
    'resident_scf_cuda_abi_v33_publication_pls_cosmo_direct_resident_cg',
    'resident_scf_fortran_layout_kind_validation',
    'resident_scf_fortran_c_int_nfirst_nlast',
    'resident_scf_eimp_density_fail_closed',
    'resident_scf_complete_state_copyback_checkpoint',
    'resident_scf_preallocated_checkpoints',
    'strict_resident_blocks_cpu_check_and_pls',
    'strict_resident_no_partial_host_publish',
    'strict_resident_no_timing_sync',
    'resident_scf_diagg_addhb_persistent_scratch',
    'resident_scf_preallocated_stage_buffers',
    'resident_scf_no_late_stage_resize',
    'strict_resident_sparse_fock_default_stream',
    'strict_resident_fock_abort_marker',
    'strict_legacy_fock2_abort_marker',
    'resident_scf_cpu_boundary_status',
    'resident_scf_cnvgz_stage_call_normalization',
    'resident_scf_stage_status_device',
    'resident_scf_stage_runtime_trace',
    'resident_scf_setupk_all_initial_setup',
    'resident_scf_fock_density_source',
    'resident_fock_partial_point_cpu_completion',
    'resident_scf_cnvgz_device_control',
    'resident_scf_helecz_device_total',
    'resident_scf_diagg_addhb_device_control',
    'resident_scf_isitsc_device_energy_control',
    'resident_scf_iteration_accounting',
    'resident_scf_direct_wk_guard',
    'resident_scf_loop_control_device',
    'resident_scf_runtime_risk_fixes',
    'resident_fock_point_addr_preserved',
    'resident_fock_point_kind_profile',
    'resident_scf_fortran_cosmo_direct_state_v32',
    'linear_cosmo_gpu_direct_state_v32',
)
def _source_marker_present(text, flat_text, fragment):
    value = str(fragment)
    return value in text or ' '.join(value.split()) in flat_text
def _ordered_source_markers_present(text, flat_text, ordered_fragments):
    cursor = 0
    flat_cursor = 0
    for fragment in ordered_fragments:
        if not isinstance(fragment, str):
            return False
        pos = text.find(fragment, cursor)
        if pos >= 0:
            cursor = pos + len(fragment)
            flat_cursor = len(' '.join(text[:cursor].split()))
            continue
        flat_fragment = ' '.join(fragment.split())
        flat_pos = flat_text.find(flat_fragment, flat_cursor)
        if flat_pos < 0:
            return False
        flat_cursor = flat_pos + len(flat_fragment)
    return True
if not isinstance(source_marker_contract, list):
    feature_failures.append('required_source_markers')
else:
    marker_by_name = {}
    for marker in source_marker_contract:
        if isinstance(marker, dict) and isinstance(marker.get('name'), str):
            marker_by_name[marker.get('name')] = marker
    unlisted_marker_names = sorted(set(marker_by_name) - set(required_source_marker_names))
    if unlisted_marker_names:
        feature_failures.append('required_source_markers.unlisted:' + ','.join(unlisted_marker_names[:5]))
    for name in required_source_marker_names:
        marker = marker_by_name.get(name)
        if not isinstance(marker, dict):
            feature_failures.append(f'required_source_markers.{name}')
            continue
        rel = marker.get('path')
        if not isinstance(rel, str) or rel not in manifest_hashes:
            feature_failures.append(f'required_source_markers.{name}.path')
            continue
        path = source_dir / rel
        text = path.read_text(encoding='utf-8', errors='ignore')
        flat_text = ' '.join(text.split())
        fragments = marker.get('fragments')
        if not isinstance(fragments, list) or not fragments:
            feature_failures.append(f'required_source_markers.{name}.fragments')
        else:
            for fragment in fragments:
                if not _source_marker_present(text, flat_text, fragment):
                    feature_failures.append(f'required_source_markers.{name}.missing')
                    break
        ordered_fragments = marker.get('ordered_fragments') or []
        if not isinstance(ordered_fragments, list):
            feature_failures.append(f'required_source_markers.{name}.ordered_fragments')
        elif not _ordered_source_markers_present(text, flat_text, ordered_fragments):
            feature_failures.append(f'required_source_markers.{name}.ordered')
        forbidden_fragments = marker.get('forbidden_fragments') or []
        if not isinstance(forbidden_fragments, list):
            feature_failures.append(f'required_source_markers.{name}.forbidden_fragments')
        else:
            for fragment in forbidden_fragments:
                if _source_marker_present(text, flat_text, fragment):
                    feature_failures.append(f'required_source_markers.{name}.forbidden')
                    break
if feature_failures:
    raise RuntimeError('The uploaded zip failed Colab feature validation: ' + ', '.join(feature_failures[:20]))
print('Colab feature manifest validated:', source_features.get('feature_set'))
expected_source_files = manifest_paths | {
    'MOPAC_COLAB_SOURCE_MANIFEST.sha256',
    'MOPAC_COLAB_FEATURES.json',
    'MOPAC_COLAB_SOURCE_PROVENANCE.json',
}
actual_source_files = set()
extra_extracted_files = []
for path in SOURCE_ROOT.rglob('*'):
    if not path.is_file():
        continue
    try:
        rel = path.relative_to(source_dir).as_posix()
    except ValueError:
        extra_extracted_files.append(path.relative_to(SOURCE_ROOT).as_posix())
    else:
        actual_source_files.add(rel)
missing_source_files = sorted(expected_source_files - actual_source_files)
extra_source_files = sorted(actual_source_files - expected_source_files)
if missing_source_files or extra_source_files or extra_extracted_files:
    details = []
    if missing_source_files:
        details.append('missing: ' + ', '.join(missing_source_files[:20]))
    if extra_source_files:
        details.append('extra: ' + ', '.join(extra_source_files[:20]))
    if extra_extracted_files:
        details.append('outside source root: ' + ', '.join(extra_extracted_files[:20]))
    raise RuntimeError('The uploaded zip file set does not exactly match the source manifest: ' + '; '.join(details))
print('Exact source file set validated.')
print('Source provenance:')
print('  generated_at_utc:', source_provenance.get('generated_at_utc'))
print('  git_commit:', git_info.get('commit') or 'unavailable')
print('  git_dirty:', git_info.get('dirty'))
print('  git_dirty_allowed:', git_info.get('dirty_allowed'))
print('  git_dirty_status_sha256:', git_info.get('dirty_status_sha256') or 'clean')
print('  feature_set:', source_features.get('feature_set'))
print('  full_scf_contract_version:', source_features.get('full_scf_contract_version'))
print('  source_zip_sha256:', source_zip_sha256)
print('  manifest_sha256:', manifest_sha256)
print('  zip_source_tree_marker:', source_provenance.get('zip_source_tree_marker'))
os.environ['MOPAC_COLAB_SOURCE_ZIP_SHA256'] = source_zip_sha256
os.environ['MOPAC_COLAB_SOURCE_ZIP_PATH'] = str(zip_path)
os.environ['MOPAC_COLAB_SOURCE_MANIFEST_SHA256'] = manifest_sha256
os.environ['MOPAC_COLAB_SOURCE_GIT_COMMIT'] = git_info.get('commit') or ''
os.environ['MOPAC_COLAB_SOURCE_GIT_DIRTY'] = str(bool(git_info.get('dirty'))).lower()
os.environ['MOPAC_COLAB_SOURCE_DIRTY_STATUS_SHA256'] = git_info.get('dirty_status_sha256') or 'clean'
os.environ['MOPAC_COLAB_SOURCE_GENERATED_AT_UTC'] = source_provenance.get('generated_at_utc') or ''
if git_info.get('dirty'):
    print('DEVELOPMENT SOURCE: source zip was generated from a dirty git worktree with explicit --allow-dirty provenance.')

cuda_text = (source_dir / 'src/gpu/cuda_wrappers.cu').read_text(encoding='utf-8', errors='ignore')
cmake_text = (source_dir / 'CMakeLists.txt').read_text(encoding='utf-8', errors='ignore')
src_cmake_text = (source_dir / 'src/CMakeLists.txt').read_text(encoding='utf-8', errors='ignore')
mozyme_cmake_text = (source_dir / 'src/MOZYME/CMakeLists.txt').read_text(encoding='utf-8', errors='ignore')
tests_cmake_text = (source_dir / 'tests/CMakeLists.txt').read_text(encoding='utf-8', errors='ignore')
scf_driver_text = (source_dir / 'src/MOZYME/mozyme_gpu_scf_driver.F90').read_text(encoding='utf-8', errors='ignore')
iter_text = (source_dir / 'src/MOZYME/iter_for_MOZYME.F90').read_text(encoding='utf-8', errors='ignore')
makvec_text = (source_dir / 'src/MOZYME/makvec.F90').read_text(encoding='utf-8', errors='ignore')
makvec_gpu_text = (source_dir / 'src/MOZYME/mozyme_gpu_makvec.F90').read_text(encoding='utf-8', errors='ignore')
relocal_text = (source_dir / 'src/MOZYME/mozyme_gpu_relocalize.F90').read_text(encoding='utf-8', errors='ignore')
reorth_text = (source_dir / 'src/MOZYME/mozyme_gpu_reorth.F90').read_text(encoding='utf-8', errors='ignore')
tidy_gpu_text = (source_dir / 'src/MOZYME/mozyme_gpu_tidy.F90').read_text(encoding='utf-8', errors='ignore')
plan_text = (source_dir / 'src/MOZYME/mozyme_gpu_plan.F90').read_text(encoding='utf-8', errors='ignore')
plan_flat = ' '.join(plan_text.split())
fillij_text = (source_dir / 'src/MOZYME/fillij.F90').read_text(encoding='utf-8', errors='ignore')
resident_fock_text = (source_dir / 'src/MOZYME/mozyme_resident_fock.F90').read_text(encoding='utf-8', errors='ignore')
fock1_text = (source_dir / 'src/MOZYME/fock1_for_MOZYME.F90').read_text(encoding='utf-8', errors='ignore')
fock2z_text = (source_dir / 'src/MOZYME/fock2z.F90').read_text(encoding='utf-8', errors='ignore')
run_mopac_text = (source_dir / 'src/run_mopac.F90').read_text(encoding='utf-8', errors='ignore')
resident_fock_test_text = (source_dir / 'tests/gpu_resident_fock_pair_compare.F90').read_text(encoding='utf-8', errors='ignore')
gpu_env_flag_test_text = (source_dir / 'tests/check_gpu_env_flag_parsing.py').read_text(encoding='utf-8', errors='ignore')
gpu_source_contract_test_text = (source_dir / 'tests/check_gpu_source_build_contract.py').read_text(encoding='utf-8', errors='ignore')
strict_resident_scf_test_text = (source_dir / 'tests/check_mozyme_strict_resident_scf.py').read_text(encoding='utf-8', errors='ignore')
scf_interface_text = (source_dir / 'src/gpu/gpu_mozyme_scf_interfaces.F90').read_text(encoding='utf-8', errors='ignore')
add_more_text = (source_dir / 'src/MOZYME/add_more_interactions.F90').read_text(encoding='utf-8', errors='ignore')
addhb_text = (source_dir / 'src/MOZYME/addhb.F90').read_text(encoding='utf-8', errors='ignore')
check_text = (source_dir / 'src/MOZYME/check.F90').read_text(encoding='utf-8', errors='ignore')
density_text = (source_dir / 'src/MOZYME/density_for_MOZYME.F90').read_text(encoding='utf-8', errors='ignore')
eimp_text = (source_dir / 'src/MOZYME/eimp.F90').read_text(encoding='utf-8', errors='ignore')
cnvgz_text = (source_dir / 'src/MOZYME/cnvgz.F90').read_text(encoding='utf-8', errors='ignore')
helecz_text = (source_dir / 'src/MOZYME/helecz.F90').read_text(encoding='utf-8', errors='ignore')
diagg_text = (source_dir / 'src/MOZYME/diagg.F90').read_text(encoding='utf-8', errors='ignore')
diagg1_text = (source_dir / 'src/MOZYME/diagg1.F90').read_text(encoding='utf-8', errors='ignore')
diagg2_text = (source_dir / 'src/MOZYME/diagg2.F90').read_text(encoding='utf-8', errors='ignore')
isitsc_text = (source_dir / 'src/MOZYME/isitsc.F90').read_text(encoding='utf-8', errors='ignore')
setupk_text = (source_dir / 'src/MOZYME/setupk.F90').read_text(encoding='utf-8', errors='ignore')
pinout_text = (source_dir / 'src/MOZYME/pinout.F90').read_text(encoding='utf-8', errors='ignore')
reorth_cpu_text = (source_dir / 'src/MOZYME/reorth.F90').read_text(encoding='utf-8', errors='ignore')
tidy_text = (source_dir / 'src/MOZYME/tidy.F90').read_text(encoding='utf-8', errors='ignore')
writmo_text = (source_dir / 'src/output/writmo.F90').read_text(encoding='utf-8', errors='ignore')
scf_cuda_text = (source_dir / 'src/gpu/mozyme_scf_context.cu').read_text(encoding='utf-8', errors='ignore')
linear_cosmo_text = (source_dir / 'src/solvation/linear_cosmo.F90').read_text(encoding='utf-8', errors='ignore')
scf_cuda_flat = ' '.join(scf_cuda_text.split())
report_text = (source_dir / 'scripts/molecule_benchmark_report.py').read_text(encoding='utf-8', errors='ignore')
create_zip_text = (source_dir / 'scripts/create_colab_gpu_zip.py').read_text(encoding='utf-8', errors='ignore')
verify_zip_text = (source_dir / 'scripts/verify_colab_gpu_proof_zip.py').read_text(encoding='utf-8', errors='ignore')
gpu_report_text = (source_dir / 'scripts/gpu_benchmark_report.py').read_text(encoding='utf-8', errors='ignore')
gpu_bench_text = (source_dir / 'tests/gpu_bench.F90').read_text(encoding='utf-8', errors='ignore')
colab_text = (source_dir / 'colab/mopac_cublas_gpu_bench_colab.ipynb').read_text(encoding='utf-8', errors='ignore')
hardcoded_critical_source_markers = {
    'driver_no_fallback_policy': (scf_driver_text, (
        'public :: mozyme_gpu_scf_no_fallback_required',
        'public :: mozyme_gpu_scf_reset_request_state',
        'mozyme_gpu_scf_no_fallback_required = no_fallback_required',
        'blocked_nscf = -1',
        "strict_requested = env_is_one('MOPAC_MOZYME_SCF_STRICT_RESIDENT')",
        "full_scf_requested = env_is_one('MOPAC_MOZYME_SCF_GPU')",
        "resident_requested = env_is_one('MOPAC_MOZYME_RESIDENT_SCF')",
        'request_present = resident_requested .or. full_scf_requested .or.',
        'no_fallback_required = strict_requested .or. full_scf_requested',
        '.not. (strict_requested .or. full_scf_requested) .and.',
    )),
    'run_mopac_no_cpu_disable': (run_mopac_text, (
        'mozyme_gpu_scf_no_fallback_required',
        'mozyme_gpu_scf_reset_request_state',
        'MOPAC_MOZYME_SCF_GPU',
        'strict_full_gpu_required = mozyme .and. strict_full_gpu_required',
        'strict_full_gpu_required = strict_mozyme_scf',
        'Preserve ordinary MOZYME behavior unless GPU execution was requested',
        'call mozyme_gpu_scf_reset_request_state()',
        'write(*,*) "This MOPAC executable was not compiled with MDI support"\n          return',
        'call geout (iarc)\n        goto 100',
        'read(list,*,iostat=stat_env)',
        'read(keyup(pos:),*,iostat=stat_env)',
        'read(pair,*,iostat=stat_env)',
        'strict_gpu_disabled_by_nogpu_keyword',
        'strict_not_gpu_build',
        'strict_resident_fock_gpu_disabled',
        'strict_no_gpu_device',
        'strict_gpu_disabled_by_mopac_nogpu',
        'strict_gpu_disabled_by_scftask_cpu',
        'strict_gpu_disabled_by_mozyme_gpu_off',
    )),
    'fock2z_no_cpu_fallback': (fock2z_text, (
        'resident_strict_requested',
        'strict_resident_fock_cpu_fallback',
        'MOZYME GPU strict resident Fock did not complete before CPU Fock',
    )),
    'fock1_no_cpu_fallback': (fock1_text, (
        'use mozyme_gpu_scf_driver, only: mozyme_gpu_scf_no_fallback_required',
        'mozyme_gpu_scf_no_fallback_required()',
        'strict_fock1_cpu_fallback',
        'MOZYME GPU strict resident SCF does not support CPU one-center Fock construction',
    )),
    'pinout_no_cpu_fallback': (pinout_text, (
        'use mozyme_gpu_scf_driver, only: mozyme_gpu_scf_no_fallback_required',
        '[MOZYME CPU pinout]',
        'strict_pinout_cpu_fallback',
        'MOZYME GPU strict resident SCF does not support CPU pinout',
        'call mozyme_gpu_strict_abort',
    )),
    'add_more_interactions_no_cpu_fallback': (add_more_text, (
        'use mozyme_gpu_scf_driver, only: mozyme_gpu_scf_no_fallback_required',
        'external :: mozyme_gpu_strict_abort',
        'strict_add_more_interactions_cpu_fallback',
        'MOZYME GPU strict resident SCF does not support CPU interaction promotion',
        'call fillij (.false.)',
    )),
    'strict_fillij_uses_gpu_builder': (fillij_text, (
        'mopac_cuda_mozyme_fillij_count',
        'mopac_cuda_mozyme_fillij_nijbo',
        'mozyme_gpu_scf_no_fallback_required()',
        'fillij_gpu=1 count=',
        'strict_fillij_gpu_failed',
        'strict_fillij_gpu_unavailable',
    )),
    'strict_fillij_cuda_builder': (cuda_text, (
        'mozyme_fillij_kernel',
        'mopac_cuda_mozyme_fillij_count',
        'mopac_cuda_mozyme_fillij_nijbo',
        'mozyme_fillij_gpu_run',
        'g_mz_fillij_nijbo',
        'mozyme_fillij mode=%s',
    )),
    'strict_resident_fock_cuda_count_builder': (cuda_text, (
        'mozyme_resident_fock_count_kernel',
        'mopac_cuda_mozyme_resident_fock_count_plan',
        'g_mz_res_count_nijbo',
        'mozyme_resident_fock_count atoms=%d',
        'mozyme_resident_point_supported_dev',
        'kMozymeResidentFallbackBasisBins = 11',
        'mozyme_resident_fallback_basis_bin_dev',
        'if (mozyme_resident_basis_supported_dev(iab))',
        '++one_center_cpu_count;',
        'out[17] = one_center_cpu_count;',
    )),
    'strict_resident_fock_gpu_pack_plan': (resident_fock_text, (
        'mopac_cuda_mozyme_resident_fock_pack_plan',
        'gpu_count=1 plan_id=',
        'gpu_pack=1 plan_id=',
        'gpu_point_weights=1 point=',
        'source=pack',
        'mozyme_resident_direct_sp_basis_supported',
        'mozyme_resident_direct_sp_basis_supported = nbasis == 1 .or. nbasis == 4',
        'mozyme_resident_direct_basis_supported = nbasis == 1 .or. nbasis == 4 .or. nbasis == 9',
        'merge(1_c_int, 0_c_int, direct)',
        'am, ad, aq, dd, qq',
        'jindex(m) = ifact(lk) + kl',
        'strict_resident_fock_gpu_pack_failed',
        'strict_resident_fock_gpu_pack_required',
        'resident_gpu_pack_ready',
        'mopac_cuda_mozyme_sparse_fock_plan_ready',
        'use iso_c_binding, only: c_int64_t',
        'integer(c_int64_t), save :: last_signature',
        'signature = last_signature(plan_id) + 1_c_int64_t',
        'signature = mozyme_resident_signature(iorbs, nat, ifact, wj, wk,',
        'signature, gpu_pack_full_coverage',
        'point_w_values, signature, coverage_complete',
        'gpu_pack_stale=1 plan_id=',
        'gpu_pack_required=1 plan_id=',
        'resident_full_coverage(plan_id) = gpu_pack_full_coverage /= 0_c_int',
        'log_resident_coverage_gpu_counts',
    )),
    'strict_resident_fock_cuda_pack_builder': (cuda_text, (
        'mozyme_resident_fock_pack_plan_kernel',
        'mopac_cuda_mozyme_resident_fock_pack_plan',
        'mozyme_sparse_fock_invalidate_plan',
        'mopac_cuda_mozyme_sparse_fock_plan_ready',
        'int64_t signature',
        'plan->ready && plan->has_executable_work',
        'plan->signature == signature',
        'g_mz_res_pack_status',
        'g_mz_res_pack_aq',
        'g_mz_res_pack_qq',
        'g_mz_res_pack_direct_scratch',
        'kMozymeResidentDirectPackScratchDoubles',
        'kMozymeDirectSpdScratchReppdArg',
        'kMozymeDirectSpdScratchRotP',
        'mozyme_resident_pack_point_weights_dev',
        'mozyme_direct_reppd_sp_dev',
        'mozyme_direct_sp_w_dev',
        'mozyme_resident_pair_supported_for_direct_dev',
        '(direct_flag == 0 && !wk)',
        '(direct_flag == 0 && !g_mz_res_pack_wk.ensure(w_bytes))',
        'if (direct_flag == 0) {\n    code |= copy_double(g_mz_res_pack_wk, wk, w_bytes',
        'direct_flag == 0 ? g_mz_res_pack_wk.ptr : nullptr',
        'mozyme_resident_fock_pack one=%d',
    )),
    'strict_resident_fock_cuda_pack_point_weight_builder': (cuda_text, (
        'mozyme_resident_to_point_dev',
        'mozyme_resident_pack_point_weights_dev',
        'if (!mozyme_resident_pack_point_weights_dev(',
        'point_w + static_cast<size_t>(point_pos) * 7u',
        'status[0] = 6;',
    ), (
        'mozyme_resident_point_weights_kernel',
        'mopac_cuda_mozyme_resident_fock_point_weights',
        'g_mz_res_point_out',
    )),
    'writmo_no_optional_cpu_outputs': (writmo_text, (
        'use mozyme_gpu_scf_driver, only : mozyme_gpu_scf_no_fallback_required',
        'strict_mozyme_gpu_scf = mozyme .and. mozyme_gpu_scf_no_fallback_required()',
        'subroutine mozyme_gpu_strict_writmo_abort(reason)',
        'call mopend(reason)',
        "error stop 'MOZYME GPU strict writmo abort'",
        'writmo_relocal_output=skipped resident_gpu=1',
        'strict_writmo_vec_host_output',
        'strict_writmo_mecip_host_density',
        'strict_writmo_pm7ts_host_compfg',
        'strict_writmo_deriv_host_output',
        'strict_writmo_fock_host_output',
        'strict_writmo_dens_host_output',
        'writmo_pops_output=skipped resident_gpu=1',
        'strict_writmo_pi_host_output',
        'strict_writmo_spin_host_output',
        'strict_writmo_bonds_host_output',
        'strict_writmo_local_host_output',
        'strict_writmo_1ele_host_output',
        'strict_writmo_enpart_host_output',
        'strict_writmo_denout_host_output',
        'strict_writmo_mullik_host_output',
    )),
    'strict_resident_scf_ctest_fail_closed': (strict_resident_scf_test_text, (
        'MOPAC_MOZYME_SCF_STRICT_RESIDENT',
        'MOPAC_MOZYME_SCF_GPU',
        'FATAL_STATUS_RE',
        'RESIDENT_FOCK_CPU_POINT_PAIRS_RE',
        'resident_fock\\][^\\n]*\\bcpu_point_pairs',
        'SUCCESS_STATUS_RE',
        'MOZYME_REORTH_RE',
        'strict resident SCF: resident=1 final GPU REORTH success marker was not found',
        'MARKER_FIELD_RE',
        'marker_fields',
        'cnvgz_active_noop_ready',
        'STAGE_NAMES_RE',
        'STRICT_PROOF_RE',
        'RESIDENT_FOCK_PLAN_RE',
        'FINAL_DENSITY_RE',
        'HOST_COMMIT_RE',
        'MOZYME_SECTION_RE',
        'strict_disallowed_section_names',
        'stage_completed != stage_required',
        'MOZYME_SCF_STAGE_FULL = 1023',
        'MOZYME_SCF_STAGE_FULL_NAMES',
        'resident Fock plan did not prove full coverage',
        'covered_mask != required_mask',
        'resident Fock plan masks must exactly match',
        'stage_required != MOZYME_SCF_STAGE_FULL',
        'backend_cpu_boundary',
        'final_density=current_resident marker was not found',
        'host_commit_only=1 phase=final_publication marker was not found',
        'disallowed CPU MOZYME section(s) ran',
        'host .den checkpoint artifacts were produced',
    )),
    'notebook_strict_readiness_guards': (colab_text, (
        'strict_readiness_fatal_status_lines',
        'stage mask mismatch: missing=',
        'full_scf_gpu_stage_completed has extra bits',
        'run_molecule_benchmark=True requires the strict readiness probe to pass first',
        'assert_full_scf_readiness_cpu_compare_artifact',
        'full_scf_gpu_readiness_cpu_compare.json',
        '--full-scf-readiness-cpu-compare',
        'direct COSMO point-weight calls',
        'direct COSMO point-weight point',
    )),
}
hardcoded_source_failures = []
for marker_name, marker_entry in hardcoded_critical_source_markers.items():
    if len(marker_entry) == 2:
        marker_text, marker_fragments = marker_entry
        marker_forbidden_fragments = ()
    elif len(marker_entry) == 3:
        marker_text, marker_fragments, marker_forbidden_fragments = marker_entry
    else:
        hardcoded_source_failures.append(f'{marker_name}:invalid marker tuple')
        continue
    marker_flat = ' '.join(marker_text.split())
    for marker_fragment in marker_fragments:
        if not _source_marker_present(marker_text, marker_flat, marker_fragment):
            hardcoded_source_failures.append(f'{marker_name}:{marker_fragment}')
    for marker_fragment in marker_forbidden_fragments:
        if _source_marker_present(marker_text, marker_flat, marker_fragment):
            hardcoded_source_failures.append(f'{marker_name}:forbidden:{marker_fragment}')
if hardcoded_source_failures:
    raise RuntimeError('The uploaded zip failed hardcoded critical source validation: ' + ', '.join(hardcoded_source_failures[:20]))
colab_required_gpu_error_markers = (
    '[GPU ERROR]',
    'ACCURACY_FAIL',
    'BENCH_FAIL',
    'cuBLAS status',
    'cuSOLVER status',
    'CUBLAS_STATUS_',
    'CUSOLVER_STATUS_',
    'CUDA error',
    'cudaError',
    'illegal memory access',
    'device-side assert',
    'device assert',
    'misaligned address',
    'unspecified launch failure',
    'out of memory',
    'Segmentation fault',
    'SIGSEGV',
    'core dumped',
)
strict_readiness_fields = (
    'gpu_error_marker_count',
    'gpu_has_device',
    'gpu_lgpu_final',
    'mozyme_section_times',
    'mozyme_plan_resident_fock_gpu',
    'mozyme_plan_f2_gpu',
    'requires_direct_cosmo_gpu',
    'full_scf_gpu_requested',
    'full_scf_gpu_executed',
    'full_scf_gpu_ready',
    'full_scf_gpu_code',
    'full_scf_gpu_backend_ready',
    'full_scf_gpu_resident',
    'full_scf_gpu_compact_index_route',
    'full_scf_gpu_use_nijbo',
    'full_scf_gpu_stage_completed',
    'full_scf_gpu_stage_required',
    'full_scf_gpu_stage_missing',
    'full_scf_gpu_resident_decision',
    'full_scf_gpu_stage_completed_names_raw',
    'full_scf_gpu_stage_missing_names_raw',
    'full_scf_gpu_strict_resident',
    'full_scf_gpu_no_fallback_required',
    'full_scf_gpu_full_stage_mask',
    'full_scf_gpu_resident_decision_complete',
    'full_scf_gpu_strict_resident_host_syncs',
    'full_scf_gpu_strict_resident_control_polls',
    'full_scf_gpu_resident_fock_plan_id',
    'full_scf_gpu_resident_fock_plan_full_coverage',
    'full_scf_gpu_resident_fock_plan_partial_coverage',
    'full_scf_gpu_resident_fock_plan_required_mask',
    'full_scf_gpu_resident_fock_plan_covered_mask',
    'full_scf_gpu_stage_upload_calls',
    'full_scf_gpu_stage_eimp_calls',
    'full_scf_gpu_stage_diagg_calls',
    'full_scf_gpu_stage_density_calls',
    'full_scf_gpu_stage_fock_calls',
    'full_scf_gpu_stage_cnvgz_calls',
    'full_scf_gpu_stage_helecz_calls',
    'full_scf_gpu_stage_isitsc_calls',
    'full_scf_gpu_stage_addhb_calls',
    'full_scf_gpu_stage_check_calls',
    'full_scf_gpu_stage_upload_ms',
    'full_scf_gpu_stage_eimp_ms',
    'full_scf_gpu_stage_diagg_ms',
    'full_scf_gpu_stage_density_ms',
    'full_scf_gpu_stage_fock_ms',
    'full_scf_gpu_stage_cnvgz_ms',
    'full_scf_gpu_stage_helecz_ms',
    'full_scf_gpu_stage_isitsc_ms',
    'full_scf_gpu_stage_addhb_ms',
    'full_scf_gpu_stage_check_ms',
    'full_scf_gpu_isitsc_okscf',
    'full_scf_gpu_pls_supervisor_calls',
    'full_scf_gpu_pls_restart_required',
    'full_scf_gpu_pls_history_count',
    'full_scf_gpu_pls_ovmax_delta',
    'full_scf_gpu_pls_energy_delta',
    'full_scf_gpu_pls_restart_reset_device_calls',
    'full_scf_gpu_pls_restart_done',
    'full_scf_gpu_final_iterations',
    'full_scf_gpu_final_density_resident',
    'full_scf_gpu_olden_setup_only',
    'full_scf_gpu_fillij_gpu_count_calls',
    'full_scf_gpu_fillij_gpu_fill_calls',
    'full_scf_gpu_fillij_gpu_last_mpack',
    'full_scf_gpu_fillij_gpu_last_n2elec',
    'full_scf_gpu_fillij_gpu_last_ij_dim',
    'full_scf_gpu_resident_fock_gpu_count_calls',
    'full_scf_gpu_resident_fock_gpu_count_plan_id',
    'full_scf_gpu_resident_fock_gpu_count_one',
    'full_scf_gpu_resident_fock_gpu_count_pair',
    'full_scf_gpu_resident_fock_gpu_count_pair4x1',
    'full_scf_gpu_resident_fock_gpu_count_point',
    'full_scf_gpu_resident_fock_gpu_count_full_coverage',
    'full_scf_gpu_resident_fock_gpu_pack_calls',
    'full_scf_gpu_resident_fock_gpu_pack_plan_id',
    'full_scf_gpu_resident_fock_gpu_pack_one',
    'full_scf_gpu_resident_fock_gpu_pack_pair',
    'full_scf_gpu_resident_fock_gpu_pack_pair4x1',
    'full_scf_gpu_resident_fock_gpu_pack_point',
    'full_scf_gpu_resident_fock_gpu_pack_full_coverage',
    'full_scf_gpu_resident_fock_gpu_point_weight_calls',
    'full_scf_gpu_resident_fock_gpu_point_weight_point',
    'full_scf_gpu_resident_fock_gpu_point_weight_max_abs_diff',
    'full_scf_gpu_cnvgz_active_calls',
    'full_scf_gpu_cnvgz_noop_calls',
    'full_scf_gpu_cpu_mozyme_setup_only_calls',
    'full_scf_gpu_cpu_resident_fock_plan_setup_calls',
    'full_scf_gpu_cpu_resident_fock_plan_setup_plan_id',
    'full_scf_gpu_cpu_resident_fock_plan_setup_one',
    'full_scf_gpu_cpu_resident_fock_plan_setup_pair',
    'full_scf_gpu_cpu_resident_fock_plan_setup_pair4x1',
    'full_scf_gpu_cpu_resident_fock_plan_setup_point',
    'full_scf_gpu_cpu_resident_fock_plan_setup_full_coverage',
    'full_scf_gpu_host_commit_only_calls',
    'full_scf_gpu_host_commit_arrays',
    'full_scf_gpu_host_commit_bytes',
    'full_scf_gpu_host_commit_cosmo',
    'full_scf_gpu_cpu_pinout_calls',
    'full_scf_gpu_cpu_mutating_section_count',
    'full_scf_gpu_cpu_mutating_call_count',
    'full_scf_gpu_cpu_mutating_ms',
    'full_scf_gpu_device_id',
    'full_scf_probe_decision',
    'full_scf_gpu_contract_violations',
    'full_scf_gpu_scf_success_calls',
    'full_scf_gpu_scf_fallback_calls',
    'full_scf_gpu_resident_step_calls',
    'full_scf_gpu_cpu_boundary_calls',
    'mozyme_gpu_helper_fatal_marker_count',
    'mozyme_gpu_helper_fatal_markers',
    'full_scf_gpu_strict_host_route_marker_count',
    'full_scf_gpu_strict_host_route_markers',
    'full_scf_gpu_wall_ms',
    'full_scf_gpu_energy_total',
    'full_scf_gpu_cosmo_enabled',
    'full_scf_gpu_cosmo_fock_calls',
    'full_scf_gpu_cosmo_matvec_calls',
    'full_scf_gpu_cosmo_cg_iterations',
    'full_scf_gpu_cosmo_nps',
    'full_scf_gpu_cosmo_lm61',
    'full_scf_gpu_cosmo_pair_count',
    'full_scf_gpu_cosmo_solv_energy',
    'full_scf_gpu_cosmo_ediel',
    'full_scf_gpu_cosmo_last_residual',
    'full_scf_gpu_cosmo_cg_control_resident',
    'full_scf_gpu_cosmo_cg_converged',
    'full_scf_gpu_cosmo_cg_breakdown',
    'full_scf_gpu_cosmo_cg_host_syncs',
    'full_scf_gpu_cosmo_cg_target_tol',
    'full_scf_gpu_diagg_sumt',
    'full_scf_gpu_diagg_sumb',
    'mozyme_sparse_fock_run_calls',
    'mozyme_sparse_fock_run_one_tasks',
    'mozyme_sparse_fock_run_pair_tasks',
    'mozyme_sparse_fock_run_4x1_tasks',
    'mozyme_sparse_fock_run_point_tasks',
    'mozyme_sparse_fock_run_point_dipole_tasks',
    'mozyme_sparse_fock_run_point_monopole_tasks',
    'mozyme_sparse_fock_run_zero_work_calls',
    'mozyme_sparse_fock_run_ms',
    'mozyme_fock_plan_point_charge_pairs',
    'mozyme_fock_plan_point_dipole_pairs',
    'mozyme_fock_plan_point_monopole_pairs',
    'mozyme_fock_resident_supported_point_pairs',
    'mozyme_fock_resident_unsupported_point_pairs',
    'mozyme_sparse_fock_setup_point_tasks',
    'mozyme_sparse_fock_setup_point_dipole_tasks',
    'mozyme_sparse_fock_setup_point_monopole_tasks',
    'mozyme_resident_fock_coverage_mode',
    'mozyme_resident_fock_real_pairs',
    'mozyme_resident_fock_gpu_real_pairs',
    'mozyme_resident_fock_inactive_real_pairs',
    'mozyme_resident_fock_point_pairs',
    'mozyme_resident_fock_gpu_point_pairs',
    'mozyme_makvec_gpu_success_calls',
    'mozyme_makvec_gpu_existing_lmo_calls',
    'mozyme_makvec_gpu_existing_lmo_last_reason',
    'mozyme_makvec_gpu_fallback_calls',
    'mozyme_makvec_gpu_last_ms',
    'mozyme_relocal_gpu_success_calls',
    'mozyme_relocal_gpu_occupied_success_calls',
    'mozyme_relocal_gpu_virtual_success_calls',
    'mozyme_reorth_gpu_success_calls',
    'mozyme_reorth_gpu_resident_success_calls',
    'mozyme_setupk_gpu_success_calls',
    'mozyme_setupk_gpu_fallback_calls',
    'mozyme_setupk_gpu_initial_setup_success_calls',
    'mozyme_setupk_gpu_initial_setup_fallback_calls',
    'mozyme_setupk_gpu_initial_setup_all_paths_calls',
    'mozyme_setupk_gpu_initial_setup_last_ms',
)
resident_fock_fallback_fields = (
    'mozyme_resident_fock_cpu_real_pairs',
    'mozyme_resident_fock_fallback_pairs',
    'mozyme_resident_fock_basis_limit_fallback_pairs',
    'mozyme_resident_fock_other_fallback_pairs',
    'mozyme_resident_fock_cpu_point_pairs',
    'mozyme_resident_fock_basis_limit_point_fallback_pairs',
    'mozyme_resident_fock_other_point_fallback_pairs',
)
stage_fallback_fields = (
    'full_scf_gpu_fallback',
    'full_scf_gpu_scf_fallback_calls',
    'full_scf_gpu_resident_step_calls',
    'full_scf_gpu_cpu_boundary_calls',
    'mozyme_gpu_helper_fatal_marker_count',
    'mozyme_gpu_helper_fatal_markers',
    'mozyme_makvec_gpu_fallback_calls',
    'mozyme_relocal_gpu_fallback_calls',
    'mozyme_reorth_gpu_fallback_calls',
    'mozyme_setupk_gpu_fallback_calls',
    'mozyme_setupk_gpu_initial_setup_fallback_calls',
    'density_cpu_diag_blocks',
    'density_cpu_offdiag_blocks',
    'density_batch_gpu_fallback_calls',
    'mozyme_eimp_gpu_fallback_calls',
    'mozyme_diagg1_aocc_gpu_fallback_calls',
    'mozyme_diagg1_avir_gpu_fallback_calls',
    'mozyme_diagg1_construct_gpu_fallback_calls',
    'mozyme_diagg2_rotprep_gpu_fallback_calls',
    'mozyme_diagg2_rotate_gpu_fallback_calls',
    'mozyme_isitsc_gpu_fallback_calls',
    'mozyme_cnvgz_gpu_fallback_calls',
    'mozyme_helecz_gpu_fallback_calls',
    'mozyme_fock1_batch_gpu_fallback_calls',
    'mozyme_fock1_batch_gpu_fallback_tasks',
    'mozyme_fock1_batch_gpu_fallback_pairs',
    'mozyme_fock2_4x1_batch_gpu_fallback_calls',
    'mozyme_fock2_4x1_batch_gpu_fallback_tasks',
    'mozyme_fock1_gpu_fallback_seen',
    'mozyme_fock2_gpu_fallback_seen',
)
tests_block_match = re.search(r'(?mi)^\s*if\s*\(\s*TESTS\b', cmake_text)
tests_block_index = tests_block_match.start() if tests_block_match else -1

def cmake_if_block_ranges(text, condition):
    pattern = re.compile(r'^\s*(if|endif)\s*\(([^)]*)\)', re.IGNORECASE | re.MULTILINE)
    stack = []
    ranges = []
    condition_norm = ' '.join(condition.lower().split())
    for match in pattern.finditer(text):
        keyword = match.group(1).lower()
        expr_norm = ' '.join(match.group(2).lower().split())
        if keyword == 'if':
            stack.append(match.end() if expr_norm == condition_norm else None)
        elif keyword == 'endif':
            start = stack.pop() if stack else None
            if start is not None:
                ranges.append((start, match.start()))
    return ranges

def cmake_if_block_range(text, condition):
    ranges = cmake_if_block_ranges(text, condition)
    return ranges[0] if ranges else None

def cmake_gpu_target_before_tests(target_name, source_name):
    target_signature = f'add_executable({target_name} {source_name})'
    target_index = cmake_text.find(target_signature)
    gpu_blocks = cmake_if_block_ranges(cmake_text, 'GPU')
    if target_index < 0:
        return False
    if cmake_text.count(target_signature) != 1:
        return False
    if not any(block_start <= target_index < block_end for block_start, block_end in gpu_blocks):
        return False
    if tests_block_index >= 0 and target_index > tests_block_index:
        return False
    return True

def source_region_between(text, start_fragment, end_fragment):
    start = text.find(start_fragment)
    if start < 0:
        return ''
    end = text.find(end_fragment, start + len(start_fragment))
    if end < 0:
        return ''
    return text[start:end]

def final_reorth_commit_flag_scoped(text):
    body = source_region_between(
        text,
        'bool apply_final_reorth_on_gpu(',
        '\nbool compute_initial_setup_on_gpu(',
    )
    declaration = 'bool device_final_reorth_committed = false;'
    assignment = 'device_final_reorth_committed = true;'
    decision = 'ok || device_final_reorth_committed'
    if not body:
        return False
    declaration_index = body.find(declaration)
    assignment_index = body.find(assignment)
    decision_index = body.find(decision)
    return (
        text.count(declaration) == 1
        and body.count(declaration) == 1
        and declaration_index >= 0
        and assignment_index > declaration_index
        and decision_index > assignment_index
    )

needles = [
    ('resident final reorth commit flag scoped', final_reorth_commit_flag_scoped(scf_cuda_text) and 'final_reorth_commit_flag_scoped' in gpu_source_contract_test_text and 'final_reorth_commit_flag_scoped' in create_zip_text),
    ('standalone GPU benchmark target', cmake_gpu_target_before_tests('mopac-gpu-bench', 'tests/gpu_bench.F90')),
    ('resident sparse setup', 'mopac_cuda_mozyme_sparse_fock_setup' in cuda_text),
    ('dense GPU benchmark hard-fail guards', 'GPU_ERROR_MARKERS' in gpu_report_text and 'BENCH_FAIL' in gpu_bench_text and '[GPU ERROR]' in cuda_text and 'poison_host_doubles' in cuda_text and 'report_cusolver_error' in cuda_text),
    ('direct Colab and molecule readiness GPU error guards', 'GPU_ERROR_MARKERS' in report_text and 'gpu_error_markers_in_text' in report_text and 'gpu_error_markers' in report_text and 'GPU_ERROR_MARKERS' in globals() and all(marker in GPU_ERROR_MARKERS for marker in colab_required_gpu_error_markers) and all(marker in report_text for marker in colab_required_gpu_error_markers)),
    ('run_mopac has no raw stop', re.search(r'(?m)^\\s*stop\\b', run_mopac_text) is None),
    ('run_mopac preserves ordinary MOZYME GPU opt-in', 'Preserve ordinary MOZYME behavior unless GPU execution was requested' in run_mopac_text and 'If MOZYME is active and GPU is enabled, default to MOZYME GPU unless explicitly disabled' not in run_mopac_text),
    ('resident sparse run', 'mopac_cuda_mozyme_sparse_fock_run' in cuda_text),
    ('resident sparse generic d-pair support', cmake_gpu_target_before_tests('mopac-gpu-resident-fock-pair-compare', 'tests/gpu_resident_fock_pair_compare.F90') and 'run_pair4x1_case(.true.' in resident_fock_test_text and 'run_pair4x1_case(.false.' in resident_fock_test_text and 'apply_pair4x1_cpu' in resident_fock_test_text and 'run_case(1, 1' in resident_fock_test_text and 'run_case(1, 9' in resident_fock_test_text and 'run_case(9, 1' in resident_fock_test_text and 'run_case(4, 9' in resident_fock_test_text and 'run_case(9, 4' in resident_fock_test_text and 'run_case(9, 9' in resident_fock_test_text and 'run_diag_case(4' in resident_fock_test_text and 'run_diag_case(9' in resident_fock_test_text and 'run_point_case(1, 9, -2' in resident_fock_test_text and 'run_point_case(9, 1, -2' in resident_fock_test_text and 'mozyme_resident_basis_supported' in resident_fock_text and 'nbasis == 1 .or. nbasis == 4 .or. nbasis == 9' in resident_fock_text and 'mozyme_resident_direct_basis_supported = nbasis == 1 .or. nbasis == 4 .or. nbasis == 9' in resident_fock_text and 'mozyme_direct_spd_w_dev' in cuda_text and 'mozyme_direct_reppd2_rep_dev' in cuda_text and 'mozyme_mndod_ind2_dev' in cuda_text and 'mozyme_mndod_isym_dev' in cuda_text and 'g_mz_res_pack_direct_scratch' in cuda_text and 'kMozymeResidentDirectPackScratchDoubles' in cuda_text and 'kMozymeDirectSpdScratchReppdArg' in cuda_text and 'kMozymeDirectSpdScratchRotP' in cuda_text and 'method_pm7_flag' in cuda_text and 'if (direct_flag != 0 && semidr_flag == 0) return 4;' not in cuda_text and 'double direct_w[2025]' not in cuda_text and 'double direct_w[100]' not in cuda_text and 'double arg[72]' not in cuda_text and 'double sqr[72]' not in cuda_text and 'double p[3][3]' not in cuda_text and 'double d[5][5]' not in cuda_text and 'bool logv' not in cuda_text and 'integer(c_int64_t) function mozyme_resident_signature' in resident_fock_text and 'mozyme_resident_integral_signature_hash(iorbs, wj, wk, use_nijbo, h1, h2)' in resident_fock_text and 'mozyme_integral_slice_signature_hash(wk, kr + 1, term_count, 43, h1, h2)' in resident_fock_text and 'plan->signature == signature' in cuda_text and 'pair_diag_flags[idx] == 0 &&' in cuda_text and 'pair_diag_flags[idx] == 1 && pair_cross_offsets[idx] < 1' in cuda_text and 'mozyme_sparse_fock_basis_supported' in cuda_text and '!plan->full_coverage' in cuda_text and 'has_executable_work' in cuda_text and 'max_resident_diag_basis = 9' in resident_fock_text and 'if (!(iab == 1 && jba == 1)) return;' not in cuda_text),
    ('resident SCF planner scoped branch', all(_source_marker_present(plan_text, plan_flat, fragment) for fragment in ("if (mozyme_resident_fock_gpu) then\n        resident_scf = .true.\n        gpu_scf_stream_available = .true.\n      else\n        lgpu = .false.\n        resident_scf = .false.\n        gpu_scf_stream_available = .false.\n      end if", "mozyme_gpu_enabled = mozyme_resident_fock_gpu .or. mozyme_fock1_batch_gpu .or. mozyme_fock2_4x1_batch_gpu .or. &\n      (mozyme_gpu .and. lgpu)", '.not. (mozyme_fock_gpu .and. mozyme_f2_gpu) .and. eligible_density_pairs == 0', 'fock_resident_full_coverage_planned = &', 'fock_resident_supported_one_center', 'fock_resident_unsupported_one_center', 'fock_resident_unsupported_one_center == 0 .and. &', 'fock_resident_executable_tasks = fock_resident_supported_one_center + fock_resident_supported_pairs + &', 'fock_production_gpu_tasks = fock_resident_executable_tasks', 'if (mozyme_fock_gpu .and. mozyme_f2_gpu) fock_production_gpu_tasks = fock_candidate_gpu_tasks', 'if (.not. (mozyme_fock_gpu .and. mozyme_f2_gpu)) then', 'mozyme_fock_gpu .and. mozyme_f2_gpu', "'f2_gpu=', mozyme_f2_gpu", 'fock_resident_supported_point_pairs', 'fock_resident_unsupported_point_pairs', 'fock_resident_direct_basis_pairs', 'fock_resident_direct_basis_point_pairs', 'fock_resident_noop_pairs', 'fock_resident_full_coverage_planned', 'fock_resident_executable_tasks', 'mozyme_plan_resident_basis_supported', 'mozyme_plan_resident_basis_supported(iab) .and. &', 'nbasis == 1 .or. nbasis == 4 .or. nbasis == 9', 'mozyme_plan_resident_direct_basis_supported', 'mozyme_plan_resident_direct_basis_supported(iab) .and. &', 'mozyme_plan_resident_direct_basis_supported = nbasis == 1 .or. nbasis == 4 .or. nbasis == 9', 'mozyme_plan_resident_direct_basis_fallback', 'mozyme_plan_resident_point_supported', 'if (direct) then', '[MOZYME GPU plan] direct=', 'fock_resident_direct_unsupported_pairs', 'fock_resident_direct_unsupported_point_pairs', 'fock_point_dipole_pairs', 'fock_point_monopole_pairs', 'resident_one_center supported=')) and not _source_marker_present(plan_text, plan_flat, "if (mozyme_resident_fock_gpu) then\n        lgpu = .false.") and not _source_marker_present(plan_text, plan_flat, "iab > 0 .and. jba > 0 .and. iab <= 9 .and. jba <= 9")),
    ('resident sparse point-charge/dipole compare', 'resident sparse point-charge/dipole 9-orbital comparison' in resident_fock_test_text and 'run_point_case(9, 4, -2' in resident_fock_test_text and 'run_point_case(9, 9, -2' in resident_fock_test_text and 'run_point_case(9, 9, -1' in resident_fock_test_text and 'apply_point_cpu' in resident_fock_test_text),
    ('strict helper aliases and direct CPU guards', 'MOPAC_MOZYME_GPU_STRICT' in makvec_gpu_text and 'MOPAC_MOZYME_FULL_SCF_GPU' in makvec_gpu_text and 'MOPAC_MOZYME_GPU_STRICT' in relocal_text and 'MOPAC_MOZYME_FULL_SCF_GPU' in relocal_text and 'MOPAC_MOZYME_GPU_STRICT' in reorth_text and 'MOPAC_MOZYME_FULL_SCF_GPU' in reorth_text and 'strict_addhb_cpu_fallback' in addhb_text and 'strict_check_cpu_fallback' in check_text and 'strict_check_gpu_host_fallback' in check_text and 'strict_cpu_makvec_direct' in makvec_text and 'strict_density_direct_host_rebuild' in density_text and 'strict_diagg_cpu_fallback' in diagg_text and 'strict_fock1_cpu_fallback' in fock1_text and 'strict_reorth_cpu_fallback' in reorth_cpu_text and 'strict_tidy_cpu_fallback' in tidy_text and "env_is_one('MOPAC_MOZYME_SCF_GPU')" in makvec_gpu_text and "env_is_one('MOPAC_MOZYME_RESIDENT_SCF')" in makvec_gpu_text and "env_is_one('MOPAC_MOZYME_SCF_GPU')" in relocal_text and "env_is_one('MOPAC_MOZYME_RESIDENT_SCF')" in relocal_text and "env_is_one('MOPAC_MOZYME_SCF_GPU')" in reorth_text and "env_is_one('MOPAC_MOZYME_RESIDENT_SCF')" in reorth_text),
    ('gpu env flag parsing static guard', 'gpu-env-flag-parsing' in tests_cmake_text and 'check_gpu_env_flag_parsing.py' in tests_cmake_text and 'MOPAC_FASTGPU' in gpu_env_flag_test_text and 'MOPAC_RESIDENT_SCF' in gpu_env_flag_test_text and 'MOPAC_MOZYME_RESIDENT_FOCK_GPU' in gpu_env_flag_test_text and 'MOPAC_GPU_AUTOPOLICY_OFF' in gpu_env_flag_test_text and 'MOPAC_GPU_GRAD_EXPERIMENTAL' in gpu_env_flag_test_text and 'TRUE_TOKENS_FORTRAN' in gpu_env_flag_test_text and 'FALSE_TOKENS_FORTRAN' in gpu_env_flag_test_text and 'REQUIRED_FRAGMENTS' in gpu_env_flag_test_text and 'FORBIDDEN_REGEXES' in gpu_env_flag_test_text and 'env_truthy_ci(skip)' in gpu_env_flag_test_text),
    ('gpu source build contract static guard', 'gpu-source-build-contract' in tests_cmake_text and 'check_gpu_source_build_contract.py' in tests_cmake_text and 'GPU_CORE_SOURCES' in gpu_source_contract_test_text and 'CUDA_LANGUAGE_SOURCES' in gpu_source_contract_test_text and 'MOZYME_CORE_MODULES' in gpu_source_contract_test_text and 'target_sources(mopac-core PRIVATE ${{CMAKE_CURRENT_SOURCE_DIR}}' in gpu_source_contract_test_text and 'PROPERTIES LANGUAGE CUDA' in gpu_source_contract_test_text and 'SOURCE_MARKER_CONTRACT_SHA256' in gpu_source_contract_test_text and 'posix_path.parts' in gpu_source_contract_test_text and 'mozyme_final_reorth_status_kernel' in gpu_source_contract_test_text and 'device_final_reorth_committed' in gpu_source_contract_test_text and 'strict resident Fock pack must use GPU-authored setup/counts' in gpu_source_contract_test_text and 'resident stream finalizer must fail closed on strict Fock cache miss' in gpu_source_contract_test_text and 'strict resident SCF ABI v33 must reject host sync/control polling in proof mode' in gpu_source_contract_test_text and 'target_sources(mopac-core PRIVATE ${CMAKE_CURRENT_SOURCE_DIR}/gpu/mozyme_scf_context.cu)' in src_cmake_text and 'target_sources(mopac-core PRIVATE ${CMAKE_CURRENT_SOURCE_DIR}/gpu/gpu_mozyme_scf_interfaces.F90)' in src_cmake_text and 'mozyme_gpu_scf_driver' in mozyme_cmake_text and 'mozyme_resident_fock' in mozyme_cmake_text and 'fillij' in mozyme_cmake_text),
    ('strict resident SCF CTest fail-closed', 'gpu-mozyme-strict-resident-scf' in tests_cmake_text and 'check_mozyme_strict_resident_scf.py' in tests_cmake_text and 'MOPAC_MOZYME_SCF_STRICT_RESIDENT' in strict_resident_scf_test_text and 'MOPAC_MOZYME_SCF_GPU' in strict_resident_scf_test_text and 'FATAL_STATUS_RE' in strict_resident_scf_test_text and 'HELPER_FATAL_STATUS_RE' in strict_resident_scf_test_text and 'FOCK_FAMILY_FALLBACK_RE' in strict_resident_scf_test_text and 'RESIDENT_FOCK_FALLBACK_REAL_PAIRS_RE' in strict_resident_scf_test_text and 'RESIDENT_FOCK_CPU_POINT_PAIRS_RE' in strict_resident_scf_test_text and 'SUCCESS_STATUS_RE' in strict_resident_scf_test_text and 'MOZYME_REORTH_RE' in strict_resident_scf_test_text and 'strict resident SCF: resident=1 final GPU REORTH success marker was not found' in strict_resident_scf_test_text and 'GPU_ERROR_MARKERS' in strict_resident_scf_test_text and 'CNVGZ_ACTIVITY_RE' in strict_resident_scf_test_text and 'RESIDENT_DECISION_RE' in strict_resident_scf_test_text and 'FINAL_PUBLICATION_RE' in strict_resident_scf_test_text and 'stage_completed != stage_required' in strict_resident_scf_test_text and 'resident_decision must be CompleteAndPublish' in strict_resident_scf_test_text and 'MOZYME_SCF_STAGE_FULL = 1023' in strict_resident_scf_test_text and 'stage_required != MOZYME_SCF_STAGE_FULL' in strict_resident_scf_test_text and 'final_density=current_resident marker was not found' in strict_resident_scf_test_text and 'host_commit_only=1 phase=final_publication marker was not found' in strict_resident_scf_test_text and 'final_publication_done typed marker was not found' in strict_resident_scf_test_text and 'final_publication_done marker did not prove final publication' in strict_resident_scf_test_text and 'disallowed CPU MOZYME section(s) ran' in strict_resident_scf_test_text and 'host .den checkpoint artifacts were produced' in strict_resident_scf_test_text),
    ('strict resident SCF CTest explicit proof markers', 'STAGE_NAMES_RE' in strict_resident_scf_test_text and 'STRICT_PROOF_RE' in strict_resident_scf_test_text and 'RESIDENT_FOCK_PLAN_RE' in strict_resident_scf_test_text and 'FINAL_DENSITY_RE' in strict_resident_scf_test_text and 'HOST_COMMIT_RE' in strict_resident_scf_test_text and 'FINAL_PUBLICATION_RE' in strict_resident_scf_test_text and 'MOZYME_SECTION_RE' in strict_resident_scf_test_text and 'MOZYME_SCF_STAGE_FULL_NAMES' in strict_resident_scf_test_text and 'resident Fock plan did not prove full coverage' in strict_resident_scf_test_text and 'covered_mask != required_mask' in strict_resident_scf_test_text and 'resident Fock plan masks must exactly match' in strict_resident_scf_test_text),
    ('resident sparse plan prepared before resident SCF', 'mozyme_resident_fock_prepare' in resident_fock_text and 'mozyme_resident_fock_prepare' in scf_driver_text and 'detail=resident_fock_setup' in scf_driver_text),
    ('resident point addr preservation', 'point_addr_flags(point_pos) = mozyme_c_int_checked(addr)' in resident_fock_text and 'point_addr_flags(point_pos) = mozyme_c_int_nonnegative_or_zero(addr)' not in resident_fock_text),
    ('point task profiling', 'point=%d point_dipole=%d point_monopole=%d' in cuda_text and 'if (addr_flag >= 0) return fail_setup(3);' in cuda_text and 'mozyme_sparse_fock_basis_supported' in cuda_text),
    ('resident point parser', 'mozyme_sparse_fock_run_point_tasks' in report_text and 'mozyme_sparse_fock_run_point_dipole_tasks' in report_text and 'mozyme_sparse_fock_run_point_monopole_tasks' in report_text and 'mozyme_sparse_fock_setup_point_monopole_tasks' in report_text and 'point_monopole=(\\d+)' in report_text and 'validate_resident_point_charge_coverage' in report_text and 'mozyme_resident_fock_cpu_point_pairs' in report_text and 'MOZYME_RESIDENT_FOCK_CPU_POINT_PAIRS_RE' in report_text and 'resident_fock\\][^\\n]*\\bcpu_point_pairs' in report_text and 'MOZYME_RESIDENT_FOCK_POINT_COVERAGE_RE' in report_text and 'parse_resident_fock_point_coverage' in report_text),
    ('MOZYME section parser', 'MOZYME_SECTION' in report_text and 'mozyme_section_times.csv' in report_text),
    ('strict host route marker coverage', 'STRICT_HOST_ROUTE_MARKERS' in report_text and 'strict_setup_mozyme_arrays_cpu_setup' in report_text and 'strict_resident_fock_cpu_plan_setup' in report_text and 'strict_resident_fock_gpu_count_setup_failed' in report_text and 'strict_resident_fock_gpu_count_mismatch' in report_text and 'strict_resident_fock_gpu_pack_failed' in report_text and 'strict_resident_fock_gpu_pack_mismatch' in report_text and 'strict_fillij_gpu_failed' in report_text and 'strict_fillij_gpu_unavailable' in report_text and 'strict_fillij_nijbo_missing' in report_text and 'strict_add_more_interactions_cpu_fallback' in report_text and 'strict_pinout_cpu_fallback' in report_text and 'strict_nijbo_alloc_failed' in report_text and 'strict_no_gpu_work' in report_text and 'strict_cpu_iteration_work' in report_text and 'strict_addhb_cpu_fallback' in report_text and 'strict_check_cpu_fallback' in report_text and 'strict_check_gpu_host_fallback' in report_text and 'strict_density_direct_host_rebuild' in report_text and 'strict_cpu_makvec_direct' in report_text and 'strict_diagg_cpu_fallback' in report_text and 'strict_fock1_cpu_fallback' in report_text and 'strict_fock2z_cpu_fallback' in report_text and 'strict_fock2_4x1_batch_cpu_fallback' in report_text and 'strict_buildf_cpu_fallback' in report_text and 'strict_setupk_cpu_fallback' in report_text and 'strict_cnvgz_cpu_fallback' in report_text and 'strict_eimp_cpu_fallback' in report_text and 'strict_helecz_cpu_fallback' in report_text and 'strict_isitsc_cpu_fallback' in report_text and 'strict_diagg1_cpu_fallback' in report_text and 'strict_diagg2_cpu_fallback' in report_text and 'strict_density_batch_fallback' in report_text and 'strict_reorth_cpu_fallback' in report_text and 'strict_tidy_cpu_fallback' in report_text and 'strict_run_mopac_olden_host_restore' in report_text and 'strict_writmo_reloc_host_output' in report_text and 'strict_writmo_mecip_host_density' in report_text and 'strict_writmo_pm7ts_host_compfg' in report_text and 'strict_writmo_deriv_host_output' in report_text and 'strict_writmo_denout_host_output' in report_text),
    ('full SCF report filters unverified GPU rows', 'MOPAC_MOZYME_FULL_SCF_GPU=1' in report_text and 'MOPAC_MOZYME_GPU_STRICT=1' in report_text and 'env["MOPAC_MOZYME_FULL_SCF_GPU"] = "1"' in report_text and 'env["MOPAC_MOZYME_GPU_STRICT"] = "1"' in report_text and 'MOPAC_MOZYME_SCF_GPU=1' in report_text and 'env["MOPAC_MOZYME_SCF_GPU"] = "1"' in report_text and 'MOZYME_SCF_STRICT_ABORT_REASON_RE' in report_text and 'mode == "GPU" and parse_int_value(row.get("full_scf_gpu_requested")) == 1' in report_text and 'parse_int_value(row.get("full_scf_gpu_ready")) != 1' in report_text and 'full_scf_gpu_contract_violation_reasons(row)' in report_text),
    ('strict full SCF readiness CPU companion compare', '--full-scf-readiness-cpu-compare' in report_text and 'run_full_scf_readiness_cpu_compare' in report_text and 'full_scf_gpu_readiness_cpu_compare.json' in report_text and 'strict_full_scf_readiness_cpu_companion' in report_text and 'strict readiness CPU/GPU heat comparison failed' in report_text and 'strict readiness CPU/GPU heat comparison cannot pass on' in report_text and 'per-atom tolerance alone' in report_text and 'DEFAULT_FULL_SCF_READINESS_CPU_COMPARE_ABS_TOL = 5.0e-3' in report_text and 'DEFAULT_FULL_SCF_READINESS_CPU_COMPARE_REL_TOL = 1.0e-5' in report_text and 'DEFAULT_FULL_SCF_READINESS_CPU_COMPARE_PER_ATOM_TOL = 5.0e-4' in report_text and 'direct COSMO proof planned point-charge/dipole Fock work but did not run' in report_text and 'FULL_SCF_READINESS_CPU_COMPARE_BINDING_FIELDS' in report_text and 'readiness_binding' in report_text and 'cpu_companion_gpu_leakage_reasons' in report_text and 'cpu_only_reasons' in report_text and 'CPU companion reported GPU work units' in report_text and 'CPU companion finished with lgpu=T' in report_text and '--require-full-scf-gpu requires --full-scf-readiness-cpu-compare' in report_text and 'assert_full_scf_readiness_cpu_compare_artifact' in colab_text and 'cpu_only_reasons' in colab_text and 'readiness_binding=missing' in colab_text and colab_text.count('--full-scf-readiness-cpu-compare') >= 3 and colab_text.count('full_scf_gpu_readiness_cpu_compare.json') >= 4),
    ('strict proof explicit markers', 'MOZYME_SCF_STAGE_NAMES_RAW_RE' in report_text and 'MOZYME_SCF_STRICT_PROOF_RE' in report_text and 'MOZYME_SCF_RESIDENT_FOCK_PLAN_RE' in report_text and 'full_scf_gpu_stage_completed_names_raw' in report_text and 'full_scf_gpu_strict_resident' in report_text and 'full_scf_gpu_resident_fock_plan_full_coverage' in report_text and 'full_scf_gpu_resident_fock_plan_required_mask' in report_text and 'full_scf_gpu_resident_fock_plan_covered_mask' in report_text and 'strict_proof' in scf_driver_text and 'resident_fock_plan_full_coverage' in scf_driver_text and 'resident_fock_plan_required_mask' in scf_driver_text and 'backend_resident_fock_partial_coverage' in scf_driver_text),
    ('source marker contract v83', 'SOURCE_MARKER_CONTRACT_VERSION = "mopac-colab-source-markers-explicit-proof-v83"' in create_zip_text and 'SOURCE_MARKER_CONTRACT_SHA256 = "17aa11cd4550db1b4d2bbf2f4b15bfc13b9c6e06faad28f6609ee42c7ab69240"' in create_zip_text and 'source_marker_contract_sha256' in create_zip_text and 'source_marker_contract_sha256' in colab_text and 'source_marker_contract_version' in create_zip_text and 'source_marker_contract_version' in colab_text and 'proof_eligible' in create_zip_text and 'proof_ineligible_reason' in create_zip_text and 'path.is_symlink()' in create_zip_text and 'Refusing to package symlinked source files' in create_zip_text and 'path.resolve().relative_to(root_resolved)' in create_zip_text),
    ('Colab proof zip verifier', 'Validate a MOZYME GPU Colab proof source zip before upload.' in verify_zip_text and 'SOURCE_MANIFEST_NAME = "MOPAC_COLAB_SOURCE_MANIFEST.sha256"' in verify_zip_text and 'SOURCE_MARKER_CONTRACT_SHA256 = "17aa11cd4550db1b4d2bbf2f4b15bfc13b9c6e06faad28f6609ee42c7ab69240"' in verify_zip_text and 'def safe_zip_member(name: str) -> bool:' in verify_zip_text and '".." not in posix_path.parts' in verify_zip_text and 'def verify_manifest(' in verify_zip_text and 'zip member set does not match manifest' in verify_zip_text and 'def source_marker_contract_sha256(markers: list[dict[str, Any]]) -> str:' in verify_zip_text and 'required_source_markers sha256 mismatch' in verify_zip_text and 'def verify_source_markers(' in verify_zip_text and 'notebook contains stale unsafe-path substring check' in verify_zip_text and 'Colab proof zip verification passed:' in verify_zip_text),
    ('resident Fock GPU plan cache invalidation', 'mopac_cuda_mozyme_sparse_fock_plan_ready' in resident_fock_text and 'gpu_pack_stale=1 plan_id=' in resident_fock_text and 'mozyme_sparse_fock_invalidate_plan' in cuda_text and 'plan->ready && plan->has_executable_work' in cuda_text and 'plan->signature == signature' in cuda_text),
    ('strict full SCF readiness contract final resident reorth PLS-reset COSMO-direct resident-CG point-kind CPU-compare explicit-proof v58', 'resident-scf-strict-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-resident-cg-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v58' in report_text and 'mozyme-full-scf-gpu-makvec-relocal-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v38-20260702' in report_text and 'molecule_mozyme_full_scf_gpu' in report_text and 'publication_claim' in report_text and 'MOZYME section profile markers were not present' in report_text and 'raw output contained GPU error marker' in report_text and 'MOPAC returned non-zero exit code' in report_text and 'MOPAC did not finish normally with a parsed heat of formation' in report_text and 'no resident-SCF fallback_cpu, strict_abort, or resident_step marker may appear anywhere in the log' in report_text and 'full_scf_gpu_scf_fallback_calls' in report_text and 'full_scf_gpu_resident_step_calls' in report_text and 'full_scf_gpu_cpu_boundary_calls' in report_text and 'STRICT_HOST_ROUTE_MARKERS' in report_text and 'full_scf_gpu_strict_host_route_markers' in report_text and 'full_scf_gpu_strict_host_route_marker_count' in report_text and 'backend_cpu_boundary' in report_text and 'backend_pls_restart_required' in report_text and 'full_scf_gpu_pls_restart_required' in report_text and 'resident PLS restart must be unnecessary or completed on GPU' in report_text and 'full_scf_gpu_pls_restart_required must finish at 0' in report_text and 'MOZYME_SCF_PLS_RESET_RE' in report_text and 'MOZYME_SCF_COMPACT_RE' in report_text and 'MOZYME_SCF_COSMO_CG_RE' in report_text and 'full_scf_gpu_use_nijbo' in report_text and 'full_scf_gpu_compact_index_route' in report_text and 'full_scf_gpu_pls_restart_done' in report_text and 'full_scf_gpu_pls_restart_reset_device_calls' in report_text and 'full_scf_gpu_cosmo_fock_calls' in report_text and 'full_scf_gpu_cosmo_matvec_calls' in report_text and 'full_scf_gpu_cosmo_last_residual' in report_text and 'full_scf_gpu_cosmo_cg_control_resident' in report_text and 'full_scf_gpu_cosmo_cg_converged' in report_text and 'full_scf_gpu_cosmo_cg_breakdown' in report_text and 'full_scf_gpu_cosmo_cg_host_syncs' in report_text and 'full_scf_gpu_cosmo_cg_target_tol' in report_text and 'validate_full_scf_cosmo_fields' in report_text and 'validate_direct_cosmo_contract' in report_text and 'validate_resident_plan_coverage' in report_text and 'mozyme_plan_direct_mode' in report_text and 'mozyme_fock_resident_full_coverage_planned' in report_text and 'mozyme_fock_resident_executable_tasks' in report_text and 'mozyme_fock_resident_direct_unsupported_pairs' in report_text and 'mozyme_fock_resident_direct_unsupported_point_pairs' in report_text and 'mozyme_resident_fock_direct_basis_fallback_pairs' in report_text and 'direct COSMO proof encountered resident Fock direct-basis fallback' in report_text and 'MOZYME GPU plan reported incomplete resident Fock coverage' in report_text and 'requires_direct_cosmo_gpu' in report_text and 'parse_int_value(row.get("requires_direct_cosmo_gpu")) == 1' in report_text and '--require-direct-cosmo-gpu' in report_text and '--require-full-scf-gpu requires --require-direct-cosmo-gpu for the v58' in report_text and 'requires a full SCF probe input with EPS=78.4' in report_text and 'full_scf_gpu_stage_completed must exactly equal' in report_text and 'full_scf_gpu_resident_decision' in report_text and 'full_scf_gpu_resident_decision must be CompleteAndPublish' in report_text and 'row_uses_eps' in report_text and 'input_eps' in report_text and 'ordinary EPS/COSMO rows must report resident COSMO Fock, matrix-free CG matvec calls, ' in report_text and 'resident CG control/convergence, and zero host CG syncs' in report_text and 'resident sparse Fock must run real GPU work every resident iteration' in report_text and 'zero-work run call' in report_text and 'mozyme_sparse_fock_run_zero_work_calls' in report_text and 'point-charge/dipole work must be semantically covered when present' in report_text and 'point-charge/dipole setup coverage is incomplete' in report_text and 'point-charge/dipole run coverage is incomplete' in report_text and 'resident sparse Fock real-pair coverage is incomplete' in report_text and 'resident sparse Fock coverage counters are inconsistent' in report_text and 'MOZYME makvec initial LMO construction did not report GPU success' in report_text and 'OLD_SCF existing-LMO marker is not accepted as complete GPU makvec proof' in report_text and 'MOZYME setupk GPU did not report a success marker' in report_text and 'MOZYME_SETUPK_RE' in report_text and 'MOZYME_SCF_OLDEN_SETUP_RE' in report_text and 'MOZYME_SCF_CPU_SETUP_RE' in report_text and 'MOZYME_SCF_FILLIJ_GPU_RE' in report_text and 'MOZYME_RESIDENT_FOCK_GPU_COUNT_RE' in report_text and 'MOZYME_RESIDENT_FOCK_GPU_PACK_RE' in report_text and 'MOZYME_RESIDENT_FOCK_GPU_POINT_WEIGHTS_RE' in report_text and 'MOZYME_RESIDENT_FOCK_CPU_PLAN_RE' in report_text and 'MOZYME_SCF_HOST_COMMIT_RE' in report_text and 'MOZYME_SCF_FINAL_PUBLICATION_RE' in report_text and 'MOZYME_CPU_PINOUT_RE' in report_text and 'full_scf_gpu_final_publication_done' in report_text and 'full_scf_gpu_final_publication_arrays' in report_text and 'full_scf_gpu_final_publication_bytes' in report_text and 'full_scf_gpu_final_publication_cosmo' in report_text and 'full_scf_gpu_olden_setup_only' in report_text and 'full_scf_gpu_fillij_gpu_count_calls' in report_text and 'full_scf_gpu_fillij_gpu_fill_calls' in report_text and 'full_scf_gpu_resident_fock_gpu_count_calls' in report_text and 'full_scf_gpu_resident_fock_gpu_pack_calls' in report_text and 'full_scf_gpu_resident_fock_gpu_point_weight_calls' in report_text and 'strict resident SCF did not report GPU resident-Fock count setup' in report_text and 'strict resident SCF did not report GPU resident-Fock plan pack' in report_text and 'strict resident SCF did not report GPU resident-Fock point weights' in report_text and 'full_scf_gpu_cpu_mozyme_setup_only_calls' in report_text and 'full_scf_gpu_cpu_resident_fock_plan_setup_calls' in report_text and 'full_scf_gpu_host_commit_only_calls' in report_text and 'full_scf_gpu_host_commit_phase' in report_text and 'phase=final_publication' in report_text and 'full_scf_gpu_host_commit_bytes' in report_text and 'full_scf_gpu_cpu_pinout_calls' in report_text and 'validate_strict_host_route_contract' in report_text and 'validate_full_scf_host_commit_contract' in report_text and 'resident-SCF final host publication did not emit host_commit_only=1' in report_text and 'resident-SCF host publication did not report phase=final_publication' in report_text and 'OLDEN/OLDENS host LMO restore marker appeared in strict GPU proof' in report_text and 'strict resident SCF did not report GPU fillij count setup' in report_text and 'strict resident SCF did not report GPU fillij nijbo setup' in report_text and 'CPU MOZYME array setup marker appeared in strict GPU proof' in report_text and 'CPU resident Fock plan construction marker appeared in strict GPU proof' in report_text and 'CPU pinout host I/O marker appeared' in report_text and 'strict GPU proof produced .den host checkpoint artifact(s)' in report_text and 'MOZYME_MAKVEC_EXISTING_RE' in report_text and 'mozyme_makvec_gpu_success_calls' in report_text and 'mozyme_makvec_gpu_existing_lmo_calls' in report_text and 'mozyme_makvec_gpu_fallback_calls' in report_text and 'mozyme_setupk_gpu_success_calls' in report_text and 'mozyme_setupk_gpu_fallback_calls' in report_text and 'MOZYME_REORTH_RE' in report_text and 'MOZYME_TIDY_RE' in report_text and 'mozyme_tidy_gpu_occupied_success_calls' in report_text and 'mozyme_tidy_gpu_virtual_success_calls' in report_text and 'mozyme_tidy_gpu_fallback_calls' in report_text and 'MOPAC_MOZYME_TIDY_GPU=1' in report_text and 'MOZYME GPU TIDY emitted' in report_text and 'mozyme_reorth_gpu_success_calls' in report_text and 'mozyme_reorth_gpu_resident_success_calls' in report_text and 'marker_field_equals(extra, "resident", "1")' in report_text and 'forced REORTH probe did not report a resident=1 final reorth success marker' in report_text and 'FULL_SCF_PROBE_FORCED_KEYWORDS' in report_text and 'FULL_SCF_PROBE_FORCED_ENV' in report_text and 'force_full_scf_probe_keywords' in report_text and 'full_scf_probe_forced_keywords' in report_text and 'full_scf_probe_forced_env' in report_text and 'MOPAC_MOZYME_SCF_FORCE_FINAL_REORTH' in report_text and 'CPU final reorthogonalization sections ran in strict proof' in report_text and '[MOZYME GPU reorth] status=success resident=1' in report_text and 'full_scf_probe_decision' in report_text and 'helper_fatal' in report_text and 'fock1_batch' in report_text and 'fock2_4x1_batch' in report_text and 'fallback_real_pairs' in report_text and 'MOZYME_STRICT_PROOF_ALLOWED_SECTION_NAMES' in report_text and 'mozyme_disallowed_strict_section_stats' in report_text and 'proof_identity_violation_reasons' in report_text and 'source_git_dirty is not true/false' in report_text and 'source_metadata_valid' in report_text and 'source_manifest_file_sha256' in report_text and 'source_required_marker_count' in report_text and 'source_required_markers_verified' in report_text and 'source_required_marker_violation_count' in report_text and 'verify_required_source_markers' in report_text and 'required source markers invalid' in report_text and 'mozyme_gpu_relocal_fortran_wrapper marker is missing' in report_text and 'source feature contract version does not match current contract' in report_text and 'CUDA architecture identity was not recorded' in report_text and 'row.update(proof_identity)' in report_text and 'MOZYME_MAKVEC_SUCCESS_RE' in report_text and '[MOZYME GPU makvec]' in makvec_gpu_text and 'MOPAC_MOZYME_MAKVEC_GPU' in report_text and 'mopac_cuda_mozyme_makvec' in scf_cuda_text and 'mozyme_gpu_makvec_try' in iter_text and 'all parsed fallback counters' in report_text and 'iter_makvec' in report_text and 'CPU makvec initial LMO construction' in iter_text and 'strict_old_scf_existing_lmo' in iter_text and 'does not accept OLD_SCF host-existing LMOs as makvec proof' in iter_text and 'iter_tidy_occ' in report_text and 'iter_setupk' in report_text and 'iter_olden_load' in report_text and 'iter_density_olden' in report_text and 'iter_reloc_occ' in report_text and 'iter_reloc_virt' in report_text and 'iter_isitsc' in report_text and 'iter_pls_faulty' in report_text and 'iter_helecz_initial' in report_text and 'iter_helecz_iter' in report_text and 'iter_helecz_reorth' in report_text),
    ('Colab resident SCF smoke requires final SCF success', '_scf_success_lines' in colab_text and 'status=success code=0 ready=1 resident=1' in colab_text and 'fatal_status_lines' in colab_text and 'isitsc|tidy' in colab_text and 'cnvgz_active_calls' in colab_text and 'CNVGZ GPU stage work' in colab_text and 'fock1_batch' in colab_text and 'fock2_4x1_batch' in colab_text and 'fallback_real_pairs' in colab_text and 'Experimental resident SCF reported fallback_cpu, strict_abort, or resident_step in a fatal smoke check.' in colab_text and 'Experimental resident SCF did not report [MOZYME GPU SCF] status=success code=0 ready=1 resident=1.' in colab_text),
    ('MOZYME reorth success marker requires final SCF success', "index(prefix, 'status=success') > 0" in scf_driver_text and '[MOZYME GPU reorth]' in scf_driver_text and 'status=success resident=1' in scf_driver_text),
    ('MOZYME GPU relocal source contract', 'mozyme_gpu_relocalize_try' in relocal_text and 'bind(C,name=\'mopac_cuda_mozyme_relocalize\')' in relocal_text and 'kind_mismatch' in relocal_text and 'kind=' in relocal_text and 'type=' not in relocal_text and '[MOZYME GPU relocal]' in relocal_text and 'mopac_cuda_mozyme_relocalize' in scf_cuda_text and 'mozyme_relocalize_kernel' in scf_cuda_text and 'mozyme_relocalize_eigs_device' in scf_cuda_text and 'std::vector<double> host_c(c_sz);' in scf_cuda_text and 'std::copy(host_c.begin(), host_c.end(), c);' in scf_cuda_text and 'mozyme_relocal_gpu_success_calls' in report_text and 'MOZYME_RELOCAL_RE' in report_text and 'MOPAC_MOZYME_RELOCAL_GPU' in report_text),
    ('MOZYME GPU reorth source contract', 'mozyme_gpu_reorth_try' in reorth_text and 'bind(C,name=\'mopac_cuda_mozyme_reorth\')' in reorth_text and 'kind_mismatch' in reorth_text and '[MOZYME GPU reorth]' in reorth_text and 'mopac_cuda_mozyme_reorth' in scf_cuda_text and 'mozyme_reorth_kernel' in scf_cuda_text and 'mozyme_reorth_adjvec_device' in scf_cuda_text and 'kMozymeScfFlagFinalReorth' in scf_cuda_text and 'apply_final_reorth_on_gpu' in scf_cuda_text and 'resident final reorth kernel' in scf_cuda_text and 'status->final_reorth_applied = 1' in scf_cuda_text and 'status->final_reorth_ms' in scf_cuda_text and 'status->final_reorth_sum' in scf_cuda_text and 'DeviceBuffer<double> final_reorth_ws;' in scf_cuda_text and 'dev.final_reorth_ws.resize(norbs_count)' in scf_cuda_text and 'resident final reorth status reset' in scf_cuda_text and 'DeviceBuffer<double> ws;\n  DeviceBuffer<double> sumtot;\n  DeviceBuffer<int> latom;' not in scf_cuda_text and 'if (!reorth_status.upload(&zero_i, 1))' not in scf_cuda_text and 'if (!sumtot.upload(&zero_d, 1))' not in scf_cuda_text and 'mozyme_reorth_gpu_success_calls' in report_text and 'mozyme_reorth_gpu_resident_success_calls' in report_text and 'MOZYME_REORTH_RE' in report_text and 'MOPAC_MOZYME_REORTH_GPU' in report_text),
    ('MOZYME GPU tidy source contract', 'module mozyme_gpu_tidy' in tidy_gpu_text and 'public :: mozyme_gpu_tidy_try' in tidy_gpu_text and "bind(C,name='mopac_cuda_mozyme_tidy')" in tidy_gpu_text and 'MOPAC_MOZYME_TIDY_GPU' in tidy_gpu_text and '[MOZYME GPU tidy]' in tidy_gpu_text and 'selmos=' in tidy_gpu_text and 'selected=' in tidy_gpu_text and 'use_selmos_c' in tidy_gpu_text and 'type(c_ptr), value :: jopt_c' in tidy_gpu_text and 'jopt_ptr = c_null_ptr' in tidy_gpu_text and 'jopt_ptr = c_loc(jopt(1))' in tidy_gpu_text and 'numred_out_of_range' in tidy_gpu_text and 'error_code' in tidy_gpu_text and 'resident_tidy_imode(2) = 0' in iter_text and 'step_num > 1+step_num0' in iter_text and 'step_num /= resident_tidy_imode(1)' in iter_text and 'step_num /= resident_tidy_imode(2)' in iter_text and 'mozyme_gpu_tidy_try(1, lno, mn, &' in iter_text and 'mozyme_gpu_tidy_try(2, lnv, mn, &' in iter_text and 'use_selmos=resident_tidy_select_lmos' in iter_text and 'error_code=resident_tidy_code' in iter_text and 'resident_tidy_code == -506' in iter_text and 'mozyme_gpu_grow_lmo_storage' in iter_text and 'strict_resident_tidy_selmos_missing' not in iter_text and 'initial_tidy_done=resident_tidy_done' in iter_text and 'backend_resident_tidy_complete(initial_setup_requested,' in scf_driver_text and 'backend_resident_tidy_missing' in scf_driver_text),
    ('MOZYME GPU tidy CUDA helper', 'mozyme_tidy_kernel' in scf_cuda_text and 'mopac_cuda_mozyme_tidy' in scf_cuda_text and 'mozyme_tidy_selmos_device' in scf_cuda_text and 'mozyme_tidy_compct_device' in scf_cuda_text and 'mozyme_tidy_space_device' in scf_cuda_text and 'nc[lmo] <= 0' in scf_cuda_text and 'use_selmos != 0 && numred > natoms' in scf_cuda_text and '*status = -520' in scf_cuda_text and '*status = -521' in scf_cuda_text and 'int code = -530' in scf_cuda_text and 'jopt' in scf_cuda_text and 'selected_out' in scf_cuda_text and 'tidy kernel' in scf_cuda_text and 'tidy result copy' in scf_cuda_text and 'tidy nc copy' in scf_cuda_text and 'tidy nnc copy' in scf_cuda_text and 'tidy ncmo copy' in scf_cuda_text and 'tidy elapsed time' in scf_cuda_text),
    ('resident scalar stage proof', 'full_scf_gpu_stage_cnvgz_calls' in report_text and 'full_scf_gpu_stage_helecz_calls' in report_text and 'full_scf_gpu_cnvgz_active_calls' in report_text and 'full_scf_gpu_cnvgz_noop_calls' in report_text and 'MOZYME_SCF_CNVGZ_ACTIVITY_RE' in report_text and 'cnvgz_active_noop_ready' in report_text and 'append_cnvgz_active_noop_reasons' in report_text and 'parse_scf_cnvgz_activity' in report_text and 'cnvgz_active_calls=' in scf_driver_text and 'complete_stage_from_int(kMozymeScfStageCnvgz,' in scf_cuda_text and 'complete_stage_from_int(kMozymeScfStageHelecz,' in scf_cuda_text),
    ('resident setupk initial marker contract', 'mozyme_setupk_gpu_initial_setup_success_calls' in report_text and 'mozyme_setupk_gpu_initial_setup_fallback_calls' in report_text and 'mozyme_setupk_gpu_initial_setup_all_paths_calls' in report_text and 'mozyme_setupk_gpu_initial_setup_last_ms' in report_text and 'MOZYME setupk GPU did not report an initial_setup=1 success marker' in report_text and 'MOZYME setupk GPU reported an initial_setup=1 fallback marker' in report_text and 'MOZYME setupk all-initial-setup path marker was not reported' in report_text and 'initial_setup=1' in scf_driver_text and 'fock_mode=' in scf_driver_text and 'all_initial_setup_paths=1' in scf_driver_text),
    ('resident SCF device evidence', 'status%device_id >= 0_c_int' in scf_driver_text and 'device_id=' in scf_driver_text and 'resident status cudaGetDevice' in scf_cuda_text and 'full_scf_gpu_device_id' in report_text),
    ('strict full SCF readiness artifact fields', all(field in report_text for field in strict_readiness_fields) and 'MOZYME_SCF_STAGE_FULL' in report_text and 'full_scf_gpu_final_iterations must be at least 1' in report_text),
    ('strict full SCF fallback counters', all(field in report_text for field in resident_fock_fallback_fields) and all(field in report_text for field in stage_fallback_fields)),
    ('full SCF readiness fields', 'full_scf_gpu_status' in report_text and 'full_scf_gpu_final_density_resident' in report_text and 'full_scf_gpu_cpu_mutating_sections' in report_text and 'full_scf_gpu_cpu_mutating_section_count' in report_text and 'full_scf_gpu_cpu_mutating_call_count' in report_text and 'full_scf_gpu_cpu_mutating_ms' in report_text and 'final_density=current_resident' in report_text and 'final_density=current_resident' in iter_text and '--require-full-scf-gpu' in report_text and '--full-scf-readiness-only' in report_text),
    ('resident SCF pre-tidy strict entry', 'resident_pre_tidy_attempt_needed = mozyme_gpu_scf_requested()' in iter_text and 'niter == 0' in iter_text and 'resident_fock_mode = 1' in iter_text and 'resident_fock_mode, idiagg, nhb, resident_fock_mode' in iter_text and 'initial_setup=.true.' in iter_text and 'block_on_failure=resident_strict_required' in iter_text and 'MOZYME GPU strict resident SCF failed before CPU tidy' in iter_text and 'MOZYME GPU strict resident SCF failed before CPU SCF body' in iter_text and 'MOZYME GPU strict resident SCF could not start before CPU tidy' in iter_text and 'mozyme_gpu_scf_strict_resident' in scf_driver_text and 'block_failures' in scf_driver_text and 'backend_pls_resolved(status)' in scf_driver_text and 'backend_resident_decision' in scf_driver_text and 'backend_resident_tidy_missing' in scf_driver_text and 'backend_resident_tidy_complete(initial_setup_requested, &' in scf_driver_text and 'full_success = backend_completed_scf(code, status, niter, &\n      initial_setup_requested, initial_tidy_completed)' in scf_driver_text and 'status, niter, initial_setup_requested, initial_tidy_completed))' in scf_driver_text and 'status%resident_decision == &' in scf_driver_text and 'GPU_MOZYME_SCF_RESIDENT_DECISION_COMPLETE' in scf_driver_text and '.not. useps' not in iter_text),
    ('resident SCF integer helper import', 'use mozyme_gpu_int_utils, only: mozyme_c_int_checked' in scf_driver_text and 'config%resident_fock_plan_id = mozyme_c_int_checked(resident_fock_plan_id)' in scf_driver_text),
    ('resident SCF CUDA error surfacing', 'cuda_context_ok' in scf_cuda_text and '[GPU ERROR] MOZYME SCF' in scf_cuda_text and 'resident cnvgz elapsed time' in scf_cuda_text and 'diagg2 rotate elapsed time' in scf_cuda_text),
    ('resident SCF readiness artifacts', 'Complete GPU SCF readiness probe passed.' in report_text and 'full_scf_gpu_readiness.json' in report_text and 'full_scf_gpu_readiness_failure.txt' in report_text and 'full_scf_gpu_readiness_cpu_compare.json' in report_text and 'select_mozyme_preflight_input' in report_text and 'MOPAC_MOZYME_SCF_STRICT_RESIDENT' in report_text and report_text.count('create_bundle_zip(bundle_path, out_dir)') >= 3),
    ('MOZYME setupk GPU helper', 'mopac_cuda_mozyme_setupk' in setupk_text and 'mopac_cuda_mozyme_setupk' in scf_cuda_text and 'mozyme_setupk_mark_kernel' in scf_cuda_text and 'mozyme_setupk_compress_kernel' in scf_cuda_text and 'atom <= natoms' in scf_cuda_text and '[MOZYME GPU setupk]' in setupk_text),
    ('resident SCF report real-pair hard gate', 'resident_fock_partial_coverage' in report_text and 'validate_resident_real_pair_coverage' in report_text and 'select_resident_fock_coverage' in report_text and 'real_pairs != gpu_pairs' in report_text and 'real_pairs != gpu_pairs + cpu_pairs' in report_text and 'mozyme_resident_fock_inactive_real_pairs is nonzero' not in report_text and 'coverage_mode is None or coverage_mode == 0' not in report_text),
    ('full SCF DIAGG report fields', 'MOZYME_SCF_DIAGG_RE' in report_text and 'full_scf_gpu_diagg_sumb' in report_text),
    ('resident SCF stage masks', 'stage_completed' in scf_driver_text and 'stage_missing' in scf_driver_text and 'MOZYME_SCF_STAGE_RE' in report_text and 'MOZYME_SCF_ISITSC_RE' in report_text),
    ('resident SCF per-stage runtime counters', 'resident_stage_calls(10)' in scf_interface_text and 'resident_stage_ms(10)' in scf_interface_text and "'[MOZYME GPU SCF]', 'resident_stage_calls'" in scf_driver_text and "'[MOZYME GPU SCF]', 'resident_stage_ms'" in scf_driver_text and 'MOZYME_SCF_STAGE_CALLS_RE' in report_text and 'MOZYME_SCF_STAGE_MS_RE' in report_text and 'full_scf_gpu_stage_{name}_calls' in report_text and 'resident-SCF stage {stage_name} calls=' in report_text and 'DeviceBuffer<int> resident_stage_calls;' in scf_cuda_text and 'dev.resident_stage_calls.upload(initial_stage_calls' in scf_cuda_text and 'mozyme_resident_stage_upload_accept_kernel' in scf_cuda_text and 'stage_calls[kResidentStageSlotUpload] = 1' in scf_cuda_text and 'stage_calls[stage_slot] += 1' in scf_cuda_text and 'resident_stage_device_call_count' in scf_cuda_text and 'copy_resident_stage_ints_from_gpu' in scf_cuda_text and 'copy_resident_stage_calls_from_gpu' in scf_cuda_text and 'resident stage device counters copy' in scf_cuda_text and 'resident_stage_device_confirmed' in scf_cuda_text),
    ('resident SCF missing-stage reason', 'backend_missing_stages' in scf_driver_text),
    ('resident SCF command-line smoke gate', 'reason=denout_checkpoint' in scf_driver_text and 'iter_olden_load' in iter_text and 'iter_density_olden' in iter_text and 'iter_reloc_occ' in iter_text and 'iter_reloc_virt' in iter_text and 'reason=solvent_fock' in scf_driver_text and 'if (lpka) then' in scf_driver_text and 'if (useps .or. lpka) then' not in scf_driver_text and 'reason=resident_fock_partial_coverage' in scf_driver_text and 'MOPAC_MOZYME_SCF_EARLY_PROBE' in scf_driver_text and 'strict_denout_host_output' in iter_text and 'strict_olden_host_lmo_restore' in iter_text and 'strict_pka_host_output' in iter_text),
    ('resident SCF gate', 'MOPAC_MOZYME_SCF_EXPERIMENTAL' in scf_driver_text),
    ('resident SCF Fortran COSMO direct state', 'mozyme_cosmo_prepare_gpu_state' in scf_driver_text and 'mozyme_cosmo_gpu_state' in scf_driver_text and 'call mozyme_cosmo_prepare_gpu_state(cosmo_prepare_ok)' in scf_driver_text and 'state%cosmo_enabled = 1_c_int' in scf_driver_text and 'cosmo_state_supported = .false.' in scf_driver_text and "scf_failure_message('reason=solvent_fock detail=cosmo_prepare')" in scf_driver_text and 'cosmo_fock_calls=' in scf_driver_text and 'cosmo_matvec_calls=' in scf_driver_text and 'cosmo_last_residual=' in scf_driver_text and 'cosmo_cg_control_resident=' in scf_driver_text and 'cosmo_cg_converged=' in scf_driver_text and 'cosmo_cg_breakdown=' in scf_driver_text and 'cosmo_cg_host_syncs=' in scf_driver_text and 'cosmo_cg_target_tol=' in scf_driver_text and 'state%cosmo_cosurf_rows = &' in scf_driver_text and 'c_int_or_zero(size1_or_zero_real_2d(cosurf))' in scf_driver_text and 'state%cosmo_a_part_i = cosmo_a_part_i_ptr' in scf_driver_text and 'state%cosmo_solv_energy_ptr = c_loc(solv_energy)' in scf_driver_text and 'state%cosmo_ediel_ptr = c_loc(ediel)' in scf_driver_text and 'state%cosmo_enabled == 1_c_int' in scf_driver_text and 'missing_field = \'cosmo_npoints_dim\'' in scf_driver_text),
    ('linear COSMO direct GPU state export', 'mozyme_cosmo_prepare_gpu_state' in linear_cosmo_text and 'mozyme_cosmo_gpu_state' in linear_cosmo_text and 'mozyme_cosmo_allocate_a_part_pairs' in linear_cosmo_text and 'mozyme_cosmo_build_a_part_pairs' in linear_cosmo_text and 'count_short_ints(cosurf, 4, simulate_aq_dir_int, .true.)' in linear_cosmo_text and 'a_part_i(npos) = ii' in linear_cosmo_text and 'a_part_j(npos) = jj' in linear_cosmo_text and 'npoints_ptr = c_loc(npoints(1))' in linear_cosmo_text and 'new_surface_flag = merge(1_c_int, 0_c_int, new_surface)' in linear_cosmo_text),
    (
        'resident SCF ABI v33 final-publication/reorth/PLS/COSMO-direct fields',
        'GPU_MOZYME_SCF_ABI_VERSION = 33_c_int' in scf_interface_text
        and 'GPU_MOZYME_SCF_RESIDENT_DECISION_COMPLETE = 1_c_int' in scf_interface_text
        and 'kMozymeScfAbiVersion = 33' in scf_cuda_text
        and 'resident_decision = 0_c_int' in scf_interface_text
        and 'resident_decision_to_code' in scf_cuda_text
        and all(field in scf_interface_text for field in ('partp_dim', 'partf_dim', 'nocc_slots', 'nvir_slots', 'p_dim', 'f_dim', 'h_dim', 'pold_dim', 'p1_dim', 'p2_dim', 'p3_dim', 'idiag_dim', 'iorbs_dim', 'kopt_dim', 'ncf_dim', 'nncf_dim', 'ncocc_dim', 'nce_dim', 'nnce_dim', 'ncvir_dim', 'ifmo_rows', 'ifmo_cols', 'eigs_dim', 'nfmo_dim', 'nfirst_dim', 'nlast_dim', 'nijbo_rows', 'nijbo_cols', 'resident_stage_calls(10)', 'resident_stage_ms(10)', 'final_publication_done', 'final_publication_arrays', 'final_publication_bytes', 'final_publication_cosmo', 'final_reorth_applied', 'final_reorth_ms', 'final_reorth_sum', 'pls_supervisor_calls', 'pls_restart_required', 'pls_history_count', 'pls_ovmax_delta', 'pls_energy_delta', 'pls_restart_reset_device_calls', 'pls_restart_done', 'coord_rows', 'coord_cols', 'nat_dim', 'cosmo_cosurf_rows', 'cosmo_cosurf_cols', 'cosmo_phinet_rows', 'cosmo_phinet_cols', 'cosmo_qscnet_rows', 'cosmo_qscnet_cols', 'cosmo_qdenet_rows', 'cosmo_qdenet_cols', 'cosmo_qscat_dim', 'cosmo_srad_dim', 'cosmo_npoints_dim', 'cosmo_a_diag_dim', 'cosmo_a_part_dim', 'cosmo_m_vec_dim', 'cosmo_iblock_pos_dim', 'cosmo_new_surface', 'param_dim', 'cosmo_fepsi', 'cosmo_disex2', 'cosmo_solv_energy', 'cosmo_ediel', 'cosmo_a0', 'cosmo_ev', 'cosmo_fock_calls', 'cosmo_matvec_calls', 'cosmo_cg_iterations', 'cosmo_pair_count', 'cosmo_last_residual', 'cosmo_cg_control_resident', 'cosmo_cg_converged', 'cosmo_cg_breakdown', 'cosmo_cg_host_syncs', 'cosmo_cg_target_tol', 'cnvgz_active_calls', 'cnvgz_noop_calls', 'strict_resident_host_syncs', 'strict_resident_control_polls', 'param_dd', 'param_qq', 'param_tore', 'cosmo_npoints', 'cosmo_a_part_i', 'cosmo_a_part_j', 'cosmo_solv_energy_ptr', 'cosmo_ediel_ptr'))
        and all(field in scf_cuda_text for field in ('p_dim', 'f_dim', 'h_dim', 'pold_dim', 'partp_dim', 'partf_dim', 'p1_dim', 'p2_dim', 'p3_dim', 'idiag_dim', 'eigs_dim', 'nfmo_dim', 'iorbs_dim', 'kopt_dim', 'nfirst_dim', 'nlast_dim', 'ncf_dim', 'nncf_dim', 'ncocc_dim', 'nce_dim', 'nnce_dim', 'ncvir_dim', 'resident_stage_calls[kResidentStageSlotCount]', 'resident_stage_ms[kResidentStageSlotCount]', 'final_publication_done', 'final_publication_arrays', 'final_publication_bytes', 'final_publication_cosmo', 'strict_resident_host_syncs', 'strict_resident_control_polls', 'ResidentFinalPublicationProof', 'mark_final_publication_done', 'final_reorth_applied', 'final_reorth_ms', 'final_reorth_sum', 'mozyme_pls_supervisor_device', 'ResidentReturnDecision::PlsRestart', 'publish_pls_runtime_status', 'publish_cosmo_status', 'publish_cosmo_status_from_gpu', 'preserve_resident_cosmo_cg_status', 'capture_host_checkpoint', 'restore_host_checkpoint', 'status->cosmo_fock_calls', 'kCosmoCgMatvecCalls', 'kCosmoStatusControlResident', 'kCosmoStatusCgConverged', 'kCosmoStatusHostSyncs', 'status->cosmo_cg_control_resident', 'status->cosmo_cg_converged', 'status->cosmo_cg_host_syncs', 'mozyme_cosmo_cg_finalize_status_kernel', 'mozyme_cosmo_cg_mark_fock_status_kernel', 'mozyme_cosmo_cg_prepare_iteration_kernel', 'mozyme_cosmo_cg_prepare_update_kernel', 'mozyme_cosmo_cg_finish_iteration_kernel', 'resident final COSMO CG scalar status copy', 'state.use_nijbo != 1 || !state.nijbo', 'apply_cosmo_fock_on_gpu', 'mozyme_final_reorth_status_kernel', 'final_reorth_status_scalars', 'resident final status scalar copy', 'device_final_reorth_committed', 'ok || device_final_reorth_committed', 'mozyme_cosmo_build_potential_kernel', 'run_cosmo_matvec', 'mozyme_cosmo_matvec_far_kernel', 'mozyme_cosmo_matvec_close_kernel', 'mozyme_cosmo_fock_correction_kernel', 'cosmo_surface_at', 'atomicAdd_double(out', 'atomicAdd_double(qscat', 'dev.cosmo_ipiden.upload', 'dev.cosmo_qdenet.upload', 'checkpoint_cosmo_qscnet', 'copy_device_buffer(dev.cosmo_qscnet, dev.checkpoint_cosmo_qscnet)', 'stage_and_commit_device_vector(ctx.state.cosmo_qscat', 'stage_and_commit_device_vector(ctx.state.cosmo_phinet', 'stage_and_commit_device_vector(ctx.state.cosmo_qdenet', 'std::memcpy(ctx.state.cosmo_solv_energy_ptr', 'std::memcpy(ctx.state.cosmo_ediel_ptr', 'if (mode == 0 && !apply_cosmo_fock_on_gpu(ctx, density_p, output_f))', 'state.cosmo_a_part_dim > 0', 'state.cosmo_npoints_dim < numat + 1', 'kResidentControlPlsRestartRequired', 'kResidentControlPlsRestartResetCalls', 'kResidentControlPlsRestartDone', 'status->pls_restart_required', 'status->pls_restart_reset_device_calls', 'status->pls_restart_done', 'mozyme_resident_pls_restart_zero_kernel', 'mozyme_resident_pls_restart_finalize_kernel', 'apply_resident_pls_restart_if_requested_on_gpu', 'bool valid_state(const MozymeScfConfig &config, const MozymeScfState &state)', 'state.ifmo_rows != 2', 'state.nijbo_rows != numat', 'if (!valid_state(ctx->config, *state)) return kMozymeScfBadArgument;', 'if (!valid_state(ctx.config, ctx.state)) return false;'))
        and 'state_layout_has_required_sizes(config, state, missing_field)' in scf_driver_text
        and 'reason=state_layout_invalid missing=' in scf_driver_text
        and 'storage_size(0) == storage_size(0_c_int)' in scf_driver_text
        and 'storage_size(0.0d0) == storage_size(0.0_c_double)' in scf_driver_text
        and 'size(partp) >= mpack' in scf_driver_text
        and 'size(partf) >= mpack' in scf_driver_text
        and 'size(ncf) < int(state%nocc_slots)' in scf_driver_text
        and 'size(nce) < int(state%nvir_slots)' in scf_driver_text
        and 'size(ifmo, 1) == 2' in scf_driver_text
        and 'size(nijbo, 1) == numat' in scf_driver_text
        and 'ptr_or_null_c_int' in scf_driver_text
        and 'GPU_MOZYME_SCF_STAGE_ADDHB' in scf_interface_text
        and 'GPU_MOZYME_SCF_STAGE_CHECK' in scf_interface_text
        and 'copy_device_to_host_raw' in scf_cuda_text
        and 'compute_initial_setup_on_gpu' in scf_cuda_text
        and 'cosmo_cg_target_tolerance(ctx)' not in scf_cuda_text
        and 'ctx.cosmo_matvec_calls += host_cg_ints[kCosmoCgMatvecCalls]' not in scf_cuda_text
        and 'cudaMemcpy(host_cg_scalars' not in scf_cuda_text
        and 'cudaMemcpy(host_cg_ints' not in scf_cuda_text
        and 'resident COSMO CG scalar status copy' not in scf_cuda_text
        and 'resident COSMO CG integer status copy' not in scf_cuda_text
        and 'iter == 0 ? 1 : 0' not in scf_cuda_text,
    ),
    ('resident SCF ABI v33 resident Fock proof fields', 'integer(c_int) :: resident_fock_plan_id = 0_c_int' in scf_interface_text and 'integer(c_int) :: resident_fock_plan_full_coverage = 0_c_int' in scf_interface_text and 'integer(c_int) :: resident_fock_plan_partial_coverage = 0_c_int' in scf_interface_text and 'integer(c_int) :: resident_fock_plan_required_mask = 0_c_int' in scf_interface_text and 'integer(c_int) :: resident_fock_plan_covered_mask = 0_c_int' in scf_interface_text and 'int resident_fock_plan_id;' in scf_cuda_text and 'int resident_fock_plan_full_coverage;' in scf_cuda_text and 'int resident_fock_plan_partial_coverage;' in scf_cuda_text and 'int resident_fock_plan_required_mask;' in scf_cuda_text and 'int resident_fock_plan_covered_mask;' in scf_cuda_text and 'status->resident_fock_plan_full_coverage' in scf_cuda_text and 'status->resident_fock_plan_required_mask' in scf_cuda_text and 'config.resident_fock_plan_required_mask == expected_required_mask' in scf_cuda_text and 'config.resident_fock_plan_covered_mask == expected_required_mask' in scf_cuda_text),
    (
        'resident SCF partial initial setup dual fock plans',
        'mopac_cuda_mozyme_sparse_fock_setup_plan' in cuda_text
        and 'mopac_cuda_mozyme_sparse_fock_run_device_plan' in scf_cuda_text
        and 'kMozymeFockPlanFull' in scf_cuda_text
        and 'kMozymeFockPlanPartial' in scf_cuda_text
        and 'mozyme_resident_fock_prepare_plan' in scf_driver_text
        and 'if (initial_setup_requested) then' in scf_driver_text
        and 'resident_fock_required_mask = 1_c_int' in scf_driver_text
        and 'resident_fock_plan_ready = mozyme_resident_fock_prepare_plan(' in scf_driver_text
        and 'resident_fock_plan_full, iorbs, nat, ifact,' in scf_driver_text
        and 'resident_fock_plan_partial, iorbs, nat, ifact,' in scf_driver_text
        and 'detail=setupk' in scf_driver_text
        and 'full and selected partial-active-space paths' in iter_text,
    ),
    (
        'resident SCF partial fock bounded copy-back',
        'partp_upload_count' in scf_cuda_text
        and 'partf_upload_count' in scf_cuda_text
        and 'const std::size_t partp_count = std::min(' in scf_cuda_text
        and 'static_cast<std::size_t>(ctx.state.partp_dim), mpack_count);' in scf_cuda_text
        and 'const std::size_t partf_count = std::min(' in scf_cuda_text
        and 'static_cast<std::size_t>(ctx.state.partf_dim), mpack_count);' in scf_cuda_text
        and 'stage_and_commit_device_vector(ctx.state.partp, dev.partp, partp_count' in scf_cuda_text
        and 'stage_and_commit_device_vector(ctx.state.partf, dev.partf, partf_count' in scf_cuda_text
        and 'stage_and_commit_device_vector(ctx.state.partp, dev.partp, partp_count' in scf_cuda_text
        and 'stage_and_commit_device_vector(ctx.state.partf, dev.partf, partf_count' in scf_cuda_text
        and 'std::vector<double> host_partp(mpack_count);' not in scf_cuda_text
        and 'std::vector<double> host_partf(mpack_count);' not in scf_cuda_text
        and 'std::vector<double> host_partp(partp_count);' not in scf_cuda_text
        and 'copy_device_to_host_vector(host_partp, dev.partp)' not in scf_cuda_text,
    ),
    ('resident SCF ABI', 'mopac_cuda_mozyme_scf_run' in scf_cuda_text and 'mopac_cuda_mozyme_scf_register_state' in scf_cuda_text),
    ('resident SCF density selector and spin scale', 'density_mode' in scf_interface_text and 'config%density_mode' in scf_driver_text and 'mozyme_density_spin_scale_kernel' in scf_cuda_text and 'compute_density_on_gpu_impl' in scf_cuda_text and 'ctx.config.density_indi' in scf_cuda_text),
    ('resident SCF fock mode', 'fock_mode' in scf_interface_text and 'config%fock_mode' in scf_driver_text),
    ('resident SCF DIAGG control', 'diagg_mode' in scf_interface_text and 'density_indi' in scf_interface_text and 'config%diagg_mode' in scf_driver_text and 'config%shift' in scf_driver_text),
    ('resident SCF DIAGG state upload', 'fmo_dim' in scf_interface_text and 'nfmo' in scf_interface_text and 'nfirst' in scf_interface_text and 'nlast' in scf_interface_text and 'dev.nfmo.upload' in scf_cuda_text and 'dev.nfirst.upload' in scf_cuda_text),
    ('resident SCF DIAGG state handoff', 'mozyme_diagg1_set_state' in scf_driver_text and 'mozyme_diagg2_set_state' in scf_driver_text and 'diagg_nf' in scf_interface_text and 'diagg_fref' in scf_cuda_text and 'diagg_sumt' in scf_driver_text and 'diagg_sumb' in scf_driver_text and 'pmax = status%density_max' in scf_driver_text),
    ('resident SCF DIAGG scalar packs', 'diagg_ints' in scf_cuda_text and 'diagg_scalars' in scf_cuda_text and 'kDiaggIntNij' in scf_cuda_text and 'kDiaggDoubleSumt' in scf_cuda_text and 'resident final diagg integer copy' in scf_cuda_text and 'resident final diagg scalar copy' in scf_cuda_text and 'resident diagg integer copy' not in scf_cuda_text and 'cudaMemcpy(&host_nij, nij.ptr' not in scf_cuda_text),
    ('resident SCF check kernel', 'compute_check_on_gpu' in scf_cuda_text and 'mozyme_check_lmo_kernel' in scf_cuda_text and 'mozyme_check_init_kernel' in scf_cuda_text and 'resident check integer reset' in scf_cuda_text and 'resident check errors reset' in scf_cuda_text and 'mozyme_check_finalize_kernel' in scf_cuda_text and 'kMozymeScfStageCheck' in scf_cuda_text and 'check_ints' in scf_cuda_text and 'check_errors' in scf_cuda_text and 'kCheckIntOk' in scf_cuda_text and 'mozyme_check_finalize_kernel<<<1, 1>>>(nocc, nvir, dev.check_errors.ptr,' in scf_cuda_text and 'complete_stage_from_int(kMozymeScfStageCheck, ctx.device.check_ints.ptr' in scf_cuda_text and 'int host_ints[kCheckIntCount] = {}' not in scf_cuda_text and 'host_ints[kCheckIntOk] == 1' not in scf_cuda_text and 'cudaMemcpy(host_ints, dev.check_ints.ptr, sizeof(host_ints)' not in scf_cuda_text and 'host_errors[kCheckDoubleOccError]' not in scf_cuda_text and 'cudaMemcpy(&host_ok, ok_flag.ptr' not in scf_cuda_text and 'cudaMemset(dev.check_ints.ptr' not in scf_cuda_text and 'cudaMemset(dev.check_errors.ptr' not in scf_cuda_text),
    ('resident SCF DIAGG kernel', 'compute_diagg_on_gpu' in scf_cuda_text and 'kMozymeScfStageDiagg' in scf_cuda_text),
    ('resident SCF density rebuild', 'mozyme_density_resident_kernel' in scf_cuda_text and 'mozyme_density_expected_kernel' in scf_cuda_text and 'kMozymeScfStageDensity' in scf_cuda_text and 'mozyme_density_expected_kernel<<<nclose, kThreads>>>' in scf_cuda_text and 'mozyme_density_resident_kernel<<<nclose, kThreads>>>' in scf_cuda_text and 'resident density status reset' in scf_cuda_text and 'mozyme_set_int_slot_if_resident_active_kernel<<<1, 1>>>(\n        dev.density_updates.ptr' in scf_cuda_text and 'dev.density_updates.ptr + 2' in scf_cuda_text and 'mozyme_update_status_finalize_kernel<<<1, 1>>>(\n        dev.density_updates.ptr, 1, dev.resident_control_ints.ptr)' in scf_cuda_text and 'complete_stage_from_int(kMozymeScfStageDensity,' in scf_cuda_text and 'int host_density_status[3] = {0, 1, 0};' not in scf_cuda_text and 'dev.density_updates.upload(host_density_status, 3)' not in scf_cuda_text and 'host_density_status[1] != 1' not in scf_cuda_text and 'cudaMemcpy(host_density_status, dev.density_updates.ptr' not in scf_cuda_text and 'expected_density_updates' not in scf_cuda_text and 'host_updates != expected_density_updates' not in scf_cuda_text and 'const int *host_ncf = static_cast<const int *>(ctx.state.ncf);' not in scf_cuda_text and 'const int *host_nijbo = static_cast<const int *>(ctx.state.nijbo);' not in scf_cuda_text and 'cudaMemcpy(&host_updates, dev.density_updates.ptr, sizeof(int)' not in scf_cuda_text and 'threadIdx.x != 0 || lmo > nclose' not in scf_cuda_text and 'cudaMemset(dev.density_updates.ptr' not in scf_cuda_text),
    ('resident SCF ADDHB scalar packs', 'addhb_ints' in scf_cuda_text and 'addhb_scalars' in scf_cuda_text and 'kAddhbIntNij' in scf_cuda_text and 'kAddhbDoubleSumb' in scf_cuda_text and 'resident final addhb integer copy' in scf_cuda_text and 'resident addhb integer copy' not in scf_cuda_text and 'cudaMemcpy(&addhb_nij, nij.ptr' not in scf_cuda_text),
    ('resident SCF DIAGG/ADDHB persistent scratch', 'DeviceBuffer<double> diagg_aocc;' in scf_cuda_text and 'DeviceBuffer<int> addhb_iused;' in scf_cuda_text and 'dev.diagg_aocc.resize(' in scf_cuda_text and 'if (!dev.diagg_avir.resize(norbs_count)) return false;' in scf_cuda_text and 'if (!dev.diagg_aov.resize(numat_count)) return false;' in scf_cuda_text and 'if (!dev.diagg_ws.resize(norbs_count)) return false;' in scf_cuda_text and 'if (!dev.diagg_storei.resize(norbs_count)) return false;' in scf_cuda_text and 'if (!dev.diagg_storej.resize(norbs_count)) return false;' in scf_cuda_text and 'if (!dev.diagg_iused.resize(numat_count)) return false;' in scf_cuda_text and 'dev.addhb_iused.resize(addhb_iused_count)' in scf_cuda_text and 'if (!dev.addhb_latoms.resize(numat_count)) return false;' in scf_cuda_text and 'if (!dev.addhb_storei.resize(norbs_count)) return false;' in scf_cuda_text and 'if (!dev.addhb_storej.resize(norbs_count)) return false;' in scf_cuda_text and 'dev.diagg_aocc.ptr, dev.diagg_avir.ptr, dev.diagg_aov.ptr' in scf_cuda_text and 'dev.addhb_iused.ptr,\n        dev.ifmo.ptr' in scf_cuda_text and 'DeviceBuffer<double> aocc;' not in scf_cuda_text and 'DeviceBuffer<double> avir;' not in scf_cuda_text and 'if (!aocc.resize(' not in scf_cuda_text and 'if (!iused.resize(static_cast<std::size_t>(max_iused)))' not in scf_cuda_text),
    ('resident SCF preallocated stage buffers', 'dev.eimp_p.resize(mpack_count)' in scf_cuda_text and 'dev.eimp_pair_updates.resize(3)' in scf_cuda_text and 'dev.density_updates.resize(3)' in scf_cuda_text and 'dev.qe.resize(numat_count)' in scf_cuda_text and 'dev.cnvgz_ints.resize(kCnvgzIntCount)' in scf_cuda_text and 'dev.isitsc_scalars.resize(kIsitscDoubleCount)' in scf_cuda_text and 'dev.check_ints.resize(kCheckIntCount)' in scf_cuda_text and 'dev.check_errors.resize(kCheckDoubleCount)' in scf_cuda_text and 'dev.helecz_ints.resize(kHeleczIntCount)' in scf_cuda_text and 'dev.fock_ints.resize(kFockIntCount)' in scf_cuda_text and 'dev.resident_stage_ints.resize(kResidentStageIntCount)' in scf_cuda_text and 'dev.resident_stage_calls.upload(initial_stage_calls' in scf_cuda_text),
    ('resident SCF no late stage resize', all(_source_marker_present(scf_cuda_text, scf_cuda_flat, fragment) for fragment in ('bool device_buffer_ready(const DeviceBuffer<T> &device', 'cudaMemcpyAsync(dev.eimp_p.ptr, dev.p.ptr', 'device_buffer_ready(dev.eimp_p, mpack_count)', 'device_buffer_ready(dev.eimp_pair_updates, 3)', 'device_buffer_ready(dev.check_ints, kCheckIntCount)', 'device_buffer_ready(dev.density_updates, 3)', 'device_buffer_ready(dev.cosmo_cg_x, nps_count)', 'device_buffer_ready(dev.cosmo_status_scalars', 'device_buffer_ready(dev.qe, static_cast<std::size_t>(numat))', 'device_buffer_ready(dev.fock_ints, kFockIntCount)', 'device_buffer_ready(dev.helecz_ints, kHeleczIntCount)', 'device_buffer_ready(dev.isitsc_scalars, kIsitscDoubleCount)', 'device_buffer_ready(dev.cnvgz_ints, kCnvgzIntCount)', 'device_buffer_ready(dev.resident_stage_ints', 'device_buffer_ready(dev.resident_control_ints')) and 'copy_from_device(const T *device_src' not in scf_cuda_text and 'copy_from_device_async(const T *device_src' not in scf_cuda_text and 'if (!dev.eimp_p.copy_from_device_async(' not in scf_cuda_text and 'if (!dev.eimp_pair_updates.resize(3)) break;' not in scf_cuda_text and 'if (!dev.check_errors.resize(kCheckDoubleCount)) break;' not in scf_cuda_text and 'if (!dev.check_ints.resize(kCheckIntCount)) break;' not in scf_cuda_text and 'if (!dev.density_updates.resize(3)) break;' not in scf_cuda_text and 'if (!dev.cosmo_cg_x.resize(static_cast<std::size_t>(nps)))' not in scf_cuda_text and 'if (!dev.cosmo_status_scalars.resize(kCosmoStatusDoubleCount))' not in scf_cuda_text and 'if (!dev.qe.resize(static_cast<std::size_t>(numat))) break;' not in scf_cuda_text and 'if (!dev.fock_ints.resize(kFockIntCount)) break;' not in scf_cuda_text and 'if (!dev.helecz_ints.resize(kHeleczIntCount)) break;' not in scf_cuda_text and 'if (!dev.isitsc_scalars.resize(kIsitscDoubleCount)) break;' not in scf_cuda_text and 'if (!dev.resident_control_ints.resize(kResidentControlIntCount)) return false;' not in scf_cuda_text and 'if (!dev.resident_control_scalars.resize(kResidentControlDoubleCount))' not in scf_cuda_text),
    ('resident COSMO CG preallocation', 'dev.cosmo_cg_x.resize(cosmo_nps_count)' in scf_cuda_text and 'dev.cosmo_cg_scalars.resize(kCosmoCgScalarCount)' in scf_cuda_text),
    ('resident SCF ISITSC scalar pack', 'isitsc_ints' in scf_cuda_text and 'kIsitscIntOkscf' in scf_cuda_text and 'resident final isitsc integer copy' in scf_cuda_text and 'resident isitsc integer copy' not in scf_cuda_text and 'cudaMemcpy(&host_okscf, okscf.ptr' not in scf_cuda_text),
    ('resident SCF DIAGG/ADDHB device control', 'mozyme_diagg2_prepare_control_kernel' in scf_cuda_text and 'mozyme_diagg2_finalize_control_kernel' in scf_cuda_text and 'mozyme_addhb_prepare_control_kernel' in scf_cuda_text and 'mozyme_addhb_finalize_control_kernel' in scf_cuda_text and 'const int *diagg_ints, const double *diagg_scalars' in scf_cuda_text and 'dev.diagg_ints.ptr, dev.diagg_scalars.ptr, dev.addhb_ints.ptr' in scf_cuda_text and 'const int valid_control = control_ints[kAddhbIntOk];' in scf_cuda_text and 'dev.diagg_ints.ptr, kDiaggIntNij, kDiaggIntRetry' in scf_cuda_text and 'dev.addhb_ints.ptr, kAddhbIntNij, kAddhbIntRetry' in scf_cuda_text and 'publish_resident_iteration_outputs_from_gpu' in scf_cuda_text and 'resident final diagg integer copy' in scf_cuda_text and 'resident final addhb integer copy' in scf_cuda_text and 'if (host_nij > 0)' not in scf_cuda_text and 'const bool due = ctx.config.addhb_due' not in scf_cuda_text and 'diagg2_retry_from_state' not in scf_cuda_text and 'compute_addhb_on_gpu(ctx, next_tiny, diagg2_nrejct' not in scf_cuda_text and 'resident diagg integer copy' not in scf_cuda_text and 'resident addhb integer copy' not in scf_cuda_text),
    ('resident SCF ISITSC device energy/control', 'DeviceBuffer<double> isitsc_escf0;' in scf_cuda_text and 'dev.isitsc_escf0.upload(ctx.config.isitsc_escf0, 10)' in scf_cuda_text and 'dev.isitsc_ints.upload(initial_isitsc_ints, kIsitscIntCount)' in scf_cuda_text and 'mozyme_isitsc_resident_kernel' in scf_cuda_text and 'dev.energy_sums.ptr, 2, ctx.config.energy_scale' in scf_cuda_text and 'dev.diagg_scalars.ptr, kDiaggDoubleTiny' in scf_cuda_text and 'zero_ints_if_resident_active(\n            ctx, dev.isitsc_ints.ptr + kIsitscIntOkscf' in scf_cuda_text and 'resident isitsc control reset' in scf_cuda_text and 'dev.isitsc_ints.ptr + kIsitscIntValid' in scf_cuda_text and 'complete_stage_from_int(kMozymeScfStageIsitsc,' in scf_cuda_text and 'compute_isitsc_on_gpu(ctx, &isitsc_ms)' in scf_cuda_text and 'resident final isitsc history copy' in scf_cuda_text and 'resident terminal stage no-op' in scf_cuda_text and 'DeviceBuffer<double> escf0;' not in scf_cuda_text and 'if (!escf0.upload(ctx.config.isitsc_escf0' not in scf_cuda_text and 'compute_isitsc_on_gpu(ctx, diagg_tiny' not in scf_cuda_text and 'compute_isitsc_on_gpu(ctx, status, &isitsc_ms)' not in scf_cuda_text and 'int host_ints[kIsitscIntCount] = {}' not in scf_cuda_text and 'dev.isitsc_ints.upload(host_ints, kIsitscIntCount)' not in scf_cuda_text and 'resident isitsc integer copy' not in scf_cuda_text),
    ('resident SCF iteration accounting', 'call init_config(config, nocc, nvir, resident_max_iter, niter, &' in scf_driver_text and 'config%current_iter = mozyme_c_int_nonnegative_or_zero(current_iter)' in scf_driver_text and 'mod(current_iter + 1, 3) == 0 .and. nhb < 4' in scf_driver_text and 'status%stage_required == GPU_MOZYME_SCF_STAGE_FULL .and.' in scf_driver_text and 'status%stage_completed == GPU_MOZYME_SCF_STAGE_FULL .and.' in scf_driver_text and 'resident_max_iter, niter + 1' not in scf_driver_text),
    ('resident SCF direct wk guard', 'logical :: resident_fock_plan_ready' in scf_driver_text and 'if (id == 0) then\n      resident_fock_plan_ready = mozyme_resident_fock_prepare_plan(' in scf_driver_text and 'resident_fock_plan_full, iorbs, nat, ifact, wj, wj, 0, kopt,' in scf_driver_text and 'resident_fock_plan_full, iorbs, nat, ifact, wj, wk, 0, kopt,' in scf_driver_text and 'resident_fock_plan_partial, iorbs, nat, ifact, wj, wj, fock_mode,' in scf_driver_text and 'resident_fock_plan_partial, iorbs, nat, ifact, wj, wk, fock_mode,' in scf_driver_text and 'mozyme_resident_fock_prepare_plan(resident_fock_plan_full, &\n        iorbs, nat, ifact, wj, wk, 0,' not in scf_driver_text and 'mozyme_resident_fock_prepare_plan(resident_fock_plan_partial, &\n          iorbs, nat, ifact, wj, wk, fock_mode,' not in scf_driver_text),
    ('resident SCF strict terminal control break', all(_source_marker_present(scf_cuda_text, scf_cuda_flat, fragment) for fragment in ('while (resident_loop_guard < strict_loop_limit)', 'ResidentControlSnapshot control{}', 'copy_resident_control_snapshot_from_gpu(*ctx, &control, false)', 'const ResidentReturnDecision return_decision = control.decision;', 'const int previous_iter = ctx->config.current_iter;', 'advance_config_after_resident_control(*ctx, control)', 'resident_stage_status_complete(final_status, previous_iter)')) and 'if (return_decision == ResidentReturnDecision::ContinueOnDevice)' not in scf_cuda_text and 'copy_resident_control_snapshot_from_gpu(*ctx, &strict_control)' not in scf_cuda_text),
    ('resident SCF loop control device', all(_source_marker_present(scf_cuda_text, scf_cuda_flat, fragment) for fragment in ('mozyme_resident_control_advance_kernel', 'compute_resident_control_after_iteration', 'if (!advance_resident_control_on_gpu(ctx, strict_resident)) return false;\n  return copy_resident_control_snapshot_from_gpu(ctx, out);', 'advance_strict_resident_control_on_gpu', 'if (!advance_resident_control_on_gpu(ctx, true)) return false;', 'return apply_resident_pls_restart_if_requested_on_gpu(ctx);', 'if (!advance_strict_resident_control_on_gpu(*ctx))', 'resident loop control advance kernel', 'resident loop control snapshot integer copy', "const int completed_iter =\n      current_iter < 2147483647 ? current_iter + 1 : current_iter;", "else if (completed_iter >= max_iter) {\n    decision = kResidentDecisionIterationExhausted;\n  } else if (strict_resident == 0 &&\n             completed_iter > kMozymeScfPlsSupervisorLastIter) {\n    decision = kResidentDecisionCpuBoundary;\n  }", "current_iter = resident_control_int_or(\n      control_ints, kResidentControlCurrentIter, current_iter);", 'control_ints[kResidentControlDecision] = decision;', 'control_ints[kResidentControlCurrentIter] = next_iter;', 'copy_resident_control_snapshot_from_gpu', 'strict_resident_loop_limit', 'config.current_iter < 0', 'const long long remaining =', 'static_cast<long long>(config.current_iter);', 'const long long restart_budget = static_cast<long long>(config.max_iter);', 'const long long limit = (remaining > 0LL ? remaining : 0LL) + restart_budget;')) and 'apply_cpu_loop_control_for_continuation' not in scf_cuda_text and 'resident_return_decision_after_iteration' not in scf_cuda_text and 'compute_resident_control_after_iteration(*ctx, true, false' not in scf_cuda_text and 'compute_resident_control_after_iteration(*ctx, true, true' not in scf_cuda_text and 'bool compute_resident_control_after_iteration(MozymeScfContext &ctx,\n                                              bool strict_resident,\n                                              const MozymeScfStatus &status' not in scf_cuda_text and 'ctx->config.max_iter > 0 ? ctx->config.max_iter * 2 : 0' not in scf_cuda_text and 'config.current_iter <= 0' not in scf_cuda_text and 'static_cast<long long>(config.max_iter) + remaining' not in scf_cuda_text and 'resident loop control decision copy' not in scf_cuda_text and 'resident loop control integer copy' not in scf_cuda_text and 'resident loop control shift copy' not in scf_cuda_text and 'sizeof(host_decision)' not in scf_cuda_text and 'resident_decision_from_code(host_decision)' not in scf_cuda_text and 'copy_full_snapshot ? copy_resident_control_snapshot_from_gpu(ctx, out)' not in scf_cuda_text),
    ('resident SCF fock rebuild', 'mopac_cuda_mozyme_sparse_fock_run_device_plan' in cuda_text and 'compute_fock_on_gpu' in scf_cuda_text and 'resident fock status reset' in scf_cuda_text and 'kMozymeScfStageFock' in scf_cuda_text and 'MozymeSparseFockPlan' in cuda_text and 'plan->full_coverage' in cuda_text and 'plan->has_executable_work' in cuda_text and 'point_w_values, signature, coverage_complete' in resident_fock_text and 'if (plan && complete == 0) plan->full_coverage = false;' in cuda_text and 'mozyme_resident_fock_plan_full_coverage' in resident_fock_text and 'resident_full_coverage(plan_id) = one_center_cpu_count == 0 .and. &' in resident_fock_text and 'real_pair_cpu_count == 0 .and. point_pair_cpu_count == 0' in resident_fock_text and 'int(gpu_pack_counts(18)) == one_center_cpu_count' in resident_fock_text and 'one_center_coverage gpu_one_center=' in resident_fock_text and 'if (one_center_cpu_count == 0 .and. real_pair_cpu_count == 0 .and. point_pair_cpu_count == 0) then' in resident_fock_text and 'max_resident_fallback_basis = max_resident_diag_basis + 1' in resident_fock_text and 'import :: max_resident_fallback_basis' in resident_fock_text and 'fallback_basis_c(0:max_resident_fallback_basis,0:*)' in resident_fock_text and 'resident_fallback_basis_bin' in resident_fock_text and 'mozyme_sparse_point_kernel' in cuda_text and 'int host_fock_status[kFockIntCount] = {0}' not in scf_cuda_text and 'dev.fock_ints.upload(host_fock_status, kFockIntCount)' not in scf_cuda_text and 'cudaMemset(dev.fock_ints.ptr' not in scf_cuda_text),
    ('resident sparse Fock fail-closed coverage', 'mozyme_sparse_fock_basis_supported' in cuda_text and '!plan->full_coverage' in cuda_text and 'has_executable_work' in cuda_text and 'executable_work <= 0' in cuda_text and '!plan->has_executable_work' in cuda_text and 'setup rejected count mismatch' in resident_fock_text and 'counts_ok = one_pos == one_count .and. pair_pos == pair_count' in resident_fock_text and 'resident_fock_plan_full_coverage' in scf_driver_text and 'backend_resident_fock_partial_coverage' in scf_driver_text),
    ('resident partial Fock point CPU completion', 'mozyme_resident_point_supported, mozyme_point_charge_advance_kr' in fock2z_text and 'mozyme_resident_point_supported(iab, jba, ijbo(ii, jj))' in fock2z_text and 'mozyme_resident_point_supported(iab, jba, nijbo(ii, jj))' in fock2z_text and 'if (resident_fock_done) then\n                call mozyme_point_charge_advance_kr' not in fock2z_text),
    ('resident SCF energy probe', 'mozyme_helecz_kernel' in scf_cuda_text),
    ('resident SCF HELECZ device total', 'totals[2] = offdiag_sums[0] + 0.5 * diag_sums[0];' in scf_cuda_text and 'const int *ok_in' in scf_cuda_text and 'ok_in[kHeleczIntOk] != 1' in scf_cuda_text and 'DeviceBuffer<int> helecz_ints;' in scf_cuda_text and 'resident helecz status reset' in scf_cuda_text and 'dev.helecz_ints.ptr + kHeleczIntOk' in scf_cuda_text and 'const bool publish_energy = energy != nullptr;' in scf_cuda_text and 'dev.energy_sums.resize(3)' in scf_cuda_text and 'd_energy_sums.resize(3)' in scf_cuda_text and 'd_valid.ptr,\n        d_energy_sums.ptr' in scf_cuda_text and 'double host_totals[3] = {0.0, 0.0, 0.0};' in scf_cuda_text and '*energy = host_totals[2];' in scf_cuda_text and 'compute_helecz_on_gpu(ctx, nullptr, &wall_ms)' in scf_cuda_text and 'resident final energy totals copy' in scf_cuda_text and 'status->energy_total = energy_totals[2];' in scf_cuda_text and 'int host_ok_init[kHeleczIntCount] = {1}' not in scf_cuda_text and 'dev.helecz_ints.upload(host_ok_init, kHeleczIntCount)' not in scf_cuda_text and 'resident isitsc helecz totals copy' not in scf_cuda_text and 'const double total = offdiag + 0.5 * diag;' not in scf_cuda_text and 'dev.check_ints.ptr);\n    if (!cuda_context_ok(cudaGetLastError(),\n                         "resident helecz atom kernel"' not in scf_cuda_text),
    ('resident SCF convergence probe', 'mozyme_cnvgz_diff_kernel' in scf_cuda_text and 'density_max' in scf_cuda_text),
    ('resident SCF CNVGZ density history', 'Match the CPU loop: cnvgz mutates density history only while' in scf_cuda_text and 'a GPU-owned no-op so the SCF contract stays complete without CPU fallback' in scf_cuda_text and 'compute_cnvgz_probe_on_gpu(ctx, &cnvgz_ms, true)' in scf_cuda_text and 'resident final cnvgz scalar copy' in scf_cuda_text and 'resident cnvgz totals copy' not in scf_cuda_text),
    ('resident SCF CNVGZ device control', 'kCnvgzControlCount = 6' in scf_cuda_text and 'kCnvgzIntActiveCalls = 1' in scf_cuda_text and 'kCnvgzIntNoopCalls = 2' in scf_cuda_text and 'kCnvgzIntCount = 3' in scf_cuda_text and 'status_ints[kCnvgzIntActiveCalls] += 1' in scf_cuda_text and 'status_ints[kCnvgzIntNoopCalls] += 1' in scf_cuda_text and 'mozyme_cnvgz_finalize_kernel' in scf_cuda_text and 'resident terminal stage no-op' in scf_cuda_text and 'control[kCnvgzDensityRms]' in scf_cuda_text and 'control[kCnvgzFactor] = factor;' in scf_cuda_text and 'const double factor = control[kCnvgzFactor];' in scf_cuda_text and 'const double pmax = control[kCnvgzPmax];' in scf_cuda_text and 'resident_control_uses_three_point_or' in scf_cuda_text and 'mozyme_cnvgz_commit_matrix_kernel' in scf_cuda_text and 'mozyme_cnvgz_commit_diag_kernel' in scf_cuda_text and 'dev.cnvgz_sums.resize(kCnvgzControlCount)' in scf_cuda_text and  'd_cnvgz_sums.resize(kCnvgzControlCount)' in scf_cuda_text and 'dev.candidate.ptr, dev.resident_control_ints.ptr' in scf_cuda_text and 'mozyme_cnvgz_commit_matrix_kernel<<<matrix_blocks, kThreads>>>' in scf_cuda_text and 'mozyme_cnvgz_commit_diag_kernel<<<diag_blocks, kThreads>>>' in scf_cuda_text and 'int host_cnvgz_status[kCnvgzIntCount] = {0}' not in scf_cuda_text and 'dev.cnvgz_ints.upload(host_cnvgz_status, kCnvgzIntCount)' not in scf_cuda_text and 'resident cnvgz factor totals copy' not in scf_cuda_text and 'cnvgz factor totals copy' not in scf_cuda_text and 'mpack, factor, dev.p.ptr' not in scf_cuda_text and 'norbs, mpack, pmax, dev.idiag.ptr' not in scf_cuda_text),
    ('strict resident no partial host publish', all(_source_marker_present(scf_cuda_text, scf_cuda_flat, fragment) for fragment in ('enum class ResidentReturnDecision', 'CompleteAndPublish', 'CpuBoundary', 'IterationExhausted', 'PlsRestart', 'StageFailed', 'kResidentDecisionIterationExhausted', 'kResidentDecisionPlsRestart', 'kResidentDecisionStageFailed', "if (code == kResidentDecisionIterationExhausted) {\n    return ResidentReturnDecision::IterationExhausted;\n  }", "if (code == kResidentDecisionStageFailed) {\n    return ResidentReturnDecision::StageFailed;\n  }", 'return_decision == ResidentReturnDecision::StageFailed', 'strict_resident_request_enabled', 'publish_resident_control_to_status(status, control);', 'advance_strict_resident_control_on_gpu', 'return apply_resident_pls_restart_if_requested_on_gpu(ctx);', 'if (!advance_strict_resident_control_on_gpu(*ctx))', '} else if (return_decision == ResidentReturnDecision::PlsRestart) {', 'MozymeScfStatus final_status = *status;', 'publish_resident_control_to_status(&final_status, control);', '*status = final_status;', 'resident_stage_status_complete', '!resident_stage_status_complete(final_status,', 'status.stage_missing == 0', 'publish_resident_stage_status_from_gpu(*ctx, status)', 'publish_strict_stage_status_or_not_ready', 'final_code = kMozymeScfNotReady;', '[MOZYME GPU SCF] host_commit_only=1', 'phase=final_publication', 'host_commit_marker_enabled', '*status = final_status;\n              final_code = kMozymeScfSuccess;', 'const bool step_complete =\n            run_resident_iteration_on_gpu(*ctx, status, &accumulated_ms, true);', 'run_resident_iteration_on_gpu(*ctx, status, &accumulated_ms,\n                                           false)', 'mozyme_resident_pls_restart_zero_kernel', 'mozyme_resident_pls_restart_finalize_kernel', 'apply_resident_pls_restart_if_requested_on_gpu', 'device_buffer_ready(dev.resident_control_ints', 'device_buffer_ready(dev.resident_control_scalars', 'if (publish_strict_stage_status_or_not_ready()) {', 'final_code = kMozymeScfCpuBoundary;\n              status->ready = 0;', 'advance_config_after_resident_control(*ctx, control)', 'resident_stage_status_complete(final_status, previous_iter)')) and 'backend_cpu_boundary' in scf_driver_text and 'if (must_return_to_fortran_after_iteration(*ctx, *status))' not in scf_cuda_text and 'if (!strict_resident && !copy_resident_state_to_host(*ctx))' not in scf_cuda_text and 'if (strict_resident) {\n          final_code = kMozymeScfCpuBoundary;' not in scf_cuda_text and 'if (strict_resident) continue;' not in scf_cuda_text and "run_resident_iteration_on_gpu(*ctx, status, &accumulated_ms,\n                                        !strict_resident)" not in scf_cuda_text and "if (!run_resident_iteration_on_gpu(*ctx, status, &accumulated_ms,\n                                           true))" not in scf_cuda_text and 'resident PLS restart state copy' not in scf_cuda_text and 'dev.resident_control_ints.upload(control_ints,' not in scf_cuda_text and 'dev.resident_control_scalars.upload(control_scalars,' not in scf_cuda_text and 'dev.isitsc_ints.upload(isitsc_ints,' not in scf_cuda_text and 'dev.pls_ints.upload(pls_ints,' not in scf_cuda_text and 'compute_resident_control_after_iteration(*ctx, true, false' not in scf_cuda_text and 'compute_resident_control_after_iteration(*ctx, true, true' not in scf_cuda_text and 'resident loop control decision copy' not in scf_cuda_text and 'sizeof(host_decision)' not in scf_cuda_text and 'resident_decision_from_code(host_decision)' not in scf_cuda_text and 'copy_full_snapshot ? copy_resident_control_snapshot_from_gpu(ctx, out)' not in scf_cuda_text and 'if (return_decision == ResidentReturnDecision::ContinueOnDevice)' not in scf_cuda_text and 'reset_resident_after_pls_restart(*ctx)' not in scf_cuda_text and 'publish_cosmo_status(status, ctx);\n  if (final_code != kMozymeScfSuccess)' not in scf_cuda_text),
    ('resident final reorth validates recomputed GPU stages', 'validate_final_reorth_rebuild_from_gpu' in scf_cuda_text and 'resident final reorth density status copy' in scf_cuda_text and 'resident final reorth fock status copy' in scf_cuda_text and 'resident final reorth helecz status copy' in scf_cuda_text and 'density_status[1] == 1' in scf_cuda_text and 'density_status[0] == density_status[2]' in scf_cuda_text and 'fock_status[kFockIntOk] == 1' in scf_cuda_text and 'helecz_status[kHeleczIntOk] == 1' in scf_cuda_text and 'publish_cosmo_status_from_gpu(ctx, status)' in scf_cuda_text),
    ('strict/full request blocks CPU fallback', 'MOPAC_MOZYME_GPU_STRICT' in scf_driver_text and 'MOPAC_MOZYME_FULL_SCF_GPU' in scf_driver_text and 'MOPAC_MOZYME_GPU_STRICT' in run_mopac_text and 'MOPAC_MOZYME_FULL_SCF_GPU' in run_mopac_text and 'MOPAC_MOZYME_GPU_STRICT' in resident_fock_text and 'MOPAC_MOZYME_FULL_SCF_GPU' in resident_fock_text and 'MOPAC_MOZYME_GPU_STRICT' in plan_text and 'MOPAC_MOZYME_FULL_SCF_GPU' in plan_text and 'strict_diagg_cpu_fallback' in diagg_text and 'public :: mozyme_gpu_scf_no_fallback_required' in scf_driver_text and 'mozyme_gpu_scf_no_fallback_required = no_fallback_required' in scf_driver_text and 'no_fallback_required = strict_requested .or. full_scf_requested' in scf_driver_text and "env_is_one('MOPAC_MOZYME_RESIDENT_SCF')" in scf_driver_text and "env_is_one('MOPAC_MOZYME_SCF_GPU')" in scf_driver_text and 'strict_resident_request_enabled' in scf_cuda_text and 'resident_scf_request_enabled' in scf_cuda_text and 'MOPAC_MOZYME_SCF_GPU' in scf_cuda_text and 'MOPAC_MOZYME_RESIDENT_SCF' in scf_cuda_text and 'resident_strict_required = mozyme_gpu_scf_no_fallback_required()' in iter_text and 'block_on_failure=resident_strict_required' in iter_text and 'strict_resident_density = mozyme_gpu_scf_no_fallback_required()' in density_text and 'mozyme_gpu_scf_no_fallback_required' in fillij_text and 'mozyme_gpu_scf_no_fallback_required' in pinout_text and "resident_env_requested('MOPAC_MOZYME_RESIDENT_SCF') .or. &" not in resident_fock_text and "resident_env_requested('MOPAC_MOZYME_SCF_GPU')" in resident_fock_text and "mozyme_plan_env_enabled('MOPAC_MOZYME_RESIDENT_SCF')" not in plan_text and "mozyme_plan_env_enabled('MOPAC_MOZYME_SCF_GPU')" in plan_text and 'env_enabled("MOPAC_MOZYME_SCF_GPU") ||\n         env_enabled("MOPAC_MOZYME_RESIDENT_SCF")' not in scf_cuda_text and "error stop 'MOZYME_GPU preflight found no GPU production work'" in plan_text and 'mozyme_gpu_scf_no_fallback_required' in run_mopac_text and 'MOPAC_MOZYME_SCF_GPU' in run_mopac_text and 'strict_gpu_disabled_by_mopac_nogpu' in run_mopac_text and 'strict_gpu_disabled_by_scftask_cpu' in run_mopac_text and 'strict_gpu_disabled_by_mozyme_gpu_off' in run_mopac_text and 'strict_not_gpu_build' in run_mopac_text and 'strict_resident_fock_gpu_disabled' in run_mopac_text and 'strict_mozyme_scf = mozyme .and. mozyme_gpu_scf_no_fallback_required()' in run_mopac_text and 'if (strict_mozyme_scf) then\n        l_OLDDEN = .true.' in run_mopac_text and 'strict_run_mopac_olden_host_restore' in run_mopac_text and 'goto 101' in run_mopac_text and 'if (.not. strict_mozyme_scf .and. .not. l_OLDDEN .and.' in run_mopac_text and 'strict_resident_fock_cpu_fallback' in fock2z_text and 'strict_fock1_cpu_fallback' in fock1_text),
    ('resident SCF portable double atomics', 'atomicAdd_double' in scf_cuda_text and 'atomicCAS' in scf_cuda_text),
    ('resident SCF device-side scalar reductions', 'mozyme_helecz_reduce_kernel' in scf_cuda_text and 'energy_sums' in scf_cuda_text and 'mozyme_cnvgz_diff_reduce_kernel' in scf_cuda_text and 'mozyme_cnvgz_factor_reduce_kernel' in scf_cuda_text and 'cnvgz_sums' in scf_cuda_text and 'std::vector<double> host_max' not in scf_cuda_text and 'std::vector<double> host_sums' not in scf_cuda_text and 'std::vector<double> host_faca' not in scf_cuda_text),
    ('resident SCF device checkpoint', all(_source_marker_present(scf_cuda_text, scf_cuda_flat, fragment) for fragment in ('save_resident_checkpoint', 'restore_resident_checkpoint', 'checkpoint_isitsc_ints', 'checkpoint_isitsc_escf0', 'checkpoint_resident_control_ints', 'copy_device_buffer(dev.checkpoint_isitsc_ints, dev.isitsc_ints)', 'copy_device_buffer(dev.isitsc_ints, dev.checkpoint_isitsc_ints)', 'checkpoint_config', 'copy_resident_state_to_host(*ctx)) {\n            *status = checkpoint', 'resize_resident_checkpoint_buffers', 'resident checkpoint device copy', 'stage_and_commit_device_vector(ctx.state.idiag, dev.idiag, norbs_count', 'stage_and_commit_device_vector(ctx.state.nncf, dev.nncf, nocc_slots', 'stage_and_commit_device_vector(ctx.state.idiag, dev.idiag, norbs_count', 'stage_and_commit_device_vector(ctx.state.nncf, dev.nncf, nocc_slots')) and 'return src.ptr && dst.copy_from_device(src.ptr, src.count);' not in scf_cuda_text and 'copy_device_to_host_vector(host_idiag, dev.idiag)' not in scf_cuda_text and 'commit_host_vector(ctx.state.idiag, host_idiag)' not in scf_cuda_text),
    ('resident SCF preallocated checkpoints', all(_source_marker_present(scf_cuda_text, scf_cuda_flat, fragment) for fragment in ('bool resize_checkpoint_like(DeviceBuffer<T> &checkpoint', 'bool resize_resident_checkpoint_buffers(MozymeScfDeviceState &dev)', 'resize_checkpoint_like(dev.checkpoint_p, dev.p)', 'resize_checkpoint_like(dev.checkpoint_cosmo_qscnet', 'resize_checkpoint_like(dev.checkpoint_resident_control_ints', 'if (!resize_resident_checkpoint_buffers(dev)) return false;', 'if (!src.ptr || !device_buffer_ready(dst, src.count)) return false;', 'cudaMemcpy(dst.ptr, src.ptr, src.count * sizeof(T),', 'resident checkpoint device copy')) and 'return src.ptr && dst.copy_from_device(src.ptr, src.count);' not in scf_cuda_text),
    ('strict resident Fock abort marker', 'subroutine strict_resident_fock_abort(reason, message)' in resident_fock_text and '[MOZYME GPU SCF] status=strict_abort reason=' in resident_fock_text and 'strict_resident_fock_disabled' in resident_fock_text and 'strict_resident_fock_legacy_host_copy' in resident_fock_text and 'strict_resident_fock_work_alloc_failed' in resident_fock_text and "error stop 'MOZYME GPU strict resident Fock abort'" in resident_fock_text and 'allocate(f_work(mpack), stat=alloc_stat)' in resident_fock_text and 'f_work(1:mpack) = f(1:mpack)' in resident_fock_text and 'f(1:mpack) = f_work(1:mpack)' in resident_fock_text),
    ('strict legacy Fock2 abort marker', 'resident_strict_requested, &\n     strict_resident_fock_abort' in fock2z_text and 'strict_legacy_fock2_failed' in fock2z_text and 'MOZYME GPU strict legacy Fock2 failed' in fock2z_text),
    ('strict ADDHB CPU guard', 'mozyme_gpu_scf_no_fallback_required()' in addhb_text and 'strict_addhb_cpu_fallback' in addhb_text and 'MOZYME GPU strict resident SCF does not support CPU ADDHB' in addhb_text),
    ('strict CHECK CPU guard', 'mozyme_gpu_scf_no_fallback_required()' in check_text and 'strict_check_cpu_fallback' in check_text and 'strict_check_gpu_host_fallback' in check_text and 'MOZYME GPU strict resident SCF does not support CPU LMO normalization check' in check_text),
    ('strict DIAGG CPU guard', 'mozyme_gpu_scf_no_fallback_required()' in diagg_text and 'strict_diagg_cpu_fallback' in diagg_text and 'MOZYME GPU strict resident SCF does not support CPU DIAGG' in diagg_text),
    ('strict REORTH CPU guard', 'mozyme_gpu_scf_no_fallback_required()' in reorth_cpu_text and 'strict_reorth_cpu_fallback' in reorth_cpu_text and 'MOZYME GPU strict resident SCF does not support CPU reorthogonalization' in reorth_cpu_text),
    ('strict TIDY CPU guard', 'mozyme_gpu_scf_no_fallback_required()' in tidy_text and 'strict_tidy_cpu_fallback' in tidy_text and 'MOZYME GPU strict resident SCF does not support CPU LMO tidy' in tidy_text),
    ('strict Fock1 CPU guard', 'mozyme_gpu_scf_no_fallback_required()' in fock1_text and 'strict_fock1_cpu_fallback' in fock1_text and 'MOZYME GPU strict resident SCF does not support CPU one-center Fock construction' in fock1_text),
    ('strict pinout CPU guard', '[MOZYME CPU pinout]' in pinout_text and 'mozyme_gpu_scf_no_fallback_required()' in pinout_text and "mozyme_gpu_strict_abort('strict_pinout_cpu_fallback'" in pinout_text and 'MOZYME GPU strict resident SCF does not support CPU pinout' in pinout_text and "error stop 'MOZYME GPU strict pinout abort'" not in pinout_text),
    ('resident SCF strict resident loop gate', 'MOPAC_MOZYME_GPU_STRICT' in scf_cuda_text and 'MOPAC_MOZYME_FULL_SCF_GPU' in scf_cuda_text and 'MOPAC_MOZYME_SCF_STRICT_RESIDENT' in scf_cuda_text and 'MOPAC_MOZYME_SCF_GPU' in scf_cuda_text and 'MOPAC_MOZYME_RESIDENT_SCF' in scf_cuda_text and 'strict_resident_request_enabled' in scf_cuda_text and 'resident_scf_request_enabled' in scf_cuda_text and 'kMozymeScfPlsSupervisorLastIter' in scf_cuda_text and 'compute_resident_control_after_iteration' in scf_cuda_text and 'env_is_one = env_enabled(var_name)' in scf_driver_text and 'step_success = .false.' in scf_driver_text and 'env_enabled("MOPAC_MOZYME_SCF_GPU") ||\n         env_enabled("MOPAC_MOZYME_RESIDENT_SCF")' not in scf_cuda_text),
    (
        'resident SCF stage status device-resident',
        all(_source_marker_present(scf_cuda_text, scf_cuda_flat, fragment) for fragment in (
            'DeviceBuffer<int> resident_stage_ints;',
            'kResidentStageCompleted',
            'mozyme_resident_stage_reset_kernel',
            'mozyme_resident_stage_mark_if_int_kernel',
            'resident_control_ints[kResidentControlDecision] =\n          kResidentDecisionStageFailed;',
            'DeviceBuffer<int> cnvgz_ints;',
            'DeviceBuffer<int> fock_ints;',
            'kCnvgzIntOk',
            'kFockIntOk',
            'reset_resident_stage_status_on_gpu',
            'mark_resident_stage_if_int_on_gpu',
            'publish_resident_stage_status_from_gpu',
            'resident_stage_status_complete',
            'return completed == kMozymeScfStageFull;',
            'status.stage_missing == 0',
            'complete_stage_from_int',
            'const bool poll_stage_status = publish_iteration_outputs;',
            'if (publish_iteration_outputs) {\n    publish_stage_status();',
            'completed_full_stage_mask(completed) ? kMozymeScfSuccess',
            'complete_stage_from_int(kMozymeScfStageCheck, ctx.device.check_ints.ptr',
            'complete_stage_from_int(kMozymeScfStageEimp,',
            'complete_stage_from_int(kMozymeScfStageDiagg,',
            'complete_stage_from_int(kMozymeScfStageDensity,',
            'complete_stage_from_int(kMozymeScfStageAddhb,',
            'complete_stage_from_int(kMozymeScfStageCnvgz,',
            'complete_stage_from_int(kMozymeScfStageFock,',
            'complete_stage_from_int(kMozymeScfStageHelecz,',
            'complete_stage_from_int(kMozymeScfStageIsitsc,',
        )) and 'mozyme_resident_stage_mark_kernel' not in scf_cuda_text and 'mark_resident_stage_on_gpu' not in scf_cuda_text and 'complete_stage(kMozymeScfStage' not in scf_cuda_text and 'status->iterations = ctx.config.current_iter;\n  publish_stage_status();\n  if (publish_iteration_outputs) {' not in scf_cuda_text and 'if (!mark_resident_stage_if_int_on_gpu(ctx, stage_bits, values,\n                                           value_slot, expected_value)) {\n      return false;\n    }\n    if (!publish_resident_stage_status_from_gpu(ctx, status)) return false;' not in scf_cuda_text,
    ),
    ('resident SCF eimp probe', 'compute_eimp_probe_on_gpu(ctx, &eimp_ms)' in scf_cuda_text and 'kMozymeScfStageEimp' in scf_cuda_text and 'resident eimp status reset' in scf_cuda_text and 'dev.eimp_pair_updates.ptr + 2' in scf_cuda_text and 'complete_stage_from_int(kMozymeScfStageEimp,' in scf_cuda_text and 'if (base < 0) return;' in scf_cuda_text and 'atomicAdd(expected_updates, 1);' in scf_cuda_text and 'mozyme_update_status_finalize_kernel' in scf_cuda_text and 'mozyme_update_status_finalize_kernel<<<1, 1>>>(\n        dev.eimp_pair_updates.ptr, 0, dev.resident_control_ints.ptr)' in scf_cuda_text and 'int host_pair_status[3] = {0, 1, 0};' not in scf_cuda_text and 'dev.eimp_pair_updates.upload(host_pair_status, 3)' not in scf_cuda_text and 'host_pair_status[1] != 1' not in scf_cuda_text and 'cudaMemcpy(host_pair_status, dev.eimp_pair_updates.ptr' not in scf_cuda_text and 'host_pair_status[0] != host_pair_status[2]' not in scf_cuda_text and 'expected_pair_updates' not in scf_cuda_text and 'expected_updates_ll' not in scf_cuda_text and 'cudaMemcpy(&host_updates, dev.eimp_pair_updates.ptr' not in scf_cuda_text),
    ('resident SCF runtime risk fixes', 'atomicExch(&atom_flags[atom - 1], 1)' in scf_cuda_text and 'mozyme_update_status_finalize_kernel<<<1, 1>>>(\n        dev.density_updates.ptr, 1, dev.resident_control_ints.ptr)' in scf_cuda_text and 'mozyme_resident_stage_mark_if_int_kernel' in scf_cuda_text and 'resident_control_ints[kResidentControlDecision] =\n          kResidentDecisionStageFailed;' in scf_cuda_text and 'resident check integer reset' in scf_cuda_text and 'resident fock status reset' in scf_cuda_text and 'resident helecz status reset' in scf_cuda_text and  'host_pair_status[2] <= 0' not in scf_cuda_text and 'host_status[2] <= 0' not in scf_cuda_text and 'host_ints[kCheckIntOk] == 1' not in scf_cuda_text and 'expected_density_updates' not in scf_cuda_text and 'host_updates != expected_density_updates' not in scf_cuda_text and 'if (base < 0) {\n      atomicExch(ok_out, 0);' in scf_cuda_text and 'resident cnvgz candidate kernels' in scf_cuda_text and 'publish_iteration_outputs || strict_resident_request_enabled()' not in scf_cuda_text),
    ('production cnvgz GPU ABI', 'mopac_cuda_mozyme_cnvgz' in scf_cuda_text and 'MOZYME GPU cnvgz' in report_text),
    ('production helecz GPU ABI', 'mopac_cuda_mozyme_helecz' in scf_cuda_text and 'MOZYME GPU helecz' in report_text),
    ('production eimp GPU ABI', 'mopac_cuda_mozyme_eimp' in scf_cuda_text and 'MOZYME GPU eimp' in report_text),
    ('production density batch GPU ABI', 'mopac_cuda_mozyme_density_values' in scf_cuda_text and 'MOZYME GPU density_batch' in report_text and 'MOPAC_MOZYME_DENSITY_BATCH_GPU' in density_text),
    ('production diagg1 construct GPU ABI', 'mopac_cuda_mozyme_diagg1_construct' in scf_cuda_text and 'MOZYME GPU diagg1_construct' in report_text and 'MOPAC_MOZYME_DIAGG1_CONSTRUCT_GPU' in diagg1_text),
    ('production diagg1 aocc GPU ABI', 'mopac_cuda_mozyme_diagg1_aocc' in scf_cuda_text and 'MOZYME GPU diagg1_aocc' in report_text),
    ('production diagg1 avir GPU ABI', 'mopac_cuda_mozyme_diagg1_avir' in scf_cuda_text and 'MOZYME GPU diagg1_avir' in report_text),
    ('production diagg2 rotate GPU ABI', 'mopac_cuda_mozyme_diagg2_rotate' in scf_cuda_text and 'MOZYME GPU diagg2_rotate' in report_text and 'MOPAC_MOZYME_DIAGG2_ROTATE_GPU' in diagg2_text),
    ('production diagg2 rotprep GPU ABI', 'mopac_cuda_mozyme_diagg2_rotprep' in scf_cuda_text and 'MOZYME GPU diagg2_rotprep' in report_text and 'MOPAC_MOZYME_DIAGG2_ROTPREP_GPU' in diagg2_text),
    ('production isitsc GPU ABI', 'mopac_cuda_mozyme_isitsc' in scf_cuda_text and 'MOZYME GPU isitsc' in report_text and 'MOPAC_MOZYME_ISITSC_GPU' in isitsc_text and 'env["MOPAC_MOZYME_ISITSC_GPU"] = "1"' in report_text),
]
for label, ok in needles:
    print(f'{label}:', 'OK' if ok else 'MISSING')
    if not ok:
        raise RuntimeError(f'The uploaded zip does not contain the current MOZYME GPU code: {label}')


## 3. Install Build Tools

In [ ]:
run(['apt-get', 'update'])
run([
    'apt-get', 'install', '-y',
    'cmake', 'gfortran', 'ninja-build', 'libblas-dev', 'liblapack-dev'
])


## 4. Configure GPU Build

In [ ]:
from pathlib import Path
import re
import shutil

source_dir = Path(source_dir)
if BUILD_DIR.exists():
    shutil.rmtree(BUILD_DIR)

base_cmake_cmd = [
    'cmake', '-S', source_dir, '-B', BUILD_DIR, '-GNinja',
    '-DGPU=ON',
    '-DTESTS=OFF',
    '-DCMAKE_BUILD_TYPE=Release',
]

configure_log = CONTENT_DIR / 'mopac_colab_configure.log'
result = run(base_cmake_cmd + ['-DCUDA_ARCHS=native'], log_path=configure_log, check=False)
if result.returncode != 0:
    print('Native CUDA architecture detection failed; retrying with the portable A100/H100/common fat binary list.')
    if BUILD_DIR.exists():
        shutil.rmtree(BUILD_DIR)
    result = run(base_cmake_cmd + ['-DCUDA_ARCHS=all'], log_path=configure_log, check=True)


targets = run(
    ['ninja', '-C', BUILD_DIR, '-t', 'targets'],
    log_path=CONTENT_DIR / 'mopac_colab_ninja_targets.log',
    check=True,
)
target_help = run(
    ['cmake', '--build', BUILD_DIR, '--target', 'help'],
    log_path=CONTENT_DIR / 'mopac_colab_build_target_help.log',
    check=False,
)
cache_text = (BUILD_DIR / 'CMakeCache.txt').read_text(encoding='utf-8', errors='ignore')
if 'GPU:BOOL=ON' not in cache_text:
    raise RuntimeError('CMake did not configure with GPU:BOOL=ON; this package cannot prove the GPU path.')
arch_match = re.search(r'^CMAKE_CUDA_ARCHITECTURES(?::[^=]*)?=(.+)$', cache_text, re.MULTILINE)
if not arch_match or not arch_match.group(1).strip():
    raise RuntimeError('CMake did not record CMAKE_CUDA_ARCHITECTURES; CUDA target architecture is not proven.')
configured_cuda_archs = arch_match.group(1).strip()
if configured_cuda_archs.lower() in {'0', 'false', 'off', 'no'}:
    raise RuntimeError(f'CMAKE_CUDA_ARCHITECTURES is disabled: {configured_cuda_archs!r}')
print('Configured CUDA architectures:', configured_cuda_archs)
def configured_target(name):
    text = targets.stdout + '\n' + target_help.stdout
    return (
        f'{name}:' in text
        or f'/{name}:' in text
        or f'... {name}' in text
        or re.search(rf'(?m)^\s*{re.escape(name)}\s*$', text) is not None
    )

HAS_GPU_BENCH = configured_target('mopac-gpu-bench')
HAS_RESIDENT_FOCK_PAIR_TEST = configured_target('mopac-gpu-resident-fock-pair-compare')
if HAS_GPU_BENCH:
    print('Configured build contains mopac-gpu-bench.')
else:
    print('Configured build does not contain mopac-gpu-bench; low-level benchmark sections will be skipped.')
if HAS_RESIDENT_FOCK_PAIR_TEST:
    print('Configured build contains mopac-gpu-resident-fock-pair-compare.')
else:
    print('Configured build does not contain mopac-gpu-resident-fock-pair-compare; standalone resident Fock pair check will be skipped.')


## 5. Build MOPAC Target

In [ ]:
build_targets = ['mopac']
if globals().get('HAS_GPU_BENCH', False):
    build_targets.insert(0, 'mopac-gpu-bench')
if globals().get('HAS_RESIDENT_FOCK_PAIR_TEST', False):
    build_targets.append('mopac-gpu-resident-fock-pair-compare')

for target in build_targets:
    run(
        ['cmake', '--build', BUILD_DIR, '--target', target, '--parallel', '2'],
        log_path=CONTENT_DIR / f'mopac_colab_build_{target}.log',
    )

expected_exes = [BUILD_DIR / target for target in build_targets]

for exe in expected_exes:
    executable = exe.exists() and os.access(exe, os.X_OK)
    print(exe, 'OK' if executable else 'MISSING_OR_NOT_EXECUTABLE')
    if not executable:
        raise RuntimeError(f'Build did not create an executable {exe}')

if globals().get('HAS_RESIDENT_FOCK_PAIR_TEST', False):
    run(
        [BUILD_DIR / 'mopac-gpu-resident-fock-pair-compare'],
        log_path=CONTENT_DIR / 'mopac_colab_resident_fock_pair_compare.log',
    )
else:
    print('Skipping standalone resident Fock pair compare because the target was not configured.')


## 6. Run Quick Benchmark

In [ ]:
def _run_low_level_quick():
    if not globals().get('HAS_GPU_BENCH', False):
        print('Skipping this low-level benchmark because mopac-gpu-bench was not configured.')
        return
    bench = BUILD_DIR / 'mopac-gpu-bench'
    cmd = [
        bench,
        '--gemm=1024,1024,256,10',
        '--syrk=1024,256,10',
        '--dsyevd=256,2',
        '--rot1=512,3',
        '--rot2=512,3',
    ]
    run(cmd, log_path=CONTENT_DIR / 'mopac_low_level_quick.log')
run_or_defer_low_level('quick low-level benchmark', _run_low_level_quick)


## 7. Run Larger Benchmark

In [ ]:
def _run_low_level_large():
    if not globals().get('HAS_GPU_BENCH', False):
        print('Skipping this low-level benchmark because mopac-gpu-bench was not configured.')
        return
    bench = BUILD_DIR / 'mopac-gpu-bench'
    cmd = [
        bench,
        '--gemm=2048,2048,512,10',
        '--syrk=2048,512,5',
        '--dsyevd=512,1',
        '--rot1=1024,2',
        '--rot2=1024,2',
    ]
    run(cmd, log_path=CONTENT_DIR / 'mopac_low_level_large.log')
run_or_defer_low_level('larger low-level benchmark', _run_low_level_large)


## 8. Optional Library Timing

In [ ]:
def _run_low_level_library_timing():
    if not globals().get('HAS_GPU_BENCH', False):
        print('Skipping this low-level benchmark because mopac-gpu-bench was not configured.')
        return
    env = os.environ.copy()
    env['MOPAC_GPU_VERBOSE'] = '1'
    env['MOPAC_GPU_PROFILE'] = '2'

    bench = BUILD_DIR / 'mopac-gpu-bench'
    cmd = [
        bench,
        '--gemm=2048,2048,512,5',
        '--syrk=2048,512,3',
        '--dsyevd=512,1',
        '--rot1=1024,1',
        '--rot2=1024,1',
    ]
    run(cmd, env=env, log_path=CONTENT_DIR / 'mopac_low_level_profile.log')
run_or_defer_low_level('library timing benchmark', _run_low_level_library_timing)


## 9. Generate Benchmark Report And Plots

In [ ]:
from google.colab import files
from IPython.display import Image, Markdown, display
from pathlib import Path
import json
import shutil

def _run_low_level_report():
    if not globals().get('HAS_GPU_BENCH', False):
        print('Skipping this low-level benchmark because mopac-gpu-bench was not configured.')
        return
    strict_readiness_ready_for_low_level_artifacts = bool(globals().get('full_scf_readiness_passed', False) and globals().get('direct_cosmo_readiness_passed', False))
    bench = BUILD_DIR / 'mopac-gpu-bench'
    report_dir = CONTENT_DIR / 'mopac_gpu_report'
    bundle_zip = CONTENT_DIR / ('mopac_gpu_publication_data.zip' if strict_readiness_ready_for_low_level_artifacts else 'mopac_gpu_diagnostic_data.zip')
    if report_dir.exists():
        shutil.rmtree(report_dir)
    if bundle_zip.exists():
        bundle_zip.unlink()

    cmd = [
        'python3', 'scripts/gpu_benchmark_report.py', bench,
        '--profile', 'standard',
        '--accuracy-size', '256,128',
        '--out-dir', report_dir,
        '--bundle-zip', bundle_zip,
    ]
    run(cmd, cwd=source_dir, log_path=CONTENT_DIR / 'mopac_gpu_report.log')
    if not strict_readiness_ready_for_low_level_artifacts:
        print('NON-PROOF DIAGNOSTIC: low-level GPU benchmark report is not full-SCF proof.')

    readme_path = report_dir / 'README.md'
    if readme_path.exists():
        display(Markdown(readme_path.read_text(encoding='utf-8')))
    for name in [
        'gpu_throughput_gflops.png',
        'gpu_wrapper_times.png',
        'gpu_first_call_overhead.png',
        'gpu_accuracy.png',
    ]:
        path = report_dir / name
        if path.exists():
            display(Image(filename=str(path)))

    files.download(str(bundle_zip))
run_or_defer_low_level('low-level benchmark report', _run_low_level_report)


## 10. Generate Molecule Benchmark Report And Download Data

In [ ]:
from google.colab import files
from IPython.display import Image, Markdown, display
from pathlib import Path
import shutil

mopac = BUILD_DIR / 'mopac'
molecule_report_dir = CONTENT_DIR / 'mopac_molecule_report'
molecule_bundle_zip = CONTENT_DIR / 'mopac_molecule_publication_data.zip'
full_scf_readiness_report_dir = CONTENT_DIR / 'mopac_full_scf_readiness_report'
full_scf_readiness_bundle_zip = CONTENT_DIR / 'mopac_full_scf_readiness_data.zip'
direct_cosmo_readiness_report_dir = CONTENT_DIR / 'mopac_direct_cosmo_readiness_report'
direct_cosmo_readiness_bundle_zip = CONTENT_DIR / 'mopac_direct_cosmo_readiness_data.zip'
for report_dir in [molecule_report_dir, full_scf_readiness_report_dir, direct_cosmo_readiness_report_dir]:
    if report_dir.exists():
        shutil.rmtree(report_dir)
for bundle_zip in [molecule_bundle_zip, full_scf_readiness_bundle_zip, direct_cosmo_readiness_bundle_zip]:
    if bundle_zip.exists():
        bundle_zip.unlink()

prep = run(
    ['python3', 'scripts/prepare_publication_benchmark_inputs.py', '--no-download'],
    cwd=source_dir,
    log_path=CONTENT_DIR / 'mopac_prepare_inputs.log',
    check=False,
)
if prep.returncode != 0:
    print('Prepared PDB files were not found in the zip; retrying with download enabled.')
    run(
        ['python3', 'scripts/prepare_publication_benchmark_inputs.py'],
        cwd=source_dir,
        log_path=CONTENT_DIR / 'mopac_prepare_inputs.log',
    )

run(
    ['python3', 'scripts/collect_existing_mopac_references.py'],
    cwd=source_dir,
    log_path=CONTENT_DIR / 'mopac_existing_references.log',
)

hydrogenation_timeout = 2400
# RNA and the largest protein can exceed two hours on busy/free Colab A100 sessions.
molecule_timeout = 14400
# Keep this at 0.0 for a usage preflight only. Set to 1.02 or higher to abort if the GPU
# preflight is not faster than CPU before spending hours on the full molecule benchmark.
preflight_min_speedup = 0.0
# Strict readiness is the default Colab proof path with CPU companion comparison. It does not start the long molecule benchmark.
require_full_scf_gpu = True
# Strict readiness probe only. This does not start the long molecule benchmark or GPU preflight.
run_full_scf_readiness_probe = True
# Direct COSMO proof uses a small peptide with EPS=78.4 to exercise resident addfckz.
run_direct_cosmo_readiness_probe = True
# Keep this False for the default proof path; the publication benchmark is long.
run_molecule_benchmark = False
# Optional fatal diagnostic for manually checking the resident boundary. Keep this False for the strict proof path with CPU companion comparison.
run_resident_scf_smoke = False
# Leave this False for a proof run. Set True only for manual debugging where no proof is claimed.
allow_non_proof_run = False
full_scf_readiness_passed = False
direct_cosmo_readiness_passed = False

def _as_int(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


def _as_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _cnvgz_active_noop_ready(active_calls, noop_calls):
    return active_calls is not None and noop_calls is not None and active_calls >= 0 and noop_calls >= 0 and active_calls + noop_calls > 0

def assert_full_scf_readiness_artifact(path):
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f'Strict full-SCF readiness artifact was not written: {path}')
    payload = json.loads(path.read_text(encoding='utf-8'))
    row = payload.get('row') if isinstance(payload, dict) else None
    if not isinstance(row, dict):
        raise RuntimeError(f'Strict full-SCF readiness artifact has no row object: {path}')
    reasons = payload.get('reasons') or []
    failures = []
    if payload.get('contract_version') != 'resident-scf-strict-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-resident-cg-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v58':
        failures.append(f"contract_version={payload.get('contract_version')!r}")
    if payload.get('feature_set') != 'mozyme-full-scf-gpu-makvec-relocal-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v38-20260702':
        failures.append(f"feature_set={payload.get('feature_set')!r}")
    if payload.get('benchmark_scope') != 'molecule_mozyme_full_scf_gpu':
        failures.append(f"benchmark_scope={payload.get('benchmark_scope')!r}")
    expected_publication_claim = (
        'Complete MOZYME full-SCF GPU compute is claimed only for rows that pass '
        'the strict resident-SCF readiness contract with no parsed CPU fallback '
        'and, in the Colab proof path, a same-input CPU companion energy comparison; '
        'final/status host commits remain allowed for MOPAC state publication.'
    )
    if payload.get('publication_claim') != expected_publication_claim:
        failures.append(f"publication_claim={payload.get('publication_claim')!r}")
    if payload.get('source_zip_sha256') != source_zip_sha256:
        failures.append(f"source_zip_sha256={payload.get('source_zip_sha256')!r}")
    if _as_int(payload.get('source_zip_verified')) != 1:
        failures.append(f"source_zip_verified={payload.get('source_zip_verified')!r}")
    if _as_int(payload.get('source_metadata_from_zip')) != 1:
        failures.append(f"source_metadata_from_zip={payload.get('source_metadata_from_zip')!r}")
    if payload.get('source_manifest_sha256') != manifest_sha256:
        failures.append(f"source_manifest_sha256={payload.get('source_manifest_sha256')!r}")
    if payload.get('source_provenance_marker_contract_version') != 'mopac-colab-source-markers-explicit-proof-v83':
        failures.append(f"source_provenance_marker_contract_version={payload.get('source_provenance_marker_contract_version')!r}")
    if payload.get('source_provenance_marker_contract_sha256') != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256:
        failures.append(f"source_provenance_marker_contract_sha256={payload.get('source_provenance_marker_contract_sha256')!r}")
    if payload.get('source_features_marker_contract_version') != 'mopac-colab-source-markers-explicit-proof-v83':
        failures.append(f"source_features_marker_contract_version={payload.get('source_features_marker_contract_version')!r}")
    if payload.get('source_features_marker_contract_sha256') != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256:
        failures.append(f"source_features_marker_contract_sha256={payload.get('source_features_marker_contract_sha256')!r}")
    if _as_int(payload.get('source_manifest_entry_count')) != _as_int(payload.get('source_manifest_verified_files')):
        failures.append(f"source_manifest_verified_files={payload.get('source_manifest_verified_files')!r} source_manifest_entry_count={payload.get('source_manifest_entry_count')!r}")
    if _as_int(payload.get('source_manifest_missing_file_count')) != 0:
        failures.append(f"source_manifest_missing_file_count={payload.get('source_manifest_missing_file_count')!r}")
    if _as_int(payload.get('source_manifest_hash_mismatch_count')) != 0:
        failures.append(f"source_manifest_hash_mismatch_count={payload.get('source_manifest_hash_mismatch_count')!r}")
    if _as_int(payload.get('source_critical_file_count')) != _as_int(payload.get('source_critical_files_verified')):
        failures.append(f"source_critical_files_verified={payload.get('source_critical_files_verified')!r} source_critical_file_count={payload.get('source_critical_file_count')!r}")
    if _as_int(payload.get('source_required_critical_file_count')) != len(required_critical_files):
        failures.append(f"source_required_critical_file_count={payload.get('source_required_critical_file_count')!r}")
    if _as_int(payload.get('source_required_critical_missing_count')) != 0:
        failures.append(f"source_required_critical_missing_count={payload.get('source_required_critical_missing_count')!r} files={payload.get('source_required_critical_missing_files')!r}")
    if payload.get('source_git_commit') != (git_info.get('commit') or ''):
        failures.append(f"source_git_commit={payload.get('source_git_commit')!r}")
    if payload.get('source_git_dirty') != str(bool(git_info.get('dirty'))).lower():
        failures.append(f"source_git_dirty={payload.get('source_git_dirty')!r}")
    if payload.get('source_dirty_status_sha256') != (git_info.get('dirty_status_sha256') or 'clean'):
        failures.append(f"source_dirty_status_sha256={payload.get('source_dirty_status_sha256')!r}")
    if payload.get('source_generated_at_utc') != (source_provenance.get('generated_at_utc') or ''):
        failures.append(f"source_generated_at_utc={payload.get('source_generated_at_utc')!r}")
    if payload.get('mopac_executable_sha256') != hashlib.sha256(Path(mopac).read_bytes()).hexdigest():
        failures.append(f"mopac_executable_sha256={payload.get('mopac_executable_sha256')!r}")
    if payload.get('cmake_gpu_bool') != 'ON':
        failures.append(f"cmake_gpu_bool={payload.get('cmake_gpu_bool')!r}")
    if not str(payload.get('cmake_cuda_architectures') or '').strip():
        failures.append(f"cmake_cuda_architectures={payload.get('cmake_cuda_architectures')!r}")
    if reasons:
        failures.append('reasons=' + '; '.join(str(reason) for reason in reasons[:5]))
    strict_readiness_fatal_status_lines = []
    readiness_log_path = Path(str(row.get('log_path') or ''))
    if not readiness_log_path.is_absolute():
        readiness_log_path = path.parent / readiness_log_path
    readiness_text_parts = []
    if readiness_log_path.exists():
        readiness_text_parts.append(readiness_log_path.read_text(encoding='utf-8', errors='ignore'))
        for output_name in str(row.get('output_files') or '').split(';'):
            output_name = output_name.strip()
            if not output_name:
                continue
            output_path = Path(output_name)
            if not output_path.is_absolute():
                output_path = path.parent / output_path
            if output_path.exists():
                readiness_text_parts.append(output_path.read_text(encoding='utf-8', errors='ignore'))
        readiness_log_text = '\n'.join(readiness_text_parts)
        scf_fatal_re = re.compile(r'\[MOZYME GPU SCF\].*\bstatus=(?:fallback_cpu|strict_abort|resident_step)\b', re.IGNORECASE)
        helper_fatal_re = re.compile(r'\[MOZYME GPU (?:(?:makvec|relocal|reorth|setupk|density(?:_batch)?|cnvgz|helecz|eimp|diagg1_(?:aocc|avir|construct)|diagg2_(?:rotprep|rotate)|isitsc|tidy)\][^\n]*(?:\bstatus\s*=\s*)?(?:fallback_cpu|strict_abort)\b|fock[12]\]\s+fallback\b|fock1_batch\]\s+fallback\b|fock2_4x1_batch\]\s+fallback\b|resident_fock\]\s+fallback_real_pairs\s+total\s*=\s*[1-9]\d*\b|resident_fock\][^\n]*\bcpu_point_pairs\s*=\s*[1-9]\d*\b)', re.IGNORECASE)
        strict_readiness_fatal_status_lines = [
            line for line in readiness_log_text.splitlines()
            if scf_fatal_re.search(line) or helper_fatal_re.search(line)
        ]
    else:
        failures.append(f'strict_readiness_log_missing={row.get("log_path")!r}')
    if strict_readiness_fatal_status_lines:
        failures.append('strict_readiness_fatal_status_lines=' + repr(strict_readiness_fatal_status_lines[-5:]))
    if _as_int(row.get('returncode')) != 0:
        failures.append(f"returncode={row.get('returncode')!r}")
    if row.get('normal_end') is not True:
        failures.append(f"normal_end={row.get('normal_end')!r}")
    if _as_int(row.get('gpu_error_marker_count')) != 0:
        failures.append(f"gpu_error_marker_count={row.get('gpu_error_marker_count')!r}")
    if row.get('gpu_has_device') != 'T':
        failures.append(f"gpu_has_device={row.get('gpu_has_device')!r}")
    if row.get('gpu_lgpu_final') != 'T':
        failures.append(f"gpu_lgpu_final={row.get('gpu_lgpu_final')!r}")
    if row.get('mozyme_plan_resident_fock_gpu') != 'T':
        failures.append(f"mozyme_plan_resident_fock_gpu={row.get('mozyme_plan_resident_fock_gpu')!r}")
    for key in [
        'full_scf_gpu_requested',
        'full_scf_gpu_executed',
        'mozyme_scf_experimental_executed',
    ]:
        if _as_int(row.get(key)) != 1:
            failures.append(f"{key}={row.get(key)!r}")
    contract_violations = str(row.get('full_scf_gpu_contract_violations') or '')
    if contract_violations:
        failures.append('full_scf_gpu_contract_violations=' + contract_violations)
    if row.get('full_scf_probe_decision') != 'complete':
        failures.append(f"full_scf_probe_decision={row.get('full_scf_probe_decision')!r}")
    section_rows = row.get('mozyme_section_times')
    if not isinstance(section_rows, list) or not section_rows:
        failures.append('mozyme_section_times=missing')
    if row.get('full_scf_gpu_status') != 'complete':
        failures.append(f"full_scf_gpu_status={row.get('full_scf_gpu_status')!r}")
    if _as_int(row.get('full_scf_gpu_code')) != 0:
        failures.append(f"full_scf_gpu_code={row.get('full_scf_gpu_code')!r}")
    for key in [
        'full_scf_gpu_ready',
        'full_scf_gpu_backend_ready',
        'full_scf_gpu_resident',
        'full_scf_gpu_isitsc_okscf',
        'full_scf_gpu_final_density_resident',
    ]:
        if _as_int(row.get(key)) != 1:
            failures.append(f"{key}={row.get(key)!r}")
    if _as_int(row.get('full_scf_gpu_olden_setup_only')) != 0:
        failures.append(f"full_scf_gpu_olden_setup_only={row.get('full_scf_gpu_olden_setup_only')!r}")
    if (_as_int(row.get('full_scf_gpu_fillij_gpu_count_calls')) or 0) <= 0:
        failures.append(f"full_scf_gpu_fillij_gpu_count_calls={row.get('full_scf_gpu_fillij_gpu_count_calls')!r}")
    if (_as_int(row.get('full_scf_gpu_fillij_gpu_fill_calls')) or 0) <= 0:
        failures.append(f"full_scf_gpu_fillij_gpu_fill_calls={row.get('full_scf_gpu_fillij_gpu_fill_calls')!r}")
    if (_as_int(row.get('full_scf_gpu_resident_fock_gpu_count_calls')) or 0) <= 0:
        failures.append(f"full_scf_gpu_resident_fock_gpu_count_calls={row.get('full_scf_gpu_resident_fock_gpu_count_calls')!r}")
    resident_pack_calls = _as_int(row.get('full_scf_gpu_resident_fock_gpu_pack_calls')) or 0
    if resident_pack_calls <= 0:
        failures.append(f"full_scf_gpu_resident_fock_gpu_pack_calls={row.get('full_scf_gpu_resident_fock_gpu_pack_calls')!r}")
    resident_point_count_keys = ('full_scf_gpu_resident_fock_gpu_count_point', 'full_scf_gpu_resident_fock_gpu_pack_point', 'full_scf_gpu_cpu_resident_fock_plan_setup_point', 'mozyme_fock_plan_point_charge_pairs', 'mozyme_sparse_fock_setup_point_tasks', 'mozyme_resident_fock_point_pairs', 'mozyme_resident_fock_gpu_point_pairs')
    resident_point_counts = [value for value in (_as_int(row.get(key)) for key in resident_point_count_keys) if value is not None and value > 0]
    resident_point_count = max(resident_point_counts, default=0)
    resident_point_run_tasks = _as_int(row.get('mozyme_sparse_fock_run_point_tasks'))
    resident_point_work_present = resident_point_count > 0 or (resident_point_run_tasks is not None and resident_point_run_tasks > 0)
    point_weight_calls = _as_int(row.get('full_scf_gpu_resident_fock_gpu_point_weight_calls'))
    point_weight_point = _as_int(row.get('full_scf_gpu_resident_fock_gpu_point_weight_point'))
    if resident_point_work_present:
        if point_weight_calls is None or point_weight_calls <= 0:
            failures.append(f"full_scf_gpu_resident_fock_gpu_point_weight_calls={row.get('full_scf_gpu_resident_fock_gpu_point_weight_calls')!r}")
        if point_weight_point is None or point_weight_point <= 0:
            failures.append(f"full_scf_gpu_resident_fock_gpu_point_weight_point={row.get('full_scf_gpu_resident_fock_gpu_point_weight_point')!r}")
        elif resident_point_count > 0 and point_weight_point < resident_point_count:
            failures.append(f"full_scf_gpu_resident_fock_gpu_point_weight_point={point_weight_point}/{resident_point_count}")
    if _as_int(row.get('full_scf_gpu_cpu_mozyme_setup_only_calls')) != 0:
        failures.append(f"full_scf_gpu_cpu_mozyme_setup_only_calls={row.get('full_scf_gpu_cpu_mozyme_setup_only_calls')!r}")
    if _as_int(row.get('full_scf_gpu_cpu_resident_fock_plan_setup_calls')) != 0:
        failures.append(f"full_scf_gpu_cpu_resident_fock_plan_setup_calls={row.get('full_scf_gpu_cpu_resident_fock_plan_setup_calls')!r}")
    if _as_int(row.get('full_scf_gpu_cpu_pinout_calls')) != 0:
        failures.append(f"full_scf_gpu_cpu_pinout_calls={row.get('full_scf_gpu_cpu_pinout_calls')!r}")
    strict_host_route_count = _as_int(row.get('full_scf_gpu_strict_host_route_marker_count'))
    if strict_host_route_count != 0:
        failures.append(f"full_scf_gpu_strict_host_route_marker_count={row.get('full_scf_gpu_strict_host_route_marker_count')!r}")
    strict_host_route_markers = str(row.get('full_scf_gpu_strict_host_route_markers') or '')
    if strict_host_route_markers:
        failures.append('full_scf_gpu_strict_host_route_markers=' + strict_host_route_markers)
    output_files = str(row.get('output_files') or '')
    if any(item.lower().endswith('.den') for item in output_files.split(';') if item):
        failures.append('strict readiness produced .den output_files=' + output_files)
    host_commit_calls = _as_int(row.get('full_scf_gpu_host_commit_only_calls'))
    if host_commit_calls != 1:
        failures.append(f"full_scf_gpu_host_commit_only_calls={row.get('full_scf_gpu_host_commit_only_calls')!r}; expected exactly 1")
    if row.get('full_scf_gpu_host_commit_phase') != 'final_publication':
        failures.append(f"full_scf_gpu_host_commit_phase={row.get('full_scf_gpu_host_commit_phase')!r}")
    host_commit_arrays = _as_int(row.get('full_scf_gpu_host_commit_arrays'))
    host_commit_bytes = _as_int(row.get('full_scf_gpu_host_commit_bytes'))
    host_commit_cosmo = _as_int(row.get('full_scf_gpu_host_commit_cosmo'))
    final_publication_arrays = _as_int(row.get('full_scf_gpu_final_publication_arrays'))
    final_publication_bytes = _as_int(row.get('full_scf_gpu_final_publication_bytes'))
    final_publication_cosmo = _as_int(row.get('full_scf_gpu_final_publication_cosmo'))
    if (host_commit_arrays or 0) <= 0:
        failures.append(f"full_scf_gpu_host_commit_arrays={row.get('full_scf_gpu_host_commit_arrays')!r}")
    if (host_commit_bytes or 0) <= 0:
        failures.append(f"full_scf_gpu_host_commit_bytes={row.get('full_scf_gpu_host_commit_bytes')!r}")
    if host_commit_cosmo != 1:
        failures.append(f"full_scf_gpu_host_commit_cosmo={row.get('full_scf_gpu_host_commit_cosmo')!r}")
    if _as_int(row.get('full_scf_gpu_final_publication_done')) != 1:
        failures.append(f"full_scf_gpu_final_publication_done={row.get('full_scf_gpu_final_publication_done')!r}")
    if (final_publication_arrays or 0) <= 0:
        failures.append(f"full_scf_gpu_final_publication_arrays={row.get('full_scf_gpu_final_publication_arrays')!r}")
    if (final_publication_bytes or 0) <= 0:
        failures.append(f"full_scf_gpu_final_publication_bytes={row.get('full_scf_gpu_final_publication_bytes')!r}")
    if final_publication_cosmo != 1:
        failures.append(f"full_scf_gpu_final_publication_cosmo={row.get('full_scf_gpu_final_publication_cosmo')!r}")
    if host_commit_arrays != final_publication_arrays:
        failures.append(f"host/typed final publication arrays disagree: {host_commit_arrays!r} != {final_publication_arrays!r}")
    if host_commit_bytes != final_publication_bytes:
        failures.append(f"host/typed final publication bytes disagree: {host_commit_bytes!r} != {final_publication_bytes!r}")
    if host_commit_cosmo != final_publication_cosmo:
        failures.append(f"host/typed final publication cosmo flags disagree: {host_commit_cosmo!r} != {final_publication_cosmo!r}")
    if _as_int(row.get('full_scf_gpu_pls_restart_required')) != 0:
        failures.append(f"full_scf_gpu_pls_restart_required={row.get('full_scf_gpu_pls_restart_required')!r}")
    if _as_int(row.get('full_scf_gpu_pls_restart_required_calls')) != 0:
        failures.append(f"full_scf_gpu_pls_restart_required_calls={row.get('full_scf_gpu_pls_restart_required_calls')!r}")
    pls_reset_calls = _as_int(row.get('full_scf_gpu_pls_restart_reset_device_calls')) or 0
    if pls_reset_calls > 0 and _as_int(row.get('full_scf_gpu_pls_restart_done')) != 1:
        failures.append(f"full_scf_gpu_pls_restart_done={row.get('full_scf_gpu_pls_restart_done')!r}; reset_calls={pls_reset_calls}")
    required_stage_mask = 1023
    stage_missing = _as_int(row.get('full_scf_gpu_stage_missing'))
    stage_required = _as_int(row.get('full_scf_gpu_stage_required'))
    stage_completed = _as_int(row.get('full_scf_gpu_stage_completed'))
    if stage_missing != 0:
        failures.append(f"full_scf_gpu_stage_missing={row.get('full_scf_gpu_stage_missing')!r}")
    if stage_required != required_stage_mask:
        failures.append(f"full_scf_gpu_stage_required={row.get('full_scf_gpu_stage_required')!r}")
    if stage_completed != required_stage_mask:
        failures.append(f"full_scf_gpu_stage_completed={row.get('full_scf_gpu_stage_completed')!r}")
    if _as_int(row.get('full_scf_gpu_resident_decision')) != 1:
        failures.append(f"full_scf_gpu_resident_decision={row.get('full_scf_gpu_resident_decision')!r}")
    expected_stage_names = 'upload+eimp+diagg+density+fock+cnvgz+helecz+isitsc+addhb+check'
    if row.get('full_scf_gpu_stage_completed_names_raw') != expected_stage_names:
        failures.append(f"full_scf_gpu_stage_completed_names_raw={row.get('full_scf_gpu_stage_completed_names_raw')!r}")
    if row.get('full_scf_gpu_stage_missing_names_raw') != 'none':
        failures.append(f"full_scf_gpu_stage_missing_names_raw={row.get('full_scf_gpu_stage_missing_names_raw')!r}")
    for key in (
        'full_scf_gpu_strict_resident',
        'full_scf_gpu_no_fallback_required',
        'full_scf_gpu_full_stage_mask',
        'full_scf_gpu_resident_decision_complete',
        'full_scf_gpu_resident_fock_plan_full_coverage',
    ):
        if _as_int(row.get(key)) != 1:
            failures.append(f"{key}={row.get(key)!r}")
    for key in (
        'full_scf_gpu_strict_resident_host_syncs',
        'full_scf_gpu_strict_resident_control_polls',
    ):
        if _as_int(row.get(key)) != 0:
            failures.append(f"{key}={row.get(key)!r}")
    if _as_int(row.get('full_scf_gpu_resident_fock_plan_id')) is None:
        failures.append(f"full_scf_gpu_resident_fock_plan_id={row.get('full_scf_gpu_resident_fock_plan_id')!r}")
    required_plan_mask = _as_int(row.get('full_scf_gpu_resident_fock_plan_required_mask'))
    covered_plan_mask = _as_int(row.get('full_scf_gpu_resident_fock_plan_covered_mask'))
    if required_plan_mask is None or required_plan_mask == 0:
        failures.append(f"full_scf_gpu_resident_fock_plan_required_mask={row.get('full_scf_gpu_resident_fock_plan_required_mask')!r}")
    if covered_plan_mask is None:
        failures.append(f"full_scf_gpu_resident_fock_plan_covered_mask={row.get('full_scf_gpu_resident_fock_plan_covered_mask')!r}")
    if required_plan_mask is not None and covered_plan_mask is not None:
        if covered_plan_mask != required_plan_mask:
            failures.append(f"resident Fock plan mask must exactly match required={required_plan_mask} covered={covered_plan_mask}")
        if required_plan_mask & 2 and _as_int(row.get('full_scf_gpu_resident_fock_plan_partial_coverage')) != 1:
            failures.append(f"full_scf_gpu_resident_fock_plan_partial_coverage={row.get('full_scf_gpu_resident_fock_plan_partial_coverage')!r}")
    if stage_required is not None and stage_completed is not None and stage_missing is not None:
        expected_stage_missing = stage_required & ~stage_completed
        if stage_missing != expected_stage_missing:
            failures.append(f"stage mask mismatch: missing={stage_missing} expected={expected_stage_missing}")
        if stage_completed & ~stage_required:
            failures.append(f"full_scf_gpu_stage_completed has extra bits: {stage_completed & ~stage_required}")
    final_iterations = _as_int(row.get('full_scf_gpu_final_iterations'))
    if final_iterations is None or final_iterations < 1:
        failures.append(f"full_scf_gpu_final_iterations={row.get('full_scf_gpu_final_iterations')!r}")
    min_stage_calls = final_iterations or 1
    for stage_name in ('upload', 'eimp', 'diagg', 'density', 'fock', 'cnvgz', 'helecz', 'isitsc', 'addhb', 'check'):
        expected_calls = 1 if stage_name == 'upload' else min_stage_calls
        calls_key = f'full_scf_gpu_stage_{stage_name}_calls'
        ms_key = f'full_scf_gpu_stage_{stage_name}_ms'
        calls = _as_int(row.get(calls_key))
        ms = _as_float(row.get(ms_key))
        if calls is None or calls < expected_calls:
            failures.append(f"{calls_key}={row.get(calls_key)!r}; expected >= {expected_calls}")
        if ms is None or ms < 0.0:
            failures.append(f"{ms_key}={row.get(ms_key)!r}")
    cnvgz_active_calls = _as_int(row.get('full_scf_gpu_cnvgz_active_calls'))
    cnvgz_noop_calls = _as_int(row.get('full_scf_gpu_cnvgz_noop_calls'))
    if cnvgz_active_calls is None:
        failures.append(f"full_scf_gpu_cnvgz_active_calls={row.get('full_scf_gpu_cnvgz_active_calls')!r}")
    elif cnvgz_active_calls < 0:
        failures.append(f"full_scf_gpu_cnvgz_active_calls={cnvgz_active_calls!r}")
    if cnvgz_noop_calls is None:
        failures.append(f"full_scf_gpu_cnvgz_noop_calls={row.get('full_scf_gpu_cnvgz_noop_calls')!r}")
    elif cnvgz_noop_calls < 0:
        failures.append(f"full_scf_gpu_cnvgz_noop_calls={cnvgz_noop_calls!r}")
    if cnvgz_active_calls is not None and cnvgz_noop_calls is not None and not _cnvgz_active_noop_ready(cnvgz_active_calls, cnvgz_noop_calls):
        failures.append(f"full_scf_gpu_cnvgz_work=active:{cnvgz_active_calls},noop:{cnvgz_noop_calls}")
    if _as_int(row.get('full_scf_gpu_cpu_mutating_section_count')) != 0:
        failures.append(f"full_scf_gpu_cpu_mutating_section_count={row.get('full_scf_gpu_cpu_mutating_section_count')!r}")
    if _as_int(row.get('full_scf_gpu_cpu_mutating_call_count')) != 0:
        failures.append(f"full_scf_gpu_cpu_mutating_call_count={row.get('full_scf_gpu_cpu_mutating_call_count')!r}")
    if _as_float(row.get('full_scf_gpu_cpu_mutating_ms')) != 0.0:
        failures.append(f"full_scf_gpu_cpu_mutating_ms={row.get('full_scf_gpu_cpu_mutating_ms')!r}")
    device_id = _as_int(row.get('full_scf_gpu_device_id'))
    if device_id is None or device_id < 0:
        failures.append(f"full_scf_gpu_device_id={row.get('full_scf_gpu_device_id')!r}")
    makvec_success = _as_int(row.get('mozyme_makvec_gpu_success_calls')) or 0
    makvec_existing = _as_int(row.get('mozyme_makvec_gpu_existing_lmo_calls')) or 0
    if makvec_success <= 0:
        failures.append(f"mozyme_makvec_gpu_success_calls={row.get('mozyme_makvec_gpu_success_calls')!r}")
    if makvec_existing > 0:
        failures.append(f"mozyme_makvec_gpu_existing_lmo_calls={row.get('mozyme_makvec_gpu_existing_lmo_calls')!r}; OLD_SCF existing-LMO is not GPU proof")
    makvec_ms = _as_float(row.get('mozyme_makvec_gpu_last_ms'))
    if makvec_success > 0 and (makvec_ms is None or makvec_ms < 0.0):
        failures.append(f"mozyme_makvec_gpu_last_ms={row.get('mozyme_makvec_gpu_last_ms')!r}")
    relocal_success = _as_int(row.get('mozyme_relocal_gpu_success_calls'))
    if relocal_success is None or relocal_success <= 0:
        failures.append(f"mozyme_relocal_gpu_success_calls={row.get('mozyme_relocal_gpu_success_calls')!r}")
    relocal_occupied_success = _as_int(row.get('mozyme_relocal_gpu_occupied_success_calls'))
    if relocal_occupied_success is None or relocal_occupied_success <= 0:
        failures.append(f"mozyme_relocal_gpu_occupied_success_calls={row.get('mozyme_relocal_gpu_occupied_success_calls')!r}")
    relocal_virtual_success = _as_int(row.get('mozyme_relocal_gpu_virtual_success_calls'))
    if relocal_virtual_success is None or relocal_virtual_success <= 0:
        failures.append(f"mozyme_relocal_gpu_virtual_success_calls={row.get('mozyme_relocal_gpu_virtual_success_calls')!r}")
    reorth_success = _as_int(row.get('mozyme_reorth_gpu_success_calls'))
    if reorth_success is None or reorth_success <= 0:
        failures.append(f"mozyme_reorth_gpu_success_calls={row.get('mozyme_reorth_gpu_success_calls')!r}")
    reorth_resident_success = _as_int(row.get('mozyme_reorth_gpu_resident_success_calls'))
    if reorth_resident_success is None or reorth_resident_success <= 0:
        failures.append(f"mozyme_reorth_gpu_resident_success_calls={row.get('mozyme_reorth_gpu_resident_success_calls')!r}")
    setupk_fallback = _as_int(row.get('mozyme_setupk_gpu_fallback_calls'))
    if setupk_fallback is None or setupk_fallback != 0:
        failures.append(f"mozyme_setupk_gpu_fallback_calls={row.get('mozyme_setupk_gpu_fallback_calls')!r}")
    setupk_success = _as_int(row.get('mozyme_setupk_gpu_success_calls'))
    if setupk_success is None or setupk_success <= 0:
        failures.append(f"mozyme_setupk_gpu_success_calls={row.get('mozyme_setupk_gpu_success_calls')!r}")
    setupk_initial_success = _as_int(row.get('mozyme_setupk_gpu_initial_setup_success_calls'))
    if setupk_initial_success is None or setupk_initial_success <= 0:
        failures.append(f"mozyme_setupk_gpu_initial_setup_success_calls={row.get('mozyme_setupk_gpu_initial_setup_success_calls')!r}")
    setupk_initial_fallback = _as_int(row.get('mozyme_setupk_gpu_initial_setup_fallback_calls'))
    if setupk_initial_fallback is None or setupk_initial_fallback != 0:
        failures.append(f"mozyme_setupk_gpu_initial_setup_fallback_calls={row.get('mozyme_setupk_gpu_initial_setup_fallback_calls')!r}")
    setupk_initial_all_paths = _as_int(row.get('mozyme_setupk_gpu_initial_setup_all_paths_calls'))
    if setupk_initial_all_paths is None or setupk_initial_all_paths <= 0:
        failures.append(f"mozyme_setupk_gpu_initial_setup_all_paths_calls={row.get('mozyme_setupk_gpu_initial_setup_all_paths_calls')!r}")
    setupk_initial_ms = _as_float(row.get('mozyme_setupk_gpu_initial_setup_last_ms'))
    if setupk_initial_ms is None or setupk_initial_ms < 0.0:
        failures.append(f"mozyme_setupk_gpu_initial_setup_last_ms={row.get('mozyme_setupk_gpu_initial_setup_last_ms')!r}")
    scf_fallback_calls = _as_int(row.get('full_scf_gpu_scf_fallback_calls'))
    if scf_fallback_calls is None or scf_fallback_calls != 0:
        failures.append(f"full_scf_gpu_scf_fallback_calls={row.get('full_scf_gpu_scf_fallback_calls')!r}")
    scf_success_calls = _as_int(row.get('full_scf_gpu_scf_success_calls'))
    if scf_success_calls is None or scf_success_calls <= 0:
        failures.append(f"full_scf_gpu_scf_success_calls={row.get('full_scf_gpu_scf_success_calls')!r}")
    resident_step_calls = _as_int(row.get('full_scf_gpu_resident_step_calls'))
    if resident_step_calls is None or resident_step_calls != 0:
        failures.append(f"full_scf_gpu_resident_step_calls={row.get('full_scf_gpu_resident_step_calls')!r}")
    cpu_boundary_calls = _as_int(row.get('full_scf_gpu_cpu_boundary_calls'))
    if cpu_boundary_calls is None or cpu_boundary_calls != 0:
        failures.append(f"full_scf_gpu_cpu_boundary_calls={row.get('full_scf_gpu_cpu_boundary_calls')!r}")
    wall_ms = _as_float(row.get('full_scf_gpu_wall_ms'))
    if wall_ms is None or wall_ms < 0.0:
        failures.append(f"full_scf_gpu_wall_ms={row.get('full_scf_gpu_wall_ms')!r}")
    for key in ('full_scf_gpu_energy_total', 'full_scf_gpu_diagg_sumt', 'full_scf_gpu_diagg_sumb'):
        if _as_float(row.get(key)) is None:
            failures.append(f"{key}={row.get(key)!r}")
    sparse_calls = _as_int(row.get('mozyme_sparse_fock_run_calls'))
    if sparse_calls is None or final_iterations is None or sparse_calls < final_iterations:
        failures.append(f"mozyme_sparse_fock_run_calls={row.get('mozyme_sparse_fock_run_calls')!r}")
    sparse_ms = _as_float(row.get('mozyme_sparse_fock_run_ms'))
    if sparse_ms is None or sparse_ms < 0.0:
        failures.append(f"mozyme_sparse_fock_run_ms={row.get('mozyme_sparse_fock_run_ms')!r}")
    sparse_work_keys = (
        'mozyme_sparse_fock_run_one_tasks',
        'mozyme_sparse_fock_run_pair_tasks',
        'mozyme_sparse_fock_run_4x1_tasks',
        'mozyme_sparse_fock_run_point_tasks',
    )
    sparse_work_values = {key: _as_int(row.get(key)) for key in sparse_work_keys}
    if any(value is None for value in sparse_work_values.values()):
        failures.append('resident_sparse_fock_work_keys=' + repr(sparse_work_values))
    elif sum(sparse_work_values.values()) <= 0:
        failures.append('resident_sparse_fock_work=0')
    if _as_int(row.get('mozyme_sparse_fock_run_zero_work_calls')) != 0:
        failures.append(f"mozyme_sparse_fock_run_zero_work_calls={row.get('mozyme_sparse_fock_run_zero_work_calls')!r}")
    planned_point_pairs_value = _as_int(row.get('mozyme_fock_plan_point_charge_pairs'))
    planned_point_dipole_value = _as_int(row.get('mozyme_fock_plan_point_dipole_pairs'))
    planned_point_monopole_value = _as_int(row.get('mozyme_fock_plan_point_monopole_pairs'))
    setup_point_tasks_value = _as_int(row.get('mozyme_sparse_fock_setup_point_tasks'))
    setup_point_dipole_value = _as_int(row.get('mozyme_sparse_fock_setup_point_dipole_tasks'))
    setup_point_monopole_value = _as_int(row.get('mozyme_sparse_fock_setup_point_monopole_tasks'))
    run_point_tasks_value = _as_int(row.get('mozyme_sparse_fock_run_point_tasks'))
    run_point_dipole_value = _as_int(row.get('mozyme_sparse_fock_run_point_dipole_tasks'))
    run_point_monopole_value = _as_int(row.get('mozyme_sparse_fock_run_point_monopole_tasks'))
    if planned_point_pairs_value is None:
        failures.append(f"mozyme_fock_plan_point_charge_pairs={row.get('mozyme_fock_plan_point_charge_pairs')!r}")
    if setup_point_tasks_value is None:
        failures.append(f"mozyme_sparse_fock_setup_point_tasks={row.get('mozyme_sparse_fock_setup_point_tasks')!r}")
    if run_point_tasks_value is None:
        failures.append(f"mozyme_sparse_fock_run_point_tasks={row.get('mozyme_sparse_fock_run_point_tasks')!r}")
    planned_point_pairs = planned_point_pairs_value or 0
    if planned_point_pairs_value is not None and planned_point_pairs > 0:
        if planned_point_dipole_value is None or planned_point_monopole_value is None:
            failures.append('mozyme_fock_plan_point_kind_pairs=missing')
        elif planned_point_dipole_value + planned_point_monopole_value != planned_point_pairs:
            failures.append(f"mozyme_fock_plan_point_kind_pairs={planned_point_dipole_value}+{planned_point_monopole_value}/{planned_point_pairs}")
        if setup_point_dipole_value is None or setup_point_monopole_value is None:
            failures.append('mozyme_sparse_fock_setup_point_kind_tasks=missing')
        elif setup_point_dipole_value + setup_point_monopole_value != (setup_point_tasks_value or 0):
            failures.append(f"mozyme_sparse_fock_setup_point_kind_tasks={setup_point_dipole_value}+{setup_point_monopole_value}/{setup_point_tasks_value}")
        if run_point_dipole_value is None or run_point_monopole_value is None:
            failures.append('mozyme_sparse_fock_run_point_kind_tasks=missing')
        elif run_point_dipole_value + run_point_monopole_value != (run_point_tasks_value or 0):
            failures.append(f"mozyme_sparse_fock_run_point_kind_tasks={run_point_dipole_value}+{run_point_monopole_value}/{run_point_tasks_value}")
        setup_point_tasks = setup_point_tasks_value or 0
        run_point_tasks = run_point_tasks_value or 0
        required_point_runs = planned_point_pairs * max(1, sparse_calls or 0)
        if setup_point_tasks < planned_point_pairs:
            failures.append(f"mozyme_sparse_fock_setup_point_tasks={setup_point_tasks}/{planned_point_pairs}")
        if run_point_tasks < required_point_runs:
            failures.append(f"mozyme_sparse_fock_run_point_tasks={run_point_tasks}/{required_point_runs}")
        if planned_point_dipole_value and (setup_point_dipole_value or 0) < planned_point_dipole_value:
            failures.append(f"mozyme_sparse_fock_setup_point_dipole_tasks={setup_point_dipole_value}/{planned_point_dipole_value}")
        if planned_point_monopole_value and (setup_point_monopole_value or 0) < planned_point_monopole_value:
            failures.append(f"mozyme_sparse_fock_setup_point_monopole_tasks={setup_point_monopole_value}/{planned_point_monopole_value}")
        if planned_point_dipole_value and (run_point_dipole_value or 0) < planned_point_dipole_value * max(1, sparse_calls or 0):
            failures.append(f"mozyme_sparse_fock_run_point_dipole_tasks={run_point_dipole_value}/{planned_point_dipole_value * max(1, sparse_calls or 0)}")
        if planned_point_monopole_value and (run_point_monopole_value or 0) < planned_point_monopole_value * max(1, sparse_calls or 0):
            failures.append(f"mozyme_sparse_fock_run_point_monopole_tasks={run_point_monopole_value}/{planned_point_monopole_value * max(1, sparse_calls or 0)}")
    resident_real_pairs = _as_int(row.get('mozyme_resident_fock_real_pairs'))
    resident_gpu_pairs = _as_int(row.get('mozyme_resident_fock_gpu_real_pairs'))
    resident_cpu_pairs = _as_int(row.get('mozyme_resident_fock_cpu_real_pairs'))
    inactive_pairs = _as_int(row.get('mozyme_resident_fock_inactive_real_pairs'))
    if any(value is None for value in (resident_real_pairs, resident_gpu_pairs, resident_cpu_pairs, inactive_pairs)):
        failures.append('resident_real_pair_coverage=missing')
    else:
        if resident_real_pairs <= 0 or resident_gpu_pairs != resident_real_pairs:
            failures.append(f"mozyme_resident_fock_gpu_real_pairs={resident_gpu_pairs}/{resident_real_pairs}")
        if resident_cpu_pairs != 0:
            failures.append(f"mozyme_resident_fock_cpu_real_pairs={resident_cpu_pairs}")
        if resident_real_pairs != resident_gpu_pairs + resident_cpu_pairs:
            failures.append(f"resident_real_pair_coverage_sum={resident_real_pairs}/{resident_gpu_pairs}+{resident_cpu_pairs}")
    resident_fock_fallback_keys = (
        'mozyme_resident_fock_cpu_real_pairs',
        'mozyme_resident_fock_fallback_pairs',
        'mozyme_resident_fock_basis_limit_fallback_pairs',
        'mozyme_resident_fock_direct_basis_fallback_pairs',
        'mozyme_resident_fock_other_fallback_pairs',
        'mozyme_resident_fock_direct_basis_point_fallback_pairs',
    )
    stage_fallback_keys = (
        'full_scf_gpu_fallback',
        'full_scf_gpu_scf_fallback_calls',
        'full_scf_gpu_resident_step_calls',
        'full_scf_gpu_cpu_boundary_calls',
        'full_scf_gpu_pls_restart_required',
        'full_scf_gpu_pls_restart_required_calls',
        'full_scf_gpu_strict_host_route_marker_count',
        'mozyme_gpu_helper_fatal_marker_count',
        'mozyme_makvec_gpu_fallback_calls',
        'mozyme_relocal_gpu_fallback_calls',
        'mozyme_reorth_gpu_fallback_calls',
        'mozyme_setupk_gpu_fallback_calls',
        'mozyme_setupk_gpu_initial_setup_fallback_calls',
        'density_cpu_diag_blocks',
        'density_cpu_offdiag_blocks',
        'density_batch_gpu_fallback_calls',
        'mozyme_eimp_gpu_fallback_calls',
        'mozyme_diagg1_aocc_gpu_fallback_calls',
        'mozyme_diagg1_avir_gpu_fallback_calls',
        'mozyme_diagg1_construct_gpu_fallback_calls',
        'mozyme_diagg2_rotprep_gpu_fallback_calls',
        'mozyme_diagg2_rotate_gpu_fallback_calls',
        'mozyme_isitsc_gpu_fallback_calls',
        'mozyme_cnvgz_gpu_fallback_calls',
        'mozyme_helecz_gpu_fallback_calls',
        'mozyme_fock1_batch_gpu_fallback_calls',
        'mozyme_fock1_batch_gpu_fallback_tasks',
        'mozyme_fock1_batch_gpu_fallback_pairs',
        'mozyme_fock2_4x1_batch_gpu_fallback_calls',
        'mozyme_fock2_4x1_batch_gpu_fallback_tasks',
        'mozyme_fock1_gpu_fallback_seen',
        'mozyme_fock2_gpu_fallback_seen',
    )
    for key in resident_fock_fallback_keys + stage_fallback_keys:
        if _as_int(row.get(key)) != 0:
            failures.append(f"{key}={row.get(key)!r}")
    if failures:
        raise RuntimeError('Strict full-SCF readiness artifact did not prove readiness: ' + '; '.join(failures))
    print('Strict full-SCF readiness artifact verified:', path)
    return payload


def assert_full_scf_readiness_cpu_compare_artifact(path, readiness_payload):
    path = Path(path)
    if not path.exists():
        raise RuntimeError(f'Strict full-SCF readiness CPU comparison artifact was not written: {path}')
    payload = json.loads(path.read_text(encoding='utf-8'))
    failures = []
    if payload.get('contract_version') != 'resident-scf-strict-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-resident-cg-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v58':
        failures.append(f"contract_version={payload.get('contract_version')!r}")
    if payload.get('feature_set') != 'mozyme-full-scf-gpu-makvec-relocal-final-resident-reorth-tidy-selmos-pls-reset-cosmo-direct-point-kind-cnvgz-active-or-noop-cpu-compare-explicit-proof-v38-20260702':
        failures.append(f"feature_set={payload.get('feature_set')!r}")
    if payload.get('benchmark_scope') != 'molecule_mozyme_full_scf_gpu':
        failures.append(f"benchmark_scope={payload.get('benchmark_scope')!r}")
    if payload.get('source_zip_sha256') != source_zip_sha256:
        failures.append(f"source_zip_sha256={payload.get('source_zip_sha256')!r}")
    if payload.get('source_manifest_sha256') != manifest_sha256:
        failures.append(f"source_manifest_sha256={payload.get('source_manifest_sha256')!r}")
    if payload.get('source_provenance_marker_contract_version') != 'mopac-colab-source-markers-explicit-proof-v83':
        failures.append(f"source_provenance_marker_contract_version={payload.get('source_provenance_marker_contract_version')!r}")
    if payload.get('source_provenance_marker_contract_sha256') != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256:
        failures.append(f"source_provenance_marker_contract_sha256={payload.get('source_provenance_marker_contract_sha256')!r}")
    if payload.get('source_features_marker_contract_version') != 'mopac-colab-source-markers-explicit-proof-v83':
        failures.append(f"source_features_marker_contract_version={payload.get('source_features_marker_contract_version')!r}")
    if payload.get('source_features_marker_contract_sha256') != EXPECTED_MOPAC_SOURCE_MARKER_CONTRACT_SHA256:
        failures.append(f"source_features_marker_contract_sha256={payload.get('source_features_marker_contract_sha256')!r}")
    if payload.get('comparison_kind') != 'strict_full_scf_readiness_cpu_companion':
        failures.append(f"comparison_kind={payload.get('comparison_kind')!r}")
    if payload.get('status') != 'PASS':
        failures.append(f"status={payload.get('status')!r}")
    if payload.get('energy_status') != 'PASS':
        failures.append(f"energy_status={payload.get('energy_status')!r}")
    else:
        accuracy_bases = {part for part in str(payload.get('accuracy_basis') or '').split('+') if part}
        if not (accuracy_bases & {'absolute', 'relative'}):
            failures.append(f"accuracy_basis={payload.get('accuracy_basis')!r}; strict readiness cannot pass on per-atom tolerance alone")
    if not str(payload.get('accuracy_basis') or '').strip():
        failures.append(f"accuracy_basis={payload.get('accuracy_basis')!r}")
    if payload.get('reasons'):
        failures.append('reasons=' + '; '.join(str(reason) for reason in payload.get('reasons')[:5]))
    cpu_only_reasons = payload.get('cpu_only_reasons') if isinstance(payload, dict) else None
    if not isinstance(cpu_only_reasons, list):
        failures.append(f"cpu_only_reasons={cpu_only_reasons!r}")
    elif cpu_only_reasons:
        failures.append('cpu_only_reasons=' + '; '.join(str(reason) for reason in cpu_only_reasons[:5]))
    cpu_row = payload.get('cpu_row') if isinstance(payload, dict) else None
    gpu_row = payload.get('gpu_row') if isinstance(payload, dict) else None
    readiness_row = readiness_payload.get('row') if isinstance(readiness_payload, dict) else None
    if isinstance(readiness_payload, dict) and payload.get('input') != readiness_payload.get('input'):
        failures.append(f"input mismatch: compare={payload.get('input')!r} readiness={readiness_payload.get('input')!r}")
    if not isinstance(cpu_row, dict):
        failures.append('cpu_row=missing')
    else:
        if cpu_row.get('mode') != 'CPU':
            failures.append(f"cpu_row.mode={cpu_row.get('mode')!r}")
        if _as_int(cpu_row.get('returncode')) != 0:
            failures.append(f"cpu_row.returncode={cpu_row.get('returncode')!r}")
        if cpu_row.get('normal_end') is not True:
            failures.append(f"cpu_row.normal_end={cpu_row.get('normal_end')!r}")
        if _as_float(cpu_row.get('heat_kcal_mol')) is None:
            failures.append(f"cpu_row.heat_kcal_mol={cpu_row.get('heat_kcal_mol')!r}")
        if _as_int(cpu_row.get('gpu_error_marker_count')) != 0:
            failures.append(f"cpu_row.gpu_error_marker_count={cpu_row.get('gpu_error_marker_count')!r}")
        if cpu_row.get('gpu_lgpu_final') == 'T':
            failures.append('cpu_row.gpu_lgpu_final=T')
        if cpu_row.get('mozyme_gpu_profile') == 'T' or cpu_row.get('mozyme_gpu_after_plan') == 'T':
            failures.append('cpu_row.mozyme_gpu_active')
        if cpu_row.get('mozyme_plan_enabled') == 'T':
            failures.append('cpu_row.mozyme_plan_enabled=T')
        cpu_gpu_zero_keys = (
            'full_scf_gpu_requested', 'full_scf_gpu_executed',
            'mozyme_scf_experimental_executed', 'full_scf_gpu_scf_success_calls',
            'full_scf_gpu_resident_step_calls', 'full_scf_gpu_cpu_boundary_calls',
            'mozyme_sparse_fock_setup_calls', 'mozyme_sparse_fock_run_calls',
            'mozyme_fock1_batch_gpu_success_calls', 'mozyme_fock2_4x1_batch_gpu_success_calls',
            'density_batch_gpu_success_calls', 'mozyme_makvec_gpu_success_calls',
            'mozyme_relocal_gpu_success_calls', 'mozyme_reorth_gpu_success_calls',
            'mozyme_cnvgz_gpu_success_calls', 'mozyme_helecz_gpu_success_calls',
            'mozyme_eimp_gpu_success_calls', 'mozyme_isitsc_gpu_success_calls',
            'mozyme_diagg1_construct_gpu_success_calls', 'mozyme_diagg1_aocc_gpu_success_calls',
            'mozyme_diagg1_avir_gpu_success_calls', 'mozyme_diagg2_rotate_gpu_success_calls',
            'mozyme_diagg2_rotprep_gpu_success_calls',
            'full_scf_gpu_scf_fallback_calls', 'mozyme_makvec_gpu_fallback_calls',
            'mozyme_relocal_gpu_fallback_calls', 'mozyme_reorth_gpu_fallback_calls',
            'mozyme_setupk_gpu_fallback_calls', 'mozyme_setupk_gpu_initial_setup_fallback_calls',
            'density_batch_gpu_fallback_calls', 'mozyme_eimp_gpu_fallback_calls',
            'mozyme_diagg1_aocc_gpu_fallback_calls', 'mozyme_diagg1_avir_gpu_fallback_calls',
            'mozyme_diagg1_construct_gpu_fallback_calls', 'mozyme_diagg2_rotprep_gpu_fallback_calls',
            'mozyme_diagg2_rotate_gpu_fallback_calls', 'mozyme_isitsc_gpu_fallback_calls',
            'mozyme_cnvgz_gpu_fallback_calls', 'mozyme_helecz_gpu_fallback_calls',
            'mozyme_fock1_batch_gpu_fallback_calls', 'mozyme_fock2_4x1_batch_gpu_fallback_calls',
            'mozyme_resident_fock_cpu_real_pairs', 'mozyme_resident_fock_cpu_point_pairs',
        )
        for key in cpu_gpu_zero_keys:
            value = _as_int(cpu_row.get(key)) or 0
            if value != 0:
                failures.append(f"cpu_row.{key}={cpu_row.get(key)!r}")
    if not isinstance(gpu_row, dict):
        failures.append('gpu_row=missing')
    else:
        if gpu_row.get('mode') != 'GPU':
            failures.append(f"gpu_row.mode={gpu_row.get('mode')!r}")
        if gpu_row.get('full_scf_gpu_status') != 'complete':
            failures.append(f"gpu_row.full_scf_gpu_status={gpu_row.get('full_scf_gpu_status')!r}")
        if _as_float(gpu_row.get('heat_kcal_mol')) is None:
            failures.append(f"gpu_row.heat_kcal_mol={gpu_row.get('heat_kcal_mol')!r}")
    if isinstance(gpu_row, dict) and isinstance(readiness_row, dict):
        for key in ('log_path', 'heat_kcal_mol', 'full_scf_gpu_resident_decision'):
            if str(gpu_row.get(key)) != str(readiness_row.get(key)):
                failures.append(f"gpu/readiness {key} mismatch: {gpu_row.get(key)!r} vs {readiness_row.get(key)!r}")
    binding = payload.get('readiness_binding') if isinstance(payload, dict) else None
    binding_keys = (
        'molecule', 'input', 'full_scf_gpu_status', 'full_scf_gpu_code',
        'full_scf_gpu_backend_ready', 'full_scf_gpu_resident',
        'full_scf_gpu_scf_success_calls', 'full_scf_gpu_final_iterations',
        'full_scf_gpu_stage_completed', 'full_scf_gpu_stage_required',
        'full_scf_gpu_stage_missing', 'full_scf_gpu_stage_completed_names_raw',
        'full_scf_gpu_stage_missing_names_raw', 'full_scf_gpu_strict_resident',
        'full_scf_gpu_no_fallback_required', 'full_scf_gpu_full_stage_mask',
        'full_scf_gpu_resident_decision_complete',
        'full_scf_gpu_strict_resident_host_syncs',
        'full_scf_gpu_strict_resident_control_polls',
        'full_scf_gpu_resident_fock_plan_id',
        'full_scf_gpu_resident_fock_plan_full_coverage',
        'full_scf_gpu_resident_fock_plan_partial_coverage',
        'full_scf_gpu_resident_fock_plan_required_mask',
        'full_scf_gpu_resident_fock_plan_covered_mask',
        'full_scf_gpu_resident_decision',
        'full_scf_gpu_final_density_resident', 'full_scf_gpu_host_commit_only_calls',
        'full_scf_gpu_final_publication_done', 'full_scf_gpu_final_publication_arrays',
        'full_scf_gpu_final_publication_bytes', 'full_scf_gpu_final_publication_cosmo',
        'full_scf_gpu_fillij_gpu_count_calls', 'full_scf_gpu_fillij_gpu_fill_calls',
        'full_scf_gpu_resident_fock_gpu_count_calls',
        'full_scf_gpu_resident_fock_gpu_count_plan_id',
        'full_scf_gpu_resident_fock_gpu_count_one',
        'full_scf_gpu_resident_fock_gpu_count_pair',
        'full_scf_gpu_resident_fock_gpu_count_pair4x1',
        'full_scf_gpu_resident_fock_gpu_count_point',
        'full_scf_gpu_resident_fock_gpu_count_full_coverage',
        'full_scf_gpu_resident_fock_gpu_pack_calls',
        'full_scf_gpu_resident_fock_gpu_pack_plan_id',
        'full_scf_gpu_resident_fock_gpu_pack_one',
        'full_scf_gpu_resident_fock_gpu_pack_pair',
        'full_scf_gpu_resident_fock_gpu_pack_pair4x1',
        'full_scf_gpu_resident_fock_gpu_pack_point',
        'full_scf_gpu_resident_fock_gpu_pack_full_coverage',
        'full_scf_gpu_resident_fock_gpu_point_weight_calls',
        'full_scf_gpu_resident_fock_gpu_point_weight_point',
        'full_scf_gpu_resident_fock_gpu_point_weight_max_abs_diff',
        'full_scf_gpu_cnvgz_active_calls', 'full_scf_gpu_cnvgz_noop_calls',
        'full_scf_gpu_cpu_mozyme_setup_only_calls',
        'full_scf_gpu_cpu_resident_fock_plan_setup_calls',
        'mozyme_plan_direct_mode', 'mozyme_fock_resident_full_coverage_planned',
        'mozyme_fock_resident_executable_tasks',
        'mozyme_fock_resident_direct_unsupported_pairs',
        'mozyme_fock_resident_direct_unsupported_point_pairs',
        'mozyme_resident_fock_direct_basis_fallback_pairs',
        'mozyme_resident_fock_direct_basis_point_fallback_pairs',
        'mopac_executable_sha256', 'cmake_gpu_bool', 'cmake_cuda_architectures',
        'heat_kcal_mol', 'source_zip_sha256', 'source_manifest_sha256',
        'source_provenance_marker_contract_version', 'source_features_marker_contract_version',
        'source_provenance_marker_contract_sha256', 'source_features_marker_contract_sha256',
    )
    if not isinstance(binding, dict):
        failures.append('readiness_binding=missing')
    elif isinstance(gpu_row, dict) and isinstance(readiness_row, dict):
        for key in binding_keys:
            if key not in binding:
                failures.append(f'readiness_binding missing {key}')
                continue
            if str(binding.get(key)) != str(readiness_row.get(key)):
                failures.append(f"binding/readiness {key} mismatch: {binding.get(key)!r} vs {readiness_row.get(key)!r}")
            if str(binding.get(key)) != str(gpu_row.get(key)):
                failures.append(f"binding/gpu_row {key} mismatch: {binding.get(key)!r} vs {gpu_row.get(key)!r}")
    if _as_float(payload.get('abs_heat_diff_kcal_mol')) is None:
        failures.append(f"abs_heat_diff_kcal_mol={payload.get('abs_heat_diff_kcal_mol')!r}")
    if _as_float(payload.get('rel_heat_diff')) is None:
        failures.append(f"rel_heat_diff={payload.get('rel_heat_diff')!r}")
    if _as_float(payload.get('abs_heat_diff_per_atom_kcal_mol')) is None:
        failures.append(f"abs_heat_diff_per_atom_kcal_mol={payload.get('abs_heat_diff_per_atom_kcal_mol')!r}")
    if failures:
        raise RuntimeError('Strict full-SCF readiness CPU companion did not prove accuracy: ' + '; '.join(failures))
    print('Strict full-SCF readiness CPU companion verified:', path)
    return payload


def assert_direct_cosmo_readiness_artifact(path):
    payload = assert_full_scf_readiness_artifact(path)
    row = payload.get('row') if isinstance(payload, dict) else None
    failures = []
    if not isinstance(row, dict):
        failures.append('row=missing')
    else:
        if Path(str(payload.get('input') or '')).name != 'direct_cosmo_peptide_gg.mop':
            failures.append(f"payload.input={payload.get('input')!r}")
        if Path(str(row.get('input') or '')).name != 'direct_cosmo_peptide_gg.mop':
            failures.append(f"row.input={row.get('input')!r}")
        if row.get('molecule') != 'direct_cosmo_peptide_gg':
            failures.append(f"molecule={row.get('molecule')!r}")
        eps = _as_float(row.get('input_eps'))
        if eps is None or abs(eps - 78.4) > 1.0e-9:
            failures.append(f"input_eps={row.get('input_eps')!r}")
        for key in (
            'full_scf_gpu_cosmo_enabled',
            'full_scf_gpu_cosmo_fock_calls',
            'full_scf_gpu_cosmo_matvec_calls',
            'full_scf_gpu_cosmo_cg_iterations',
            'full_scf_gpu_cosmo_nps',
            'full_scf_gpu_cosmo_lm61',
            'full_scf_gpu_cosmo_cg_control_resident',
            'full_scf_gpu_cosmo_cg_converged',
        ):
            value = _as_int(row.get(key))
            minimum = 1
            if value is None or value < minimum:
                failures.append(f"{key}={row.get(key)!r}")
        pair_count = _as_int(row.get('full_scf_gpu_cosmo_pair_count'))
        if pair_count is None or pair_count <= 0:
            failures.append(f"full_scf_gpu_cosmo_pair_count must be positive: {row.get('full_scf_gpu_cosmo_pair_count')!r}")
        planned_point_pairs = _as_int(row.get('mozyme_fock_plan_point_charge_pairs')) or 0
        if planned_point_pairs > 0:
            setup_point_tasks = _as_int(row.get('mozyme_sparse_fock_setup_point_tasks')) or 0
            run_point_tasks = _as_int(row.get('mozyme_sparse_fock_run_point_tasks')) or 0
            if setup_point_tasks <= 0 or run_point_tasks <= 0:
                failures.append(f"direct COSMO point work planned={planned_point_pairs} setup={setup_point_tasks} run={run_point_tasks}")
            point_weight_calls = _as_int(row.get('full_scf_gpu_resident_fock_gpu_point_weight_calls'))
            point_weight_point = _as_int(row.get('full_scf_gpu_resident_fock_gpu_point_weight_point'))
            if point_weight_calls is None or point_weight_calls <= 0:
                failures.append(f"direct COSMO point-weight calls={row.get('full_scf_gpu_resident_fock_gpu_point_weight_calls')!r}")
            if point_weight_point is None or point_weight_point < planned_point_pairs:
                failures.append(f"direct COSMO point-weight point={row.get('full_scf_gpu_resident_fock_gpu_point_weight_point')!r}/{planned_point_pairs}")
        for key in ('full_scf_gpu_cosmo_cg_breakdown', 'full_scf_gpu_cosmo_cg_host_syncs'):
            value = _as_int(row.get(key))
            if value != 0:
                failures.append(f"{key}={row.get(key)!r}")
        for key in (
            'full_scf_gpu_cosmo_solv_energy',
            'full_scf_gpu_cosmo_ediel',
            'full_scf_gpu_cosmo_last_residual',
            'full_scf_gpu_cosmo_cg_target_tol',
        ):
            value = _as_float(row.get(key))
            if value is None:
                failures.append(f"{key}={row.get(key)!r}")
            elif key == 'full_scf_gpu_cosmo_last_residual' and value < 0.0:
                failures.append(f"{key}={value!r}")
            elif key == 'full_scf_gpu_cosmo_cg_target_tol' and value <= 0.0:
                failures.append(f"{key}={value!r}")
        residual = _as_float(row.get('full_scf_gpu_cosmo_last_residual'))
        target_tol = _as_float(row.get('full_scf_gpu_cosmo_cg_target_tol'))
        if residual is not None and target_tol is not None and residual > target_tol:
            failures.append(f"full_scf_gpu_cosmo_last_residual={residual!r} > full_scf_gpu_cosmo_cg_target_tol={target_tol!r}")
    if failures:
        raise RuntimeError('Direct COSMO readiness artifact did not prove resident GPU COSMO execution: ' + '; '.join(failures))
    print('Direct COSMO resident readiness artifact verified:', path)
    return payload

if not require_full_scf_gpu and not allow_non_proof_run:
    raise RuntimeError('Proof toggles are disabled: keep require_full_scf_gpu=True, or set allow_non_proof_run=True for a non-proof debug run.')

if require_full_scf_gpu and not require_pinned_source_zip:
    raise RuntimeError('require_full_scf_gpu=True requires require_pinned_source_zip=True.')
if require_full_scf_gpu and not expected_source_zip_sha256:
    raise RuntimeError('require_full_scf_gpu=True requires an expected source zip SHA from the .zip.expected.json sidecar or manual pin.')

source_git_info = source_provenance.get('git') if isinstance(source_provenance, dict) else {}
source_zip_dirty = bool(source_git_info.get('dirty')) if isinstance(source_git_info, dict) else True
source_dirty_matches_expected_sidecar = bool(allow_expected_dirty_source_zip and expected_source_git_dirty is True and expected_source_dirty_status_sha256 and source_git_info.get('dirty_status_sha256') == expected_source_dirty_status_sha256)
if require_full_scf_gpu and not trusted_expected_source_zip_sha256:
    raise RuntimeError('Proof mode requires trusted_expected_source_zip_sha256 pasted from the local packager output; the uploaded sidecar alone is not a trusted freshness pin.')
if require_full_scf_gpu and source_zip_sha256 != trusted_expected_source_zip_sha256:
    raise RuntimeError('The uploaded source zip SHA-256 does not match trusted_expected_source_zip_sha256.')
if require_full_scf_gpu and trusted_expected_source_git_commit and str(source_git_info.get('commit', '')).lower() != trusted_expected_source_git_commit.lower():
    raise RuntimeError('The uploaded source git commit does not match trusted_expected_source_git_commit.')
if require_full_scf_gpu and expected_source_proof_eligible is not True:
    raise RuntimeError('The expected-source sidecar is not proof_eligible; reason=' + str(expected_source_proof_ineligible_reason or 'unknown'))
if require_full_scf_gpu and source_provenance.get('proof_eligible') is not True:
    raise RuntimeError('The uploaded source provenance is not proof_eligible; reason=' + str(source_provenance.get('proof_ineligible_reason') or 'unknown'))
if require_full_scf_gpu and source_zip_dirty:
    raise RuntimeError('Dirty source zip is proof-ineligible. Commit or stash changes, regenerate the zip without --allow-dirty, then rerun proof mode. Use require_full_scf_gpu=False with allow_non_proof_run=True plus a dirty-zip override only for development diagnostics.')
if source_zip_dirty and not (allow_non_proof_run and (allow_development_dirty_zip or source_dirty_matches_expected_sidecar)):
    raise RuntimeError('Dirty source zip is allowed only for a non-proof/dev run with allow_non_proof_run=True and either allow_development_dirty_zip=True or allow_expected_dirty_source_zip=True with a matching sidecar.')
if source_zip_dirty:
    print('NON-PROOF DEVELOPMENT: dirty source zip accepted only because proof mode is disabled.')
    print('  dirty_status_sha256:', source_git_info.get('dirty_status_sha256'))

# expected_source_* values are optional stale-upload guards; proof mode verifies the uploaded zip hash from provenance and readiness artifacts.

if require_full_scf_gpu and not run_full_scf_readiness_probe:
    raise RuntimeError('require_full_scf_gpu=True requires run_full_scf_readiness_probe=True.')
if require_full_scf_gpu and not run_direct_cosmo_readiness_probe:
    raise RuntimeError('require_full_scf_gpu=True requires run_direct_cosmo_readiness_probe=True for the v58 COSMO-direct point-kind CPU-compare proof contract.')

if run_molecule_benchmark or require_full_scf_gpu or run_full_scf_readiness_probe:
    run(
        ['python3', 'scripts/hydrogenate_publication_benchmark_inputs.py', mopac, '--timeout', str(hydrogenation_timeout)],
        cwd=source_dir,
        log_path=CONTENT_DIR / 'mopac_hydrogenation.log',
    )

if run_full_scf_readiness_probe or run_direct_cosmo_readiness_probe:
    direct_cosmo_input = CONTENT_DIR / 'direct_cosmo_peptide_gg.mop'
    direct_cosmo_pdb = CONTENT_DIR / 'peptide_gg.pdb'
    shutil.copy2(source_dir / 'examples/peptide_gg.pdb', direct_cosmo_pdb)
    direct_cosmo_input.write_text(
        'PM7 GEO_DAT="peptide_gg.pdb" EPS=78.4 1SCF MOZYME MOZYME_GPU MOZYME_MINBLK=1 PULAY SHIFT=-50 ITRY=80 NEWPDB PDB GEO-OK NOCOMMENTS\n'
        'Direct COSMO resident addfckz readiness probe\n'
        'peptide_gg.pdb\n',
        encoding='utf-8',
    )

if run_full_scf_readiness_probe:
    full_scf_probe_input = direct_cosmo_input
    readiness_cmd = [
        'python3', 'scripts/molecule_benchmark_report.py', mopac,
        '--profile', 'publication_large',
        '--out-dir', full_scf_readiness_report_dir,
        '--bundle-zip', full_scf_readiness_bundle_zip,
        '--preflight-timeout', '900',
        '--full-scf-probe-input', full_scf_probe_input,
        '--preflight-min-speedup', str(preflight_min_speedup),
        '--gpu-profile',
        '--mozyme-section-profile',
        '--require-full-scf-gpu',
        '--require-direct-cosmo-gpu',
        '--full-scf-readiness-only',
        '--full-scf-readiness-cpu-compare',
    ]
    readiness = run(
        readiness_cmd,
        cwd=source_dir,
        log_path=CONTENT_DIR / 'mopac_full_scf_readiness_probe.log',
        check=False,
    )
    readme = full_scf_readiness_report_dir / 'README.md'
    if readme.exists():
        display(Markdown(readme.read_text(encoding='utf-8')))
    if full_scf_readiness_bundle_zip.exists():
        files.download(str(full_scf_readiness_bundle_zip))
    if readiness.returncode != 0:
        print('Full-SCF readiness probe did not pass. Diagnostic bundle/logs were still generated when possible.')
        failure_path = full_scf_readiness_report_dir / 'full_scf_gpu_readiness_failure.txt'
        diagnostic_excerpt = ''
        if failure_path.exists():
            diagnostic_excerpt = failure_path.read_text(encoding='utf-8', errors='replace')
            print('\n===== full_scf_gpu_readiness_failure.txt =====')
            print(diagnostic_excerpt)
        readiness_json = full_scf_readiness_report_dir / 'full_scf_gpu_readiness.json'
        if readiness_json.exists():
            try:
                payload = json.loads(readiness_json.read_text(encoding='utf-8'))
                row = payload.get('row', {}) if isinstance(payload, dict) else {}
                keys = [
                    'full_scf_gpu_status',
                    'full_scf_gpu_reason',
                    'full_scf_gpu_contract_violations',
                    'full_scf_gpu_code',
                    'full_scf_gpu_stage_completed_names',
                    'full_scf_gpu_stage_missing_names',
                    'full_scf_gpu_resident_decision',
                    'gpu_error_marker_count',
                    'gpu_error_markers',
                ]
                print('\n===== readiness JSON summary =====')
                for key in keys:
                    print(f'{key}: {row.get(key, "")}')
            except Exception as exc:
                print('Could not parse full_scf_gpu_readiness.json:', repr(exc))
        probe_log = CONTENT_DIR / 'mopac_full_scf_readiness_probe.log'
        if probe_log.exists():
            print('\n===== readiness probe log tail =====')
            print('\n'.join(probe_log.read_text(encoding='utf-8', errors='replace').splitlines()[-120:]))
        if not diagnostic_excerpt:
            diagnostic_excerpt = '\n'.join(readiness.stdout.splitlines()[-180:])
        raise RuntimeError('Full-SCF readiness probe failed. Key diagnostics:\n' + diagnostic_excerpt[-8000:])
    readiness_payload = assert_full_scf_readiness_artifact(full_scf_readiness_report_dir / 'full_scf_gpu_readiness.json')
    direct_cosmo_payload = assert_direct_cosmo_readiness_artifact(full_scf_readiness_report_dir / 'full_scf_gpu_readiness.json')
    assert_full_scf_readiness_cpu_compare_artifact(full_scf_readiness_report_dir / 'full_scf_gpu_readiness_cpu_compare.json', readiness_payload)
    full_scf_readiness_passed = True
    direct_cosmo_readiness_passed = True
    print('Full-SCF readiness flag satisfied by strict readiness probe:', readiness_payload['row'].get('full_scf_gpu_status'))
    print('Direct COSMO resident readiness flag satisfied by strict readiness probe:', direct_cosmo_payload['row'].get('full_scf_gpu_status'))

if run_direct_cosmo_readiness_probe and direct_cosmo_readiness_passed:
    print('Direct COSMO resident readiness already satisfied by strict full-SCF readiness probe; skipping duplicate probe.')
elif run_direct_cosmo_readiness_probe:
    direct_cosmo_cmd = [
        'python3', 'scripts/molecule_benchmark_report.py', mopac,
        '--profile', 'quick',
        '--out-dir', direct_cosmo_readiness_report_dir,
        '--bundle-zip', direct_cosmo_readiness_bundle_zip,
        '--preflight-timeout', '900',
        '--full-scf-probe-input', direct_cosmo_input,
        '--preflight-min-speedup', str(preflight_min_speedup),
        '--gpu-profile',
        '--mozyme-section-profile',
        '--require-full-scf-gpu',
        '--require-direct-cosmo-gpu',
        '--full-scf-readiness-only',
        '--full-scf-readiness-cpu-compare',
    ]
    direct_cosmo = run(
        direct_cosmo_cmd,
        cwd=source_dir,
        log_path=CONTENT_DIR / 'mopac_direct_cosmo_readiness_probe.log',
        check=False,
    )
    readme = direct_cosmo_readiness_report_dir / 'README.md'
    if readme.exists():
        display(Markdown(readme.read_text(encoding='utf-8')))
    if direct_cosmo_readiness_bundle_zip.exists():
        files.download(str(direct_cosmo_readiness_bundle_zip))
    if direct_cosmo.returncode != 0:
        print('Direct COSMO resident readiness probe did not pass. Diagnostic bundle/logs were still generated when possible.')
        raise subprocess.CalledProcessError(direct_cosmo.returncode, direct_cosmo.args, output=direct_cosmo.stdout)
    direct_cosmo_payload = assert_direct_cosmo_readiness_artifact(direct_cosmo_readiness_report_dir / 'full_scf_gpu_readiness.json')
    assert_full_scf_readiness_cpu_compare_artifact(direct_cosmo_readiness_report_dir / 'full_scf_gpu_readiness_cpu_compare.json', direct_cosmo_payload)
    direct_cosmo_readiness_passed = True
    print('Direct COSMO resident readiness flag:', direct_cosmo_payload['row'].get('full_scf_gpu_status'))

run_deferred_low_level_benchmarks()

if not require_full_scf_gpu and not run_full_scf_readiness_probe and not run_resident_scf_smoke:
    print('Skipping experimental resident-SCF smoke; set run_resident_scf_smoke=True for diagnostics only.')

if not require_full_scf_gpu and not run_full_scf_readiness_probe and run_resident_scf_smoke:
    scf_probe_env = os.environ.copy()
    # Experimental resident-SCF boundary smoke test. This is diagnostic only;
    # strict proof is established exclusively by full_scf_gpu_readiness.json.
    scf_probe_env.update({
        'MOPAC_FORCEGPU': '1',
        'MOZYME_GPU_FORCE': '1',
        'MOPAC_GPU_PROFILE': '1',
        'MOPAC_MOZYME_SECTION_PROFILE': '1',
        # Keep the diagnostic smoke aligned with strict resident-SCF proof mode.
        # The smoke is still non-proof, but no GPU stage is intentionally disabled.
        'MOPAC_MOZYME_RESIDENT_FOCK_GPU': '1',
        'MOPAC_MOZYME_DENSITY_BATCH_GPU': '1',
        'MOPAC_MOZYME_CNVGZ_GPU': '1',
        'MOPAC_MOZYME_HELECZ_GPU': '1',
        'MOPAC_MOZYME_EIMP_GPU': '1',
        'MOPAC_MOZYME_DIAGG1_CONSTRUCT_GPU': '1',
        'MOPAC_MOZYME_DIAGG2_ROTATE_GPU': '1',
        'MOPAC_MOZYME_ISITSC_GPU': '1',
        'MOPAC_MOZYME_MAKVEC_GPU': '1',
        'MOPAC_MOZYME_RELOCAL_GPU': '1',
        'MOPAC_MOZYME_REORTH_GPU': '1',
        'MOPAC_MOZYME_SCF_EXPERIMENTAL': '1',
        'MOPAC_MOZYME_RESIDENT_SCF': '1',
        'MOPAC_MOZYME_SCF_STRICT_RESIDENT': '1',
        'MOPAC_MOZYME_FULL_SCF_GPU': '1',
        'MOPAC_MOZYME_GPU_STRICT': '1',
        'MOPAC_MOZYME_SCF_EARLY_PROBE': '0',
        'MOPAC_MOZYME_SCF_FORCE_FINAL_REORTH': '1',
        # Set MOPAC_MOZYME_SCF_EARLY_PROBE=1 only to test the controlled pre-setup fallback.
    })
    base_scf_probe_input = source_dir / 'benchmarks/publication_inputs/mop/protein_crambin_1crn.mop'
    scf_probe_input = CONTENT_DIR / 'resident_scf_smoke_crambin.mop'
    for pdb_name in ['protein_crambin_1crn.pdb', 'protein_crambin_1crn_hydrogenated.pdb']:
        src_pdb = base_scf_probe_input.parent / pdb_name
        if src_pdb.exists():
            shutil.copy2(src_pdb, CONTENT_DIR / pdb_name)
    scf_probe_lines = base_scf_probe_input.read_text(encoding='utf-8').splitlines()
    scf_probe_lines[0] = 'PM7 GEO_DAT="protein_crambin_1crn_hydrogenated.pdb" 1SCF MOZYME MOZYME_GPU RE-LOCAL=1 REORTH MOZYME_MINBLK=1 PULAY SHIFT=-50 ITRY=50 NEWPDB PDB GEO-OK NOCOMMENTS'
    scf_probe_input.write_text('\n'.join(scf_probe_lines) + '\n', encoding='utf-8')
    scf_probe_out = scf_probe_input.with_suffix('.out')
    scf_probe_arc = scf_probe_input.with_suffix('.arc')
    for old_path in [scf_probe_out, scf_probe_arc]:
        if old_path.exists():
            old_path.unlink()
    scf_probe = run(
        [mopac, scf_probe_input.name],
        cwd=scf_probe_input.parent,
        env=scf_probe_env,
        log_path=CONTENT_DIR / 'mopac_experimental_scf_probe.log',
        check=False,
    )
    scf_probe_out_text = scf_probe_out.read_text(encoding='utf-8', errors='ignore') if scf_probe_out.exists() else ''
    scf_probe_text = scf_probe.stdout + '\n' + scf_probe_out_text
    (CONTENT_DIR / 'mopac_experimental_scf_probe_combined.log').write_text(scf_probe_text, encoding='utf-8')

    def _marker_fields(line):
        return dict(re.findall(r'\b([A-Za-z_][A-Za-z0-9_]*)=([^\s]+)', line))

    def _success_marker_lines(marker, predicate):
        matches = []
        for line in scf_probe_text.splitlines():
            if marker not in line:
                continue
            fields = _marker_fields(line)
            if fields.get('status') == 'success' and predicate(fields):
                matches.append(line)
        return matches

    def _scf_success_lines():
        matches = []
        for line in scf_probe_text.splitlines():
            if '[MOZYME GPU SCF]' not in line:
                continue
            fields = _marker_fields(line)
            if fields.get('status') == 'success' and fields.get('code') == '0' and fields.get('ready') == '1' and fields.get('resident') == '1':
                matches.append(line)
        return matches

    scf_marker_missing = '[MOZYME GPU SCF]' not in scf_probe_text
    if scf_marker_missing:
        print('No [MOZYME GPU SCF] marker in stdout or .out. Last .out lines:')
        print('\n'.join(scf_probe_out_text.splitlines()[-80:]))
        raise RuntimeError('Experimental resident SCF boundary did not execute in the smoke test.')
    else:
        scf_marker_lines = [line for line in scf_probe_text.splitlines() if '[MOZYME GPU SCF]' in line]
        print('MOZYME GPU SCF marker lines:')
        print('\n'.join(scf_marker_lines[-20:]))
        scf_fatal_re = re.compile(r'\[MOZYME GPU SCF\].*\bstatus=(?:fallback_cpu|strict_abort|resident_step)\b', re.IGNORECASE)
        helper_fatal_re = re.compile(r'\[MOZYME GPU (?:(?:makvec|relocal|reorth|setupk|density(?:_batch)?|cnvgz|helecz|eimp|diagg1_(?:aocc|avir|construct)|diagg2_(?:rotprep|rotate)|isitsc|tidy)\][^\n]*(?:\bstatus\s*=\s*)?(?:fallback_cpu|strict_abort)\b|fock[12]\]\s+fallback\b|fock1_batch\]\s+fallback\b|fock2_4x1_batch\]\s+fallback\b|resident_fock\]\s+fallback_real_pairs\s+total\s*=\s*[1-9]\d*\b|resident_fock\][^\n]*\bcpu_point_pairs\s*=\s*[1-9]\d*\b)', re.IGNORECASE)
        fatal_status_lines = [
            line for line in scf_probe_text.splitlines()
            if scf_fatal_re.search(line) or helper_fatal_re.search(line)
        ]
        if fatal_status_lines:
            print('Fatal resident SCF status lines:')
            print('\n'.join(fatal_status_lines[-20:]))
            raise RuntimeError('Experimental resident SCF reported fallback_cpu, strict_abort, or resident_step in a fatal smoke check.')
        stage_match = re.search(
            r'\[MOZYME GPU SCF\]\s+stage_completed=\s*(\d+)\s+stage_required=\s*(\d+)\s+stage_missing=\s*(\d+)',
            scf_probe_text,
        )
        if not stage_match:
            raise RuntimeError('Experimental resident SCF boundary did not report full stage_completed/stage_required/stage_missing masks.')
        stage_completed, stage_required, stage_missing = map(int, stage_match.groups())
        print(
            'Experimental resident SCF stages:',
            f'completed={stage_completed}',
            f'required={stage_required}',
            f'missing={stage_missing}',
        )
        required_stage_mask = 1023
        if stage_required != required_stage_mask:
            raise RuntimeError(f'Experimental resident SCF required stage mask must be {required_stage_mask}, got {stage_required}.')
        if stage_completed != required_stage_mask:
            raise RuntimeError(f'Experimental resident SCF completed stage mask must be {required_stage_mask}, got {stage_completed}.')
        if stage_missing != 0:
            raise RuntimeError('Experimental resident SCF did not complete every required GPU stage.')
        device_match = re.search(
            r'\[MOZYME GPU SCF\]\s+device_id=\s*(-?\d+)\s+natoms=\s*\d+\s+norbs=\s*\d+\s+iterations=\s*(-?\d+)',
            scf_probe_text,
        )
        if not device_match:
            raise RuntimeError('Experimental resident SCF did not report CUDA device/iteration evidence.')
        device_id, final_iterations = map(int, device_match.groups())
        if device_id < 0 or final_iterations < 1:
            raise RuntimeError(f'Experimental resident SCF device/iteration proof failed: device_id={device_id}, iterations={final_iterations}.')
        cnvgz_activity_lines = [
            line for line in scf_probe_text.splitlines()
            if '[MOZYME GPU SCF]' in line and 'cnvgz_active_calls' in line and 'cnvgz_noop_calls' in line
        ]
        if not cnvgz_activity_lines:
            raise RuntimeError('Experimental resident SCF did not report CNVGZ active/no-op counters.')
        cnvgz_activity_fields = _marker_fields(cnvgz_activity_lines[-1])
        cnvgz_active_calls = _as_int(cnvgz_activity_fields.get('cnvgz_active_calls'))
        cnvgz_noop_calls = _as_int(cnvgz_activity_fields.get('cnvgz_noop_calls'))
        if not _cnvgz_active_noop_ready(cnvgz_active_calls, cnvgz_noop_calls):
            raise RuntimeError(f'Experimental resident SCF did not prove CNVGZ GPU stage work: active={cnvgz_active_calls}, noop={cnvgz_noop_calls}.')
        isitsc_match = re.search(
            r'\[MOZYME GPU SCF\]\s+isitsc_okscf=\s*(-?\d+)\s+isitsc_iscf=\s*(-?\d+)',
            scf_probe_text,
        )
        isitsc_okscf = int(isitsc_match.group(1)) if isitsc_match else None
        print('Experimental resident SCF ISITSC:', f'okscf={isitsc_okscf}')
        if isitsc_okscf != 1:
            raise RuntimeError('Experimental resident SCF did not report ISITSC convergence.')
        scf_success_lines = _scf_success_lines()
        if not scf_success_lines:
            raise RuntimeError('Experimental resident SCF did not report [MOZYME GPU SCF] status=success code=0 ready=1 resident=1.')
        print('MOZYME GPU SCF success proof marker lines:')
        print('\n'.join(scf_success_lines[-4:]))
        relocal_occupied_lines = _success_marker_lines(
            '[MOZYME GPU relocal]',
            lambda fields: fields.get('occupied') == '1' or fields.get('mode') == 'occupied' or fields.get('space') == 'occupied' or fields.get('kind') == 'occupied',
        )
        relocal_virtual_lines = _success_marker_lines(
            '[MOZYME GPU relocal]',
            lambda fields: fields.get('virtual') == '1' or fields.get('mode') == 'virtual' or fields.get('space') == 'virtual' or fields.get('kind') == 'virtual',
        )
        if not relocal_occupied_lines or not relocal_virtual_lines:
            relocal_marker_lines = [line for line in scf_probe_text.splitlines() if '[MOZYME GPU relocal]' in line]
            print('MOZYME GPU relocal marker lines:')
            print('\n'.join(relocal_marker_lines[-20:]))
            raise RuntimeError('Experimental resident SCF did not report both occupied and virtual MOZYME GPU relocal success markers.')
        reorth_resident_lines = _success_marker_lines(
            '[MOZYME GPU reorth]',
            lambda fields: fields.get('resident') == '1',
        )
        if not reorth_resident_lines:
            reorth_marker_lines = [line for line in scf_probe_text.splitlines() if '[MOZYME GPU reorth]' in line]
            print('MOZYME GPU reorth marker lines:')
            print('\n'.join(reorth_marker_lines[-20:]))
            raise RuntimeError('Experimental resident SCF did not report [MOZYME GPU reorth] status=success resident=1.')
        print('MOZYME GPU relocal proof marker lines:')
        print('\n'.join((relocal_occupied_lines + relocal_virtual_lines)[-4:]))
        print('MOZYME GPU resident reorth proof marker lines:')
        print('\n'.join(reorth_resident_lines[-4:]))
    if scf_probe.returncode != 0:
        raise subprocess.CalledProcessError(scf_probe.returncode, scf_probe.args, output=scf_probe.stdout)

if not run_molecule_benchmark:
    print('Skipping long molecule benchmark; set run_molecule_benchmark=True after the strict readiness report passes.')
else:
    if require_full_scf_gpu and not full_scf_readiness_passed:
        raise RuntimeError('run_molecule_benchmark=True requires the strict readiness probe to pass first; keep run_full_scf_readiness_probe=True.')
    cmd = [
        'python3', 'scripts/molecule_benchmark_report.py', mopac,
        '--profile', 'publication_large',
        '--out-dir', molecule_report_dir,
        '--bundle-zip', molecule_bundle_zip,
        '--repeats', '1',
        '--timeout', str(molecule_timeout),
        '--preflight-timeout', '900',
        '--preflight-min-speedup', str(preflight_min_speedup),
        '--energy-abs-tol', '1e-3',
        '--energy-rel-tol', '1e-6',
        '--energy-per-atom-tol', '1e-5',
        '--gpu-profile',
        '--mozyme-section-profile',
    ]
    if require_full_scf_gpu:
        cmd.append('--require-full-scf-gpu')
        cmd.append('--require-direct-cosmo-gpu')
        cmd.extend(['--full-scf-probe-input', str(direct_cosmo_input)])
        cmd.append('--full-scf-readiness-cpu-compare')
    result = run(
        cmd,
        cwd=source_dir,
        log_path=CONTENT_DIR / 'mopac_molecule_benchmark.log',
        check=False,
    )

    readme = molecule_report_dir / 'README.md'
    if readme.exists():
        display(Markdown(readme.read_text(encoding='utf-8')))

    for name in [
        'molecule_wall_times.png',
        'molecule_gpu_speedup.png',
        'molecule_heat_accuracy.png',
        'mozyme_section_times.png',
    ]:
        path = molecule_report_dir / name
        if path.exists():
            display(Image(filename=str(path)))

    if molecule_bundle_zip.exists():
        files.download(str(molecule_bundle_zip))

    if result.returncode != 0:
        print('Molecule benchmark did not complete. Diagnostic bundle/logs were still generated when possible.')
        raise subprocess.CalledProcessError(result.returncode, result.args, output=result.stdout)
    if require_full_scf_gpu:
        molecule_readiness_payload = assert_full_scf_readiness_artifact(molecule_report_dir / 'full_scf_gpu_readiness.json')
        assert_full_scf_readiness_cpu_compare_artifact(molecule_report_dir / 'full_scf_gpu_readiness_cpu_compare.json', molecule_readiness_payload)
        full_scf_readiness_passed = True
        print('Full-SCF readiness flag satisfied by molecule benchmark readiness artifact:', molecule_readiness_payload['row'].get('full_scf_gpu_status'))

if require_full_scf_gpu and not full_scf_readiness_passed:
    raise RuntimeError('Proof section ended without full_scf_readiness_passed=True.')
if require_full_scf_gpu and not direct_cosmo_readiness_passed:
    raise RuntimeError('Proof section ended without direct_cosmo_readiness_passed=True.')
if require_full_scf_gpu:
    print('Full-SCF readiness proof flag:', full_scf_readiness_passed)
    print('Direct COSMO readiness proof flag:', direct_cosmo_readiness_passed)
elif allow_non_proof_run:
    print('Non-proof debug run completed; full-SCF readiness proof was not required.')
